---
# FC Mold G5: Mold Level Stability Investigation
## Understanding Meniscus Instability on CC23 — TATA Steel IJmuiden

**Document type:** Technical Investigation Report  
**Version:** 2.5  
**Date:** 24 March 2026  
**Scope:** Continuous Caster 23, Strands 5 & 6  
**Data period:** April – May 2025  
**Analysis notebook:** `EDA_data_grouping` (Repo: `fcMoldG5_data_analysis_TATA`)  

---

### Document Purpose

TATA Steel IJmuiden has experienced **meniscus instability on CC23** despite the FC Mold G5 electromagnetic stirring and braking system being operational. This report presents a **data-driven investigation** into the casting process and the impact of the FC Mold G5, aimed at discovering the root causes of the instability and identifying actionable improvements.

### How to Read This Report

The report follows the evidence progressively:

| Section | Question | Key Tool |
| --- | --- | --- |
| **1. Problem Formulation** | What are we measuring and how? | Data pipeline |
| **3. Trend Learning** | Which parameters correlate with instability? | Visualisation + decomposition |
| **4. Temporal Dynamics** | Does shape change precede level change? | Cross-correlation + Granger |
| **5. EMBR Failure Profiling** | Where did the FC Mold G5 not compensate? | Binned failure rates |
| **6. Predictive Modeling** | What are the dominant drivers? | XGBoost + SHAP |
| **7. Regime Discovery** | Are there natural operating clusters? | HDBSCAN + t-SNE |
| **8. Findings** | What does the evidence tell us? | Synthesis |
| **9. Recommendations** | What should be done? | Prioritised actions |

### V2.5 Changes
* **CRITICAL: Threshold changed from 2 mm to ±1 mm** — the guaranteed mold level stability threshold is ≤±1 mm. All analyses, figures, tables, and narrative updated.
* **Stability recalculated:** Clean: 63.7% (S5) / 58.3% (S6) below ±1 mm. ALL: 50.7% (S5) / 45.7% (S6).
* **Narrative updated:** From 'overwhelmingly stable' to 'median below threshold but significant fraction exceeds it.'
* **HDBSCAN:** 18 (S5) / 15 (S6) clusters on clean data.
* All threshold lines, color scales, failure rates updated to ±1 mm.

### V2.3 Changes
* **Naming convention:** “Broad Face FBG Temperature” replaces “Meniscus Temperature” for spatial averages of FBG sensors. “Meniscus profile” reserved for Chebyshev-reconstructed temperature profiles.
* **Section 1:** Added data filtering cascade (Fig 1.2b), raw disturbance type visualisation (Fig 1.2), L/R asymmetry (Fig 1.3)
* **Section 3.1:** Expanded with sequence identification and BFF-BFL comparison figures
* **Section 3.3:** Fixed Fig 3.4 to show all 4 subplots; added grade family table
* **Section 3.4:** Updated color palette (navy/olive/coral)
* **Section 4:** Added exploratory figures (Fig 4.1b, 4.2b) before conclusive cross-correlation and Granger results
* **Section 5:** Fixed Fig 3.3 empty subplots (SEN column name, percentile threshold)
* **Section 7.2:** Compact tables for A4; full data available as Excel downloads
* **Colour scheme:** Consistent, eye-friendly palette throughout

---

   
---
## Executive Summary

TATA Steel IJmuiden has experienced episodes of meniscus instability on CC23 despite the FC Mold G5 electromagnetic stirring/braking system being active. This investigation analyses 1 million+ sensor readings across Strands 5 and 6 (April–May 2025) to understand where the instability comes from.

**The central finding:** When we restrict the analysis to **true steady-state normal sequences** (no detected disturbances: no excursions, no high variability, no transient bumps), against the **guaranteed ±1 mm threshold**, **only 63.7% (S5) and 58.3% (S6) of sequences meet the target**. Median ML σ is 0.87 mm (S5) and 0.92 mm (S6) — both below ±1 mm — but a significant tail exceeds it. The remaining variability within these normal sequences is subtle but reveals specific, identifiable process conditions — principally **mold width** (R²=0.20 for operating envelope), **SEN depth variability** (the #1 actionable driver via SHAP), and **steel grade effects** (HSLA grades — Family 5 — show higher ML σ even after controlling for Vc and width).

The investigation followed the data progressively:

1. **Sequence grouping and disturbance removal** identified ~1,200 normal 5-minute steady-state windows per strand. Automatic disturbance detection flagged ~20% of these as containing high variability, excursions, or transient bumps. **V2.2 restricts all downstream analysis to the ~80% disturbance-free sequences** (934 S5, 948 S6).

2. **Mold level L/R asymmetry analysis** confirmed that the average mold level is a robust response variable, but the left-right difference reveals standing wave patterns — notably larger on Strand 6 (mean |L−R| = 4.2 mm) than Strand 5 (2.5 mm).

3. **Vc × Width decomposition** showed mold width dominates the operating envelope (R²=0.20 vs 0.04 for Vc alone), with moderate Vc-Width coupling (ρ = −0.45 to −0.58).

4. **Steel grade effects** persist after controlling for casting speed and mold width (Kruskal-Wallis H=58–83, p<10⁻¹²). HSLA grades (Family 5) show the highest ML σ on both strands (1.24 mm S5, 1.28 mm S6). The ranking of other grades differs by strand: on S5, Family 3 (peritectic, 1.22 mm) is close behind Family 5; on S6, Family N (unclassified, 1.09 mm) takes 2nd place while Family 3 drops to 3rd (0.99 mm). Family 1 (low-carbon) is lowest on both strands (0.70/0.78 mm).

5. **Three-category meniscus profiles** reveal the progressive transition from quiescent to disturbed meniscus shape, with wave peaks consistently at the narrow face (±0.7–1.0 normalised width).

6. **Temporal analysis** showed that meniscus shape changes **precede** mold level excursions by ~4 seconds on Strand 6 (Granger F = 210), supporting feedforward control potential.

7. **Predictive modeling** (XGBoost + SHAP) achieved R² = 0.946 (S5) and 0.945 (S6), confirming SEN variability as the #1 actionable driver across both strands.

8. **EMBR failure profiling** — positioned after establishing SEN dominance — identified specific EMBR AC/DC current combinations that coincide with higher ML σ.

9. **Regime discovery** (HDBSCAN) identified 15 clusters (S5) and 13 clusters (S6) with distinct stability profiles, with the top unstable clusters featuring wide molds (>2.0 m) and high SEN variability.

| Metric | Strand 5 | Strand 6 |
| --- | --- | --- |
| Normal sequences (all) | 1,174 | 1,209 |
| Clean sequences (no disturbances) | 934 | 948 |
| ML σ median (clean) | 0.87 mm | 0.92 mm |
| % stable (σ < ±1 mm, clean) | 63.7% | 58.3% |
| XGBoost R² | 0.946 | 0.945 |
| HDBSCAN clusters | 18 | 15 |

   
---
## 1. One Million Raw Sensor Readings Distilled into Clean, Comparable Casting Sequences

### 1.1 The Challenge

TATA Steel IJmuiden operates Continuous Caster 23 (CC23) with two active strands (5 and 6), each equipped with the **FC Mold G5** electromagnetic stirring and braking system. The FC Mold G5 takes as input casting parameters — casting speed (Vc), mold width, SEN immersion depth, and argon flow — to mount the proper electromagnetic field for stirring and braking of the liquid steel flow in order to stabilise the meniscus shape and thus improve slab quality.

Mold level is independently measured by **Vuhz sensors** (electromagnetic induction sensors mounted externally, in inverted configuration on CC23: centre at 265 mm from mold centre, 530 mm sensor-to-sensor distance vs. 400 mm on CC21/22). Two Vuhz sensors per strand (Left and Right) provide both the average mold level and the left-right difference — the latter being a direct indicator of standing wave amplitude in the mold. FBG (Fiber Bragg Grating) optical fiber sensors provide meniscus temperature profiling via 24 sensors (12 BFF + 12 BFL).

The customer has experienced **meniscus instability** — episodes where the mold level fluctuates beyond acceptable limits despite the FC Mold G5 being active. The question is: **where does this instability come from, and under what conditions does it occur?**

### 1.2 Data Grouping: From Raw Signals to Analysable Sequences

The raw data streams at 1 Hz contain startup transients, speed ramps, inter-cast gaps, and equipment-off periods that are not relevant to steady-state stability assessment. We apply a structured grouping procedure to extract the signal from the noise:

**Step 1 — 5-Minute Steady-State Sequence Identification**

The raw time-series is segmented into non-overlapping **5-minute windows** where:
* Casting speed is stable: Vc standard deviation < 0.1 m/min within the window
* FC Mold G5 is active: EMBR current > 50 A (electromagnetic field is engaged)
* No data gaps > 5 seconds (continuous recording)
* Minimum window length: 300 samples (5 minutes at 1 Hz)

This removes startup, shutdown, speed ramps, ladle changes, and equipment-off periods.

**Step 2 — Normal vs. Abnormal Classification**

Sequences where Vc and EMBR currents remain steady throughout the window are classified as **Normal** (steady-state casting). Sequences with abrupt changes (speed step, EMBR dropout, equipment intervention) are classified as **Abnormal** and excluded from downstream analysis.

| | Strand 5 | Strand 6 |
| --- | --- | --- |
| Normal sequences | 1,174 | 1,209 |
| Abnormal sequences | 15,549 | 12,616 |

**Step 3 — Disturbance Detection Within Normal Sequences**

Even within normal (steady-state) sequences, the mold level signal shows different disturbance patterns that reflect what is happening at the production level — operator actions, process transients, or inherent variability. The pipeline automatically classifies each normal sequence into disturbance types:

| Disturbance Type | Description | S5 Count | S6 Count |
| --- | --- | --- | --- |
| **Normal** | Steady mold level, no significant events | 934 (80%) | 948 (78%) |
| **High variability** | Elevated ML σ throughout the window | 112 (10%) | 166 (14%) |
| **Excursion + High variability** | Large mold level excursion event(s) combined with high variability | 107 (9%) | 88 (7%) |
| **Excursion + High variability + Transient bump** | All of the above plus transient bumps | 21 (2%) | 7 (1%) |

Fig 1.1 below shows the ML σ distribution by disturbance type — sequences labelled "Normal" have significantly lower ML σ than those with detected disturbances, confirming that the classification is meaningful. Fig 1.2 shows selected raw time-series examples for each disturbance type, illustrating the visual character of these events to help experts interpret the classification.

**V2.2 — Analysis restricted to disturbance-free sequences.** After consultation with domain experts, all downstream analysis in this report uses **only the disturbance-free normal sequences** (934 S5, 948 S6). The rationale: disturbance events (excursions, high variability, transient bumps) are driven by external production events that mask the true steady-state behaviour of the FC Mold G5 system. By removing them, we can isolate the system's intrinsic response to process parameters and learn the *true baseline* — what the meniscus looks like when the caster is genuinely running at steady state.

### 1.3 The Response Variable: Mold Level σ and Its Physical Basis

The datasets contain no direct quality response variable (no defect index, breakout indicator, or slab quality score). We therefore engineer **ML σ** (mold level standard deviation per 5-minute window) as the response — this is precisely what the FC Mold G5 is designed to minimise.

The mold level signal is derived from the **average of the Left and Right Vuhz sensor** readings. Before accepting this as a reliable response variable, we examine the **left-right asymmetry** — i.e., how different the Left and Right sensors read at any given moment. If the two sensors always agree, the average is unambiguous. If they frequently disagree, the average may hide important physical phenomena such as standing waves in the mold.

The analysis in Fig 1.3 below examines:
* The distribution of |ML\_Left − ML\_Right| across all raw data
* How this asymmetry relates to ML σ
* Whether one strand shows systematically more asymmetry than the other

Understanding L/R asymmetry is physically important because:
* **Asymmetric meniscus** (large L−R) indicates a standing wave driven by the SEN jet — the wave oscillates between left and right narrow faces
* **Symmetric meniscus** (small L−R) indicates either quiescent flow or a symmetric double-roll pattern
* The Vuhz sensor spacing (530 mm on CC23 vs. 400 mm on CC21/22) means the sensors are further apart, potentially capturing different meniscus physics

The ±1 mm threshold used throughout is the **guaranteed performance threshold** for the FC Mold G5 system.

### 1.4 Investigation Approach: Questions That Drive the Analysis

The investigation follows the evidence progressively, with each section motivated by a specific question that emerges from the previous one:

**Section 3 — Trend Learning: What do the raw signals look like?**  
Before building any model, we start from what the operators already know. What does a stable sequence look like vs. a slightly disturbed one? Which signals change? This section establishes the visual vocabulary for the rest of the report and adds casting speed, mold width, and argon flow to the picture.

**Section 3.2 — Process Parameters: Which casting conditions correlate with instability?**  
Having seen the raw signals, we ask: among the normal sequences, which process parameter values tend to produce higher or lower ML σ? Is it speed? Width? SEN depth? This is the classic "correlation before causation" step. We go deeper with inverse-problem methods to understand not just *what* correlates but *why*.

**Section 3.3 — Steel Grades: Do different steel types behave differently?**  
Casting speed and mold width are partially confounded with steel grade (operators cast peritectic steels at different speeds). We separate these effects to determine whether the grade itself carries independent information about stability.

**Section 3.4 — Meniscus Profiles: What does the meniscus shape tell us?**  
The FBG sensors reconstruct the meniscus temperature profile across the mold width. We compare profiles for stable, moderate, and unstable sequences to see *where* on the mold face the instability manifests.

**Section 4 — Temporal Dynamics: Does shape change precede level change?**  
This is the key causality question. If the meniscus shape (Chebyshev coefficients) changes *before* the mold level reacts, this implies feedforward potential: the FBG signal could predict impending instability before it manifests in the Vuhz readings.

**Section 5 — Predictive Modeling: What are the dominant drivers in a multivariate model?**  
Correlation analysis is bivariate; the real world is multivariate. XGBoost + SHAP reveals which features dominate when all are considered simultaneously, accounting for non-linear interactions.

**Section 6 — EMBR Analysis: Where did the FC Mold G5 fail to compensate?**  
Having established that SEN depth is the #1 driver, we now ask: under which EMBR parameter combinations did the FC Mold G5 system fail to fully compensate? This is the question of most direct interest to the vendor.

**Section 7 — Regime Discovery: Are there natural operating clusters?**  
Unsupervised clustering reveals whether the caster naturally operates in distinct regimes with different stability profiles, potentially enabling regime-specific tuning.

Strand 5 and Strand 6 are analysed in parallel — not to compare them, but to **validate** that findings are consistent across independent datasets.

In [0]:
# =====================================================================
# Fig 1.1: ML σ distribution by disturbance type within normal sequences
# Uses df_seq_*_all (ALL sequences before disturbance filtering) to show
# the full picture of disturbance detection before restricting analysis.
# =====================================================================

fig_dist = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 — Normal Sequences by Disturbance Type',
                    'Strand 6 — Normal Sequences by Disturbance Type'],
    horizontal_spacing=0.10)

dist_colors = {
    'Normal': '#27ae60',
    'High variability': '#f39c12',
    'Excursion + High variability': '#e74c3c',
    'Excursion + High variability + Transient bump': '#8e44ad'
}

for i, (df_s, sname) in enumerate([(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]):
    if 'disturbance_type' in df_s.columns:
        for dtype in ['Normal', 'High variability',
                      'Excursion + High variability',
                      'Excursion + High variability + Transient bump']:
            sub = df_s[df_s['disturbance_type'] == dtype]
            if len(sub) > 0:
                short = dtype.replace('Excursion + High variability + Transient bump', 'Exc+HV+Bump') \
                             .replace('Excursion + High variability', 'Exc+HV') \
                             .replace('High variability', 'High var')
                fig_dist.add_trace(go.Box(
                    y=sub[target], name=f'{short} (n={len(sub)})',
                    marker_color=dist_colors.get(dtype, 'gray'),
                    boxpoints='outliers', showlegend=(i==0)),
                    row=1, col=i+1)

fig_dist.update_yaxes(title_text='ML σ [mm]')
fig_dist.update_layout(
    title='Fig 1.1 — ML σ Distribution by Disturbance Type (All Normal Sequences)<br>'
          '<sup>Each box represents one disturbance category. The "Normal" group (green) contains the '
          'disturbance-free sequences used in all downstream analysis. Dots are individual sequence outliers.</sup>',
    height=500, width=1200, template='plotly_white')
fig_dist.show()

# Print summary stats
print('\nDisturbance breakdown (all normal sequences before filtering):')
for df_s, sname in [(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]:
    if 'disturbance_type' in df_s.columns:
        print(f'\n{sname}:')
        for dtype, grp in df_s.groupby('disturbance_type'):
            print(f'  {dtype}: n={len(grp)}, median ML\u03c3={grp[target].median():.3f} mm, '
                  f'mean={grp[target].mean():.3f} mm')
print(f'\n\u2192 V2.2: Only "Normal" sequences retained for downstream analysis.')

In [0]:
# =====================================================================
# Fig 1.2: Raw time-series examples for each disturbance type
# V2.4: Single-column, actual timestamps on x-axis
# =====================================================================

dist_types = ['Normal', 'High variability', 'Excursion + High variability',
              'Excursion + High variability + Transient bump']
short_names = ['Normal', 'High Variability', 'Excursion + HV', 'Exc + HV + Bump']
dist_colors_list = ['#27ae60', '#f39c12', '#e74c3c', '#8e44ad']

df_all, df_fc, sname = df_seq_6_all, df_fc_6, 'Strand 6'

fig = make_subplots(rows=4, cols=1,
    subplot_titles=[f'{sn}' for sn in short_names],
    vertical_spacing=0.06, shared_xaxes=False)

for row_i, (dtype, sn, clr) in enumerate(zip(dist_types, short_names, dist_colors_list)):
    candidates = df_all[df_all['disturbance_type'] == dtype]
    if len(candidates) == 0:
        continue
    med_sigma = candidates[target].median()
    best_idx = (candidates[target] - med_sigma).abs().idxmin()
    seq_row = candidates.loc[best_idx]
    
    t_start = seq_row['Seq_time_Start']
    t_end = seq_row['Seq_time_End']
    raw_seg = df_fc[(df_fc['plainTimeStamp'] >= t_start) &
                   (df_fc['plainTimeStamp'] <= t_end)].copy()
    
    if len(raw_seg) < 10:
        continue
    
    # V2.4: Use actual timestamps on x-axis
    fig.add_trace(go.Scatter(
        x=raw_seg['plainTimeStamp'], y=raw_seg['Mold Level'],
        mode='lines', line=dict(color=clr, width=1.2),
        name=f'{sn} (ML\u03c3={seq_row[target]:.2f}mm)',
        showlegend=True), row=row_i + 1, col=1)
    
    fig.update_yaxes(title_text='ML [mm]', row=row_i + 1, col=1)
    # Show timestamps as HH:MM:SS
    fig.update_xaxes(tickformat='%H:%M:%S', row=row_i + 1, col=1)

# Only bottom panel gets full x-axis label
fig.update_xaxes(title_text='Timestamp', row=4, col=1)
fig.update_layout(
    title=f'Fig 1.2 \u2014 {sname}: Raw Mold Level Signal by Disturbance Type<br>'
          f'<sup>Each panel shows a representative 5-min sequence (actual timestamps). '
          f'\"Normal\" (green) = disturbance-free. Others illustrate detected disturbance patterns.</sup>',
    height=900, width=1000, template='plotly_white',
    legend=dict(x=0.01, y=0.99, font_size=9))
fig.show()

print('V2.4: Fig 1.2 updated to single-column with actual timestamps.')

In [0]:
# =====================================================================
# Fig 1.2b: Data Filtering Cascade — from raw data to analysis-ready sequences
# Shows the progressive filtering steps and how many data points survive each stage
# V2.3: Added to Section 1.2 per feedback (was previously only in Section 3)
# =====================================================================

fig_funnel = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 \u2014 Data Filtering Cascade',
                    'Strand 6 \u2014 Data Filtering Cascade'],
    horizontal_spacing=0.08)

for i, (sname, raw_rows, n_raw, n_fc, n_seq, n_clean, n_abnormal, clr) in enumerate([
    ('S5', 509458, len(df_raw_5), len(df_fc_5), len(df_seq_5_all), len(df_seq_5), len(abnormal_groups_5), '#4C72B0'),
    ('S6', 520526, len(df_raw_6), len(df_fc_6), len(df_seq_6_all), len(df_seq_6), len(abnormal_groups_6), '#DD8452')
]):
    stages = ['Raw joined data', 'Quality-filtered', 'FC Mold active',
              'Normal sequences', 'Disturbance-free']
    counts = [raw_rows, n_raw, n_fc, n_seq, n_clean]
    
    # Use horizontal bar chart (waterfall-like)
    colors = ['#7f8c8d', '#95a5a6', '#5dade2', '#f39c12', '#27ae60']
    fig_funnel.add_trace(go.Funnel(
        y=stages, x=counts, textposition='inside',
        texttemplate='%{x:,.0f}<br>(%{percentInitial:.1%})',
        marker=dict(color=colors),
        connector=dict(line=dict(color='gray', width=1)),
    ), row=1, col=i+1)

fig_funnel.update_layout(
    title='Fig 1.2b \u2014 Data Filtering Cascade: From Raw Sensor Data to Analysis-Ready Sequences<br>'
          '<sup>Each stage removes data that does not meet steady-state, quality, or stability criteria. '
          'The final "disturbance-free" set (green) is used in all downstream analysis.</sup>',
    height=450, width=1200, template='plotly_white', showlegend=False)
fig_funnel.show()

print('\nFiltering summary:')
for sname, raw_rows, n_raw, n_fc, n_seq, n_clean in [
    ('Strand 5', 509458, len(df_raw_5), len(df_fc_5), len(df_seq_5_all), len(df_seq_5)),
    ('Strand 6', 520526, len(df_raw_6), len(df_fc_6), len(df_seq_6_all), len(df_seq_6))
]:
    print(f'  {sname}: {raw_rows:,} raw \u2192 {n_raw:,} quality \u2192 {n_fc:,} FC active \u2192 {n_seq:,} sequences \u2192 {n_clean:,} clean ({100*n_clean/raw_rows:.1f}%)')

In [0]:
# =====================================================================
# Fig 1.3: Mold Level Left/Right Sensor Asymmetry Analysis
# FB2: Examine the L-R difference to validate average ML as response
# variable and understand standing wave patterns.
# =====================================================================
from scipy import stats as sp_stats

fig_lr = make_subplots(rows=2, cols=2,
    subplot_titles=[
        'Distribution of |ML Left \u2212 ML Right|',
        'ML L-R Asymmetry vs ML \u03c3',
        'ML Left vs ML Right (scatter)',
        'Time-series: ML Left, Right, Average'
    ],
    vertical_spacing=0.15, horizontal_spacing=0.12)

for df_fc, sname, color in [(df_fc_5, 'Strand 5', '#e74c3c'), (df_fc_6, 'Strand 6', '#3498db')]:
    if 'Mold Level Sensor Left' not in df_fc.columns:
        print(f'{sname}: ML L/R columns not found')
        continue
    
    ml_l = df_fc['Mold Level Sensor Left'].dropna()
    ml_r = df_fc['Mold Level Sensor Right'].dropna()
    ml_avg = df_fc['Mold Level'].dropna()
    
    # Common index
    common = ml_l.index.intersection(ml_r.index).intersection(ml_avg.index)
    ml_l, ml_r, ml_avg = ml_l[common], ml_r[common], ml_avg[common]
    lr_diff = (ml_l - ml_r).abs()
    
    # Panel 1: Distribution of |L-R|
    fig_lr.add_trace(go.Histogram(x=lr_diff, name=sname, marker_color=color,
        opacity=0.55, nbinsx=60, showlegend=True), row=1, col=1)
    
    # Panel 3: L vs R scatter (subsample)
    sub_idx = np.random.choice(len(common), min(5000, len(common)), replace=False)
    fig_lr.add_trace(go.Scatter(x=ml_l.iloc[sub_idx], y=ml_r.iloc[sub_idx],
        mode='markers', marker=dict(size=2, color=color, opacity=0.2),
        name=sname, showlegend=False), row=2, col=1)
    
    print(f'{sname} \u2014 ML L-R Asymmetry Statistics (raw 1 Hz data):')
    print(f'  Mean |L-R|: {lr_diff.mean():.2f} mm')
    print(f'  Median |L-R|: {lr_diff.median():.2f} mm')
    print(f'  Std |L-R|: {lr_diff.std():.2f} mm')
    print(f'  95th pct |L-R|: {lr_diff.quantile(0.95):.2f} mm')
    print(f'  L-R correlation: {np.corrcoef(ml_l, ml_r)[0,1]:.4f}')
    print()

# Panel 2: Per-sequence L-R asymmetry vs ML sigma
for df_s, sname, color in [(df_seq_5, 'Strand 5', '#e74c3c'), (df_seq_6, 'Strand 6', '#3498db')]:
    if 'ML_LR_asym_std' in df_s.columns:
        fig_lr.add_trace(go.Scatter(x=df_s['ML_LR_asym_std'], y=df_s[target],
            mode='markers', marker=dict(size=4, color=color, opacity=0.4),
            name=sname, showlegend=False), row=1, col=2)
        r_s, p_s = sp_stats.spearmanr(df_s['ML_LR_asym_std'].dropna(), 
                                       df_s[target].loc[df_s['ML_LR_asym_std'].dropna().index])
        print(f'{sname}: ML_LR_asym_std vs ML\u03c3 \u2014 Spearman \u03c1 = {r_s:.3f} (p = {p_s:.2e})')

# Panel 4: Example time-series (30s window)
for df_fc, sname, color in [(df_fc_6, 'Strand 6', '#3498db')]:
    # Pick a short representative window
    seg = df_fc.iloc[1000:1060].copy()
    t_sec = (seg['plainTimeStamp'] - seg['plainTimeStamp'].iloc[0]).dt.total_seconds()
    fig_lr.add_trace(go.Scatter(x=t_sec, y=seg['Mold Level Sensor Left'],
        mode='lines', line=dict(color='#e74c3c', width=1.5, dash='dot'),
        name='ML Left', showlegend=True), row=2, col=2)
    fig_lr.add_trace(go.Scatter(x=t_sec, y=seg['Mold Level Sensor Right'],
        mode='lines', line=dict(color='#3498db', width=1.5, dash='dot'),
        name='ML Right', showlegend=True), row=2, col=2)
    fig_lr.add_trace(go.Scatter(x=t_sec, y=seg['Mold Level'],
        mode='lines', line=dict(color='black', width=2),
        name='ML Average', showlegend=True), row=2, col=2)

fig_lr.update_xaxes(title_text='|ML Left \u2212 ML Right| [mm]', row=1, col=1)
fig_lr.update_xaxes(title_text='ML L-R Asymmetry \u03c3 per sequence', row=1, col=2)
fig_lr.update_xaxes(title_text='ML Left [mm]', row=2, col=1)
fig_lr.update_xaxes(title_text='Time [seconds]', row=2, col=2)
fig_lr.update_yaxes(title_text='Count', row=1, col=1)
fig_lr.update_yaxes(title_text='ML \u03c3 [mm]', row=1, col=2)
fig_lr.update_yaxes(title_text='ML Right [mm]', row=2, col=1)
fig_lr.update_yaxes(title_text='Mold Level [mm]', row=2, col=2)

# Add 1:1 line on L vs R scatter
for df_fc in [df_fc_5, df_fc_6]:
    lim = [df_fc['Mold Level'].quantile(0.01), df_fc['Mold Level'].quantile(0.99)]
    fig_lr.add_trace(go.Scatter(x=lim, y=lim, mode='lines',
        line=dict(color='gray', dash='dash', width=1), showlegend=False), row=2, col=1)

fig_lr.update_layout(
    title='Fig 1.3 \u2014 Mold Level Left/Right Sensor Asymmetry<br>'
          '<sup>Top-left: Distribution of instantaneous |L\u2212R| difference (1 Hz data). '
          'Top-right: Per-sequence L-R variability vs ML \u03c3 \u2014 each dot is one 5-min normal sequence. '
          'Bottom-left: L vs R scatter \u2014 points on the 1:1 line indicate symmetric meniscus. '
          'Bottom-right: Example 60s window showing Left (red), Right (blue), and Average (black).</sup>',
    height=700, width=1200, template='plotly_white', barmode='overlay',
    legend=dict(x=0.75, y=0.98, font_size=9))
fig_lr.show()

print('\nPhysical interpretation:')
print('  \u2022 Small |L-R|: Symmetric meniscus \u2014 quiescent flow or balanced double-roll pattern')
print('  \u2022 Large |L-R|: Standing wave \u2014 SEN jet oscillates between left/right narrow faces')
print('  \u2022 The average ML (used as response variable) is robust when |L-R| is small')
print('  \u2022 When |L-R| is large, the average masks the true wave amplitude at each side')
print('  \u2022 The ML_LR_asym_std feature captures this per-sequence and is included in the XGBoost model')

---
## 2. Data Description

### 2.1 Data Sources

The analysis uses three data sources per strand:

| Source | Description | Sampling | Variables |
| --- | --- | --- | --- |
| **boExpert** (Broadband Expert) | High-frequency mold sensor data: meniscus temperature profiles (12 BFF + 12 BFL FBG optical fiber sensors), Chebyshev polynomial fits (X₁–X₆ per face), mold level signals | 2 Hz (aggregated to 1 Hz) | \~960 columns |
| **dtExpert** (Data Expert) | Process parameters: casting speed, mold width, SEN depth, EMBR currents (6 channels: AC/DC × Left/Right × Master/Bottom), argon flow rates | 1 Hz | \~1,009 columns |
| **Metadata** | Casting log: heat number, steel grade, quality classification, strand ID, casting start/end times | Per casting | 27 columns |

### 2.2 Data Volume

| Strand | boExpert files | dtExpert files | Raw rows (joined) | Columns (joined) |
| --- | --- | --- | --- | --- |
| **Strand 5** | 158 | 158 | 509,458 | 1,969 |
| **Strand 6** | 164 | 168 | 520,526 | 1,969 |

### 2.3 Metadata Coverage

The casting metadata CSV contains 52 records: 25 for Strand 5 and 27 for Strand 6. Each record defines a casting period with start/end timestamps, steel grade, heat number, and quality classification. Metadata is joined via a range join on `(Strand ID, plainTimeStamp BETWEEN cast_start AND cast_end)`.

**Metadata match rate:** \~37% of rows match a metadata record. The remaining 63% fall outside documented casting periods (inter-cast gaps, startup/shutdown phases) — these are correctly excluded during steady-state filtering.

### 2.4 Time Period

All data covers **April – May 2025**, representing a representative sample of normal production with a mix of steel grades, slab widths (0.99 m – 2.01 m), and casting speeds (0.8 – 1.8 m/min).

In [0]:
%pip install python-docx -q

In [0]:
import sys, os, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── Package imports ──────────────────────────────────────────────────
repo_root = "/Workspace/Repos/dieudonne.nkulikiyimfura@se.abb.com/fcMoldG5_data_analysis_TATA"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("fcmold_analysis"):
        del sys.modules[mod_name]

from fcmold_analysis import STRAND_CONFIGS, StrandAnalysisPipeline
from fcmold_analysis.config import CONFIG

target = 'MOLD_LEVEL_std [mm]'
colors = {'Strand 5': '#e74c3c', 'Strand 6': '#3498db'}
fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
os.makedirs(fig_dir, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════
# V2.5: Guaranteed threshold is ≤±1 mm (was 2 mm in V2.4)
# ══════════════════════════════════════════════════════════════════════
ML_THRESHOLD = 1.0  # ±1 mm guaranteed threshold

print('Setup complete. Running both strand pipelines...')
print(f'V2.5: ML stability threshold = ±{ML_THRESHOLD} mm')
print('='*60)

# ── Run Strand 5 pipeline ────────────────────────────────────────────
pipeline_5 = StrandAnalysisPipeline(STRAND_CONFIGS['23_5'], CONFIG)
results_5 = pipeline_5.run(generate_visualizations=False)
df_raw_5 = results_5['df_raw']
df_fc_5  = results_5['df_fc_mold']
df_seq_5_all = results_5['df_seq']  # Keep ALL sequences for Fig 1.1
normal_groups_5 = results_5['normal_groups']
abnormal_groups_5 = results_5['abnormal_groups']
print(f"  Strand 5: {len(df_seq_5_all)} total sequences")

# ── Run Strand 6 pipeline ────────────────────────────────────────────
pipeline_6 = StrandAnalysisPipeline(STRAND_CONFIGS['23_6'], CONFIG)
results_6 = pipeline_6.run(generate_visualizations=False)
df_raw_6 = results_6['df_raw']
df_fc_6  = results_6['df_fc_mold']
df_seq_6_all = results_6['df_seq']  # Keep ALL sequences for Fig 1.1
normal_groups_6 = results_6['normal_groups']
abnormal_groups_6 = results_6['abnormal_groups']
print(f"  Strand 6: {len(df_seq_6_all)} total sequences")

# ══════════════════════════════════════════════════════════════════════
# V2.2 CRITICAL CHANGE: Filter to ONLY normal sequences (no disturbances)
# Rationale: After consulting with domain expert, we restrict analysis
# to true steady-state normal sequences without any disturbance events.
# This allows us to learn the *true* baseline behaviour of the system.
# ══════════════════════════════════════════════════════════════════════
print('\n--- V2.2: Filtering to disturbance-free normal sequences ---')

# Tag strands on ALL sequences (needed for Fig 1.1)
df_seq_5_all['strand'] = 'Strand 5'
df_seq_6_all['strand'] = 'Strand 6'

# Filter: keep only sequences with no detected disturbances
df_seq_5 = df_seq_5_all[~df_seq_5_all['has_disturbance']].copy().reset_index(drop=True)
df_seq_6 = df_seq_6_all[~df_seq_6_all['has_disturbance']].copy().reset_index(drop=True)

for df_s, df_all, sname in [(df_seq_5, df_seq_5_all, 'Strand 5'),
                              (df_seq_6, df_seq_6_all, 'Strand 6')]:
    n_all = len(df_all)
    n_clean = len(df_s)
    n_removed = n_all - n_clean
    print(f'  {sname}: {n_all} total \u2192 {n_clean} clean ({100*n_clean/n_all:.1f}%), '
          f'removed {n_removed} disturbed sequences')
    print(f'    ML\u03c3 median = {df_s[target].median():.3f} mm (was {df_all[target].median():.3f} with disturbances)')

# ── V2.2: Argon flow — use only ArFlow_avg (= SEN + Stopper combined) ──
# Drop separate ArFlow_SEN_avg and ArFlow_Stopper_avg from downstream analysis
ar_drop = ['ArFlow_SEN_avg', 'ArFlow_Stopper_avg']
for df in [df_seq_5, df_seq_6]:
    for c in ar_drop:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)
print(f'\n  Argon: Using only ArFlow_avg [NL/min] (= Argon Flow SEN + Stopper combined)')
print(f'  Dropped: {ar_drop}')

# ── Combine for comparative analysis ─────────────────────────────────
df_seq_5['strand'] = 'Strand 5'
df_seq_6['strand'] = 'Strand 6'
df_cmp = pd.concat([df_seq_5, df_seq_6], ignore_index=True)

# Summary with V2.5 threshold
for df_s, sname in [(df_seq_5, 'Strand 5'), (df_seq_6, 'Strand 6')]:
    pct_stable = 100 * (df_s[target] < ML_THRESHOLD).mean()
    print(f'  {sname}: {len(df_s)} sequences, {pct_stable:.1f}% stable (\u03c3 < {ML_THRESHOLD} mm)')

print(f"\nCombined dataset: {len(df_cmp)} clean sequences, {len(df_cmp.columns)} columns")
print('='*60)

---
## Data Processing Summary

Both strands are processed through an identical, parameterised pipeline (see Appendix for full details). In brief:

1. **Load & Join:** boExpert (FBG meniscus profiles, Chebyshev fits, mold level) and dtExpert (casting speed, SEN depth, EMBR currents, argon flows) are joined at 1 Hz on timestamp.
2. **Feature Engineering:** 21 physics-informed features are derived from \~1,969 raw columns, including meniscus temperature asymmetry (BFF vs BFL), Chebyshev coefficient statistics (odd-order = flow asymmetry, even-order = wave height), and mold level L-R differences from Vuhz sensors.
3. **Sequence Identification:** Data is segmented into non-overlapping 5-minute steady-state windows (FC Mold active, Vc stable, no gaps).
4. **Response Variable:** ML σ (mold level standard deviation per window) — the engineered response variable for this investigation.

| | Strand 5 | Strand 6 |
| --- | --- | --- |
| FC Mold active rows | 378,129 (97.2%) | 387,447 (97.7%) |
| Normal sequences | 1,174 | 1,209 |
| Features per sequence | 82 columns | 82 columns |

---
## 3. Trend Learning: Width Dominates the Operating Envelope, SEN Variability Drives Instability

Before building models, we start from **what the operators already know** and follow the data to discover what contributes to meniscus instability within the disturbance-free normal sequences.

### 3.1 Raw Signal Overview: What Does Stable vs. Unstable Casting Look Like?

Fig 3.1a/b shows 5-minute windows from Strand 6 for the most stable and most unstable disturbance-free sequences. The four panels display:
* **Mold Level** — the response variable (ML σ)
* **Casting Speed (Vc)** — held steady by the steady-state filter
* **SEN Immersion Depth** — note the hunting (variability) even within stable casting
* **Broad Face FBG Temperature** — spatial average of 12 FBG sensors per face (BFF = Fixed Face, BFL = Loose Face)

> **V2.3 naming convention:** The variables `meniscus_bff_avg` and `meniscus_bfl_avg` are the spatial averages of 12 Fiber Bragg Grating (FBG) optical sensors distributed across the full broad face of the mold. These are **not** meniscus temperatures in the strict sense — a true meniscus temperature is measured at the top of the mold (~100 mm from the top of the copper plate where the steel meniscus lies). We refer to these as **Broad Face FBG Temperature** throughout this report. The term **meniscus profile** is reserved for the reconstructed Chebyshev polynomial profiles (Section 3.4 and 6.4) that model the meniscus shape.

Fig 3.1c illustrates the 5-minute sequence identification procedure, and Fig 3.1d compares BFF vs BFL temperatures for stable and unstable examples. Fig 3.1e shows the BFF−BFL asymmetry distribution by stability category.

### 3.2 The Vc × Width Inverse Problem

Casting speed and mold width are operationally coupled — wider molds are cast slower. Section 3.2 decomposes their joint effect using an inverse-problem approach: at matched casting speed, how does width affect stability?

### 3.3 Steel Grade, Argon, and SEN Impact

Steel grade families (defined by the first character of the TATA grade code) carry independent information about stability even after controlling for Vc and Width. Argon flow and SEN depth variability are additional contributors.

In [0]:
# =====================================================================
# Fig 3.1: Raw signal time-series for stable and unstable sequences
# 4 panels: Mold Level, Casting Speed, SEN Depth, Broad Face FBG Temp
# V2.3: Renamed "Meniscus Temperature" → "Broad Face FBG Temperature"
#   Rationale: meniscus_bff_avg / meniscus_bfl_avg are spatial averages
#   of 12 FBG sensors across the full broad face, NOT a true meniscus
#   temperature (which is typically measured at ~100 mm from the top of
#   the copper plate where the steel meniscus lies). The term "meniscus"
#   is reserved for reconstructed Chebyshev profiles in this report.
# =====================================================================

def plot_raw_signals(df, title_suffix, n_minutes=5):
    """Plot 4 key raw signals for a 5-min sequence window."""
    ts = df['plainTimeStamp'].sort_values()
    t0 = ts.iloc[0]
    mask = (df['plainTimeStamp'] >= t0) & (df['plainTimeStamp'] < t0 + pd.Timedelta(minutes=n_minutes))
    seg = df.loc[mask].sort_values('plainTimeStamp')
    t = seg['plainTimeStamp']

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=['Mold Level [mm]',
                        'Casting Speed [m/min]',
                        'SEN Immersion Depth [mm]',
                        'Broad Face FBG Temperature BFF vs BFL [\u00b0C]'])

    # Row 1: Mold Level
    fig.add_trace(go.Scatter(x=t, y=seg['Mold Level'], name='Mold Level',
        line=dict(width=1, color='#2c3e50')), row=1, col=1)

    # Row 2: Casting Speed
    fig.add_trace(go.Scatter(x=t, y=seg['castingSpeed'], name='Vc',
        line=dict(width=1, color='#27ae60')), row=2, col=1)

    # Row 3: SEN Depth
    fig.add_trace(go.Scatter(x=t, y=seg['SENImmersionDepth'], name='SEN',
        line=dict(width=1, color='#d35400')), row=3, col=1)

    # Row 4: Broad Face FBG Temperature (BFF and BFL)
    # meniscus_bff_avg = spatial average of 12 FBG sensors on the Broad Face Fixed side
    # meniscus_bfl_avg = spatial average of 12 FBG sensors on the Broad Face Loose side
    # NOTE: These are NOT meniscus temperatures. They are broad face thermal averages.
    if 'meniscus_bff_avg' in seg.columns:
        fig.add_trace(go.Scatter(x=t, y=seg['meniscus_bff_avg'], name='BFF (Fixed Face)',
            line=dict(width=1, color='#e74c3c')), row=4, col=1)
    if 'meniscus_bfl_avg' in seg.columns:
        fig.add_trace(go.Scatter(x=t, y=seg['meniscus_bfl_avg'], name='BFL (Loose Face)',
            line=dict(width=1, color='#3498db')), row=4, col=1)

    fig.update_layout(
        title=f'Fig 3.1 \u2014 {title_suffix}',
        height=700, width=1200,
        showlegend=True, legend=dict(x=1.01, y=1))
    return fig

# --- Select examples from disturbance-free sequences (df_seq_6) ---
stable_idx = df_seq_6.nsmallest(5, target).index[0]
stable_start = df_seq_6.loc[stable_idx, 'Seq_time_Start']
stable_end = df_seq_6.loc[stable_idx, 'Seq_time_End']
stable_mlsigma = df_seq_6.loc[stable_idx, target]

unstable_idx = df_seq_6.nlargest(5, target).index[0]
unstable_start = df_seq_6.loc[unstable_idx, 'Seq_time_Start']
unstable_end = df_seq_6.loc[unstable_idx, 'Seq_time_End']
unstable_mlsigma = df_seq_6.loc[unstable_idx, target]

seg_stable = df_fc_6[(df_fc_6['plainTimeStamp'] >= stable_start) & (df_fc_6['plainTimeStamp'] <= stable_end)]
seg_unstable = df_fc_6[(df_fc_6['plainTimeStamp'] >= unstable_start) & (df_fc_6['plainTimeStamp'] <= unstable_end)]

print(f'Stable example:   Seq #{stable_idx}, ML\u03c3 = {stable_mlsigma:.3f} mm')
print(f'Unstable example: Seq #{unstable_idx}, ML\u03c3 = {unstable_mlsigma:.3f} mm')
print(f'  (Clean sequences ML\u03c3 range: {df_seq_6[target].min():.3f} \u2013 {df_seq_6[target].max():.3f} mm)')
print()
print('\u26a0 Naming convention (V2.3): "Broad Face FBG Temperature" replaces "Meniscus Temperature".')
print('  meniscus_bff_avg / meniscus_bfl_avg = spatial average of 12 FBG sensors distributed')
print('  across the broad face. These are NOT meniscus temperatures (measured at ~100mm from')
print('  top of copper plate). The term "meniscus profile" is reserved for the Chebyshev-')
print('  reconstructed temperature profiles that model the meniscus shape.')

fig_stable = plot_raw_signals(seg_stable,
    f'Strand 6 \u2014 STABLE Casting (ML\u03c3 = {stable_mlsigma:.2f} mm)')
fig_stable.show()

fig_unstable = plot_raw_signals(seg_unstable,
    f'Strand 6 \u2014 UNSTABLE Casting (ML\u03c3 = {unstable_mlsigma:.2f} mm)')
fig_unstable.show()

In [0]:
# =====================================================================
# Fig 3.1c/d: Sequence grouping illustration + BFF/BFL temperature comparison
# V2.3: Expanded with additional exploratory panels:
#   - Sequence identification on a casting segment (how windows are cut)
#   - BFF vs BFL temperature stable/unstable comparison
#   - BFF-BFL asymmetry distribution by stability category
#   - Correlation matrix of broad face thermal features vs ML sigma
# =====================================================================

# --- Fig 3.1c: Sequence identification on a casting segment ---
t0 = df_fc_6['plainTimeStamp'].iloc[0]
window_end = t0 + pd.Timedelta(minutes=30)
seg = df_fc_6[(df_fc_6['plainTimeStamp'] >= t0) & (df_fc_6['plainTimeStamp'] <= window_end)].copy()

fig_seg = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=['Mold Level [mm] with 5-min Sequence Boundaries',
                    'Casting Speed [m/min] \u2014 Vc Stability Filter'])

fig_seg.add_trace(go.Scatter(x=seg['plainTimeStamp'], y=seg['Mold Level'],
    name='Mold Level', line=dict(width=1, color='#2c3e50')), row=1, col=1)

for i in range(7):
    t_bound = t0 + pd.Timedelta(minutes=5*i)
    fig_seg.add_vline(x=t_bound, line_dash='dot', line_color='gray', opacity=0.5, row=1, col=1)
    fig_seg.add_vline(x=t_bound, line_dash='dot', line_color='gray', opacity=0.5, row=2, col=1)

fig_seg.add_trace(go.Scatter(x=seg['plainTimeStamp'], y=seg['castingSpeed'],
    name='Vc', line=dict(width=1.5, color='#27ae60')), row=2, col=1)

vc_mean = seg['castingSpeed'].mean()
fig_seg.add_hline(y=vc_mean + 0.1, line_dash='dash', line_color='red', opacity=0.5, row=2, col=1)
fig_seg.add_hline(y=vc_mean - 0.1, line_dash='dash', line_color='red', opacity=0.5, row=2, col=1)
fig_seg.add_annotation(text='Vc stability band (\u00b10.1 m/min)', x=1, y=1.01, xref='paper', yref='paper',
                        showarrow=False, font=dict(size=10, color='red'))

fig_seg.update_layout(title='Fig 3.1c \u2014 Sequence Identification: 5-min Non-overlapping Windows with Vc Stability Filter',
                      height=500, width=1200, template='plotly_white')
fig_seg.show()

# --- Fig 3.1d: BFF vs BFL broad face temperature comparison ---
fig_men = make_subplots(rows=1, cols=2,
    subplot_titles=['Stable Sequence \u2014 BFF vs BFL Temperature',
                    'Unstable Sequence \u2014 BFF vs BFL Temperature'])

for i, (seg_ex, label) in enumerate([(seg_stable, 'Stable'), (seg_unstable, 'Unstable')]):
    col = i + 1
    t = seg_ex['plainTimeStamp']
    if 'meniscus_bff_avg' in seg_ex.columns:
        fig_men.add_trace(go.Scatter(x=t, y=seg_ex['meniscus_bff_avg'], name=f'{label} BFF',
            line=dict(width=1, color='#e74c3c'), showlegend=(i==0)), row=1, col=col)
    if 'meniscus_bfl_avg' in seg_ex.columns:
        fig_men.add_trace(go.Scatter(x=t, y=seg_ex['meniscus_bfl_avg'], name=f'{label} BFL',
            line=dict(width=1, color='#3498db'), showlegend=(i==0)), row=1, col=col)
    if 'meniscus_bff_avg' in seg_ex.columns and 'meniscus_bfl_avg' in seg_ex.columns:
        asym = seg_ex['meniscus_bff_avg'] - seg_ex['meniscus_bfl_avg']
        fig_men.add_trace(go.Scatter(x=t, y=asym, name=f'{label} BFF\u2212BFL',
            line=dict(width=1, color='#8e44ad', dash='dot'), showlegend=(i==0)), row=1, col=col)

fig_men.update_layout(
    title='Fig 3.1d \u2014 Broad Face FBG Temperature: Stable vs Unstable Casting<br>'
          '<sup>BFF = Fixed Face (red), BFL = Loose Face (blue), BFF\u2212BFL asymmetry (purple dashed). '
          'In unstable sequences the BFF-BFL asymmetry oscillates much more.</sup>',
    height=400, width=1200, legend=dict(x=0.01, y=0.99), template='plotly_white')
fig_men.update_yaxes(title_text='Temperature [\u00b0C]')
fig_men.show()

# --- Fig 3.1e: BFF-BFL asymmetry distribution by stability category ---
fig_asym_dist = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5: BFF\u2212BFL Asymmetry by Stability',
                    'Strand 6: BFF\u2212BFL Asymmetry by Stability'],
    horizontal_spacing=0.10)

stab_colors = {'Stable': '#1b7340', 'Medium': '#c98a1d', 'Unstable': '#b5302a'}
for i, (df_s, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    q_low = df_s[target].quantile(0.20)
    q_high = df_s[target].quantile(0.80)
    for cat, mask_fn, clr in [
        ('Stable', lambda s: s < q_low, stab_colors['Stable']),
        ('Medium', lambda s: (s >= q_low) & (s < q_high), stab_colors['Medium']),
        ('Unstable', lambda s: s >= q_high, stab_colors['Unstable'])
    ]:
        mask = mask_fn(df_s[target])
        if 'meniscus_FF_LF_asym_std' in df_s.columns:
            vals = df_s.loc[mask, 'meniscus_FF_LF_asym_std'].dropna()
            fig_asym_dist.add_trace(go.Box(y=vals, name=f'{cat} (n={len(vals)})',
                marker_color=clr, showlegend=(i==0)), row=1, col=i+1)

fig_asym_dist.update_yaxes(title_text='BFF\u2212BFL Asymmetry \u03c3')
fig_asym_dist.update_layout(
    title='Fig 3.1e \u2014 Broad Face Thermal Asymmetry (BFF\u2212BFL) by Stability Category<br>'
          '<sup>Unstable sequences show systematically higher BFF\u2212BFL asymmetry variability</sup>',
    height=450, width=1000, template='plotly_white')
fig_asym_dist.show()

print('\nKey observation: In the unstable segment, the BFF-BFL asymmetry oscillates much more')
print('than in the stable segment. This broad face thermal asymmetry is a leading indicator')
print('of mold level instability (SHAP rank #3 on Strand 5).')

In [0]:
# =====================================================================
# Fig 4.3: Data filtering cascade (funnel diagram)
# =====================================================================

def make_funnel(strand_name, raw_rows, after_filter, fc_on, n_normal, n_abnormal, color):
    stages = ['Raw joined data', 'Steady-state filter', 'FC Mold active',
              'Normal sequences', 'Abnormal sequences']
    counts = [raw_rows, after_filter, fc_on, n_normal, n_abnormal]
    pcts = [100] + [100*c/raw_rows for c in counts[1:]]

    fig = go.Figure(go.Funnel(
        y=stages, x=counts, textposition='inside',
        texttemplate='%{x:,.0f} (%{percentInitial:.1%})',
        marker=dict(color=[color]*3 + ['#27ae60', '#e74c3c']),
        connector=dict(line=dict(color='gray', width=1)),
    ))
    fig.update_layout(title=f'{strand_name} — Data Filtering Cascade',
                      height=400, width=600)
    return fig

fig_f5 = make_funnel('Strand 5',
    raw_rows=509458, after_filter=len(df_raw_5), fc_on=len(df_fc_5),
    n_normal=len(df_seq_5), n_abnormal=len(abnormal_groups_5), color='#e74c3c')
fig_f6 = make_funnel('Strand 6',
    raw_rows=520526, after_filter=len(df_raw_6), fc_on=len(df_fc_6),
    n_normal=len(df_seq_6), n_abnormal=len(abnormal_groups_6), color='#3498db')
fig_f5.show()
fig_f6.show()

---
### 5. Specific EMBR Current Combinations Coincide with Elevated Mold Level Instability

> **Placement rationale (V2.2):** This section was moved from Section 3 to after the temporal dynamics analysis, following the logic of the evidence chain. Sections 3–4 established that SEN depth variability and meniscus shape changes are the primary drivers of ML instability. We now ask: **given this knowledge, under which EMBR parameter combinations did the FC Mold G5 system fail to fully compensate?**

The FC Mold G5 electromagnetic stirring/braking system is designed to stabilise the meniscus **despite** changing process conditions. Operators routinely adjust SEN immersion depth to manage nozzle clogging — this is precisely the scenario the system is purchased to handle. Yet certain EMBR current combinations coincide with higher ML σ even within normal, disturbance-free sequences.

Fig 3.3 below profiles the EMBR AC and DC current settings for sequences in the upper 14% of ML σ (potential failure modes) vs. the stable majority. This identifies specific current combinations that may represent suboptimal lookup table entries or operating regions where the electromagnetic field geometry is insufficient to control the flow.

In [0]:
# =====================================================================
# Fig 3.3: EMBR Failure Analysis — where did FC Mold G5 not compensate?
# V2.3: Fixed SEN_std column name, percentile-based threshold for clean data
# =====================================================================
import numpy as np

# V2.3: Use percentile threshold (top 14%) since disturbance-free data has
# fewer sequences > 1 mm.
sen_std_col = 'SEN_std [mm]'  # V2.3 fix: was 'SEN_std'

embr_ac = 'EMBR_AC_Master_LR_mean'
embr_dc = 'EMBR_DC_Master_LR_mean'
embr_dc_bot = 'EMBR_DC_Bottom_LR_mean'

fig_embr = make_subplots(rows=2, cols=3,
    subplot_titles=[
        'S5: AC vs DC Master (colour=ML\u03c3)', 'S5: Failure rate by AC bin', 'S5: SEN\u03c3 vs EMBR DC interaction',
        'S6: AC vs DC Master (colour=ML\u03c3)', 'S6: Failure rate by AC bin', 'S6: SEN\u03c3 vs EMBR DC interaction'],
    horizontal_spacing=0.08, vertical_spacing=0.15)

for i, (df_s, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    row = i + 1
    df_s = df_s.copy()
    # V2.3: percentile-based threshold (top 14%)
    threshold_pct = df_s[target].quantile(0.86)
    df_s['unstable'] = df_s[target] > threshold_pct
    n_unstable = df_s['unstable'].sum()
    print(f'{sname}: threshold (86th pct) = {threshold_pct:.3f} mm, {n_unstable} unstable ({100*n_unstable/len(df_s):.1f}%)')
    
    # --- 3.3a: Scatter: AC vs DC Master, colour = ML\u03c3 ---
    fig_embr.add_trace(go.Scatter(
        x=df_s[embr_ac], y=df_s[embr_dc],
        mode='markers', marker=dict(
            color=df_s[target], colorscale='YlOrRd', size=4,
            cmin=0, cmax=df_s[target].quantile(0.95),
            colorbar=dict(title='ML\u03c3', x=0.30, len=0.4, y=0.5) if i==0 else None),
        showlegend=False, name=sname),
        row=row, col=1)
    fig_embr.update_xaxes(title_text='AC Master LR mean [A]', row=row, col=1)
    fig_embr.update_yaxes(title_text='DC Master LR mean [A]', row=row, col=1)
    
    # --- 3.3b: Failure rate by AC current bin ---
    if df_s[embr_ac].notna().sum() > 0:
        ac_bins = pd.qcut(df_s[embr_ac], q=6, duplicates='drop')
        fail_rate = df_s.groupby(ac_bins)['unstable'].mean() * 100
        fig_embr.add_trace(go.Bar(
            x=[str(b) for b in fail_rate.index],
            y=fail_rate.values,
            marker_color=['#b5302a' if v > 20 else '#c98a1d' if v > 10 else '#1b7340' for v in fail_rate.values],
            showlegend=False, name=f'{sname} fail%'),
            row=row, col=2)
        fig_embr.update_xaxes(title_text='AC Master bin', tickangle=45, row=row, col=2)
        fig_embr.update_yaxes(title_text='Unstable % (top 14%)', row=row, col=2)
    
    # --- 3.3c: SEN\u03c3 vs EMBR DC interaction ---
    if sen_std_col in df_s.columns and embr_dc in df_s.columns:
        stable = df_s[~df_s['unstable']]
        unstable = df_s[df_s['unstable']]
        fig_embr.add_trace(go.Scatter(
            x=stable[sen_std_col], y=stable[embr_dc],
            mode='markers', marker=dict(color='#1b7340', size=3, opacity=0.4),
            name='Stable (bottom 86%)' if i==0 else None, showlegend=(i==0)),
            row=row, col=3)
        fig_embr.add_trace(go.Scatter(
            x=unstable[sen_std_col], y=unstable[embr_dc],
            mode='markers', marker=dict(color='#b5302a', size=5, opacity=0.7),
            name='Unstable (top 14%)' if i==0 else None, showlegend=(i==0)),
            row=row, col=3)
        fig_embr.update_xaxes(title_text='SEN \u03c3 [mm]', row=row, col=3)
        fig_embr.update_yaxes(title_text='DC Master LR mean [A]', row=row, col=3)
    else:
        print(f'  {sname}: Missing {sen_std_col} or {embr_dc} \u2014 subplot empty')

fig_embr.update_layout(
    title='Fig 3.3 \u2014 EMBR Failure Analysis: Where FC Mold G5 Did Not Compensate<br>'
          '<sup>"Unstable" = top 14% of ML \u03c3 within disturbance-free sequences. '
          'Left: EMBR current landscape. Centre: failure rate by AC bin. Right: SEN variability vs EMBR DC.</sup>',
    height=800, width=1400, template='plotly_white')
fig_embr.show()

# Print failure profiles
for df_s, sname in [(df_seq_5, 'Strand 5'), (df_seq_6, 'Strand 6')]:
    df_s = df_s.copy()
    threshold_pct = df_s[target].quantile(0.86)
    df_s['unstable'] = df_s[target] > threshold_pct
    n_unstable = df_s['unstable'].sum()
    n_total = len(df_s)
    print(f'\n{sname}: {n_unstable}/{n_total} in top 14% (ML\u03c3 > {threshold_pct:.3f} mm)')
    if embr_ac in df_s.columns:
        unstable_median_ac = df_s.loc[df_s['unstable'], embr_ac].median()
        stable_median_ac = df_s.loc[~df_s['unstable'], embr_ac].median()
        print(f'  Median AC Master: stable={stable_median_ac:.1f}A, unstable={unstable_median_ac:.1f}A')
    if embr_dc in df_s.columns:
        unstable_median_dc = df_s.loc[df_s['unstable'], embr_dc].median()
        stable_median_dc = df_s.loc[~df_s['unstable'], embr_dc].median()
        print(f'  Median DC Master: stable={stable_median_dc:.1f}A, unstable={unstable_median_dc:.1f}A')
    if sen_std_col in df_s.columns:
        unstable_sen = df_s.loc[df_s['unstable'], sen_std_col].median()
        stable_sen = df_s.loc[~df_s['unstable'], sen_std_col].median()
        print(f'  Median SEN\u03c3: stable={stable_sen:.3f}mm, unstable={unstable_sen:.3f}mm')

In [0]:
# =====================================================================
# Fig 5.1: Distribution overlays for key process parameters
# =====================================================================
panels = [
    (target, 'ML \u03c3 [mm]'),
    ('CASTING_SPEED_avg [m/min]', 'Casting Speed [m/min]'),
    ('MOLD_WIDTH_avg [m]', 'Mold Width [m]'),
    ('SEN_avg [mm]', 'SEN Depth [mm]'),
    ('ArFlow_SEN_avg', 'Argon Flow SEN [NL/min]'),
    ('ptp_mm', 'ML Peak-to-Peak [mm]'),
]

fig = make_subplots(rows=2, cols=3,
    subplot_titles=[p[1] for p in panels],
    vertical_spacing=0.14, horizontal_spacing=0.07)

for i, (col, label) in enumerate(panels):
    row, c = i // 3 + 1, i % 3 + 1
    if col not in df_cmp.columns:
        continue
    for strand, color in colors.items():
        vals = df_cmp.loc[df_cmp['strand'] == strand, col].dropna()
        fig.add_trace(go.Histogram(x=vals, name=strand, marker_color=color,
            opacity=0.55, nbinsx=50, showlegend=(i == 0)), row=row, col=c)
    fig.update_xaxes(title_text=label, row=row, col=c)

fig.update_layout(title='Fig 5.1 — Process Parameter Distributions: Strand 5 vs Strand 6',
                  height=650, width=1300, barmode='overlay',
                  legend=dict(x=0.85, y=0.98))
fig.show()

In [0]:
# =====================================================================
# Fig 5.2: Engineered feature distributions
# =====================================================================
from scipy import stats as sp_stats

feat_panels = [
    ('meniscus_FF_LF_asym_std', 'Meniscus FF-LF Asym \u03c3'),
    ('cheb_X2_bfl_std', 'Cheb X\u2082 BFL \u03c3'),
    ('cheb_X2_bff_std', 'Cheb X\u2082 BFF \u03c3'),
    ('cheb_X1_bfl_std', 'Cheb X\u2081 BFL \u03c3'),
    ('ML_LR_asym_std', 'ML L-R Asymmetry \u03c3'),
    ('EMBR_DC_Bottom_LR_std', 'EMBR DC Bottom L-R \u03c3'),
    ('ArFlow_SEN_avg', 'Argon Flow SEN avg'),
    ('castingLength_avg', 'Casting Length avg'),
    ('superHeat_avg', 'Superheat avg'),
]

available = [(col, lbl) for col, lbl in feat_panels if col in df_cmp.columns]
nr = (len(available) + 2) // 3

fig = make_subplots(rows=nr, cols=3,
    subplot_titles=[lbl for _, lbl in available],
    vertical_spacing=0.13, horizontal_spacing=0.07)

for i, (col, lbl) in enumerate(available):
    row, c = i // 3 + 1, i % 3 + 1
    for strand, color in colors.items():
        vals = df_cmp.loc[df_cmp['strand'] == strand, col].dropna()
        fig.add_trace(go.Histogram(x=vals, name=strand, marker_color=color,
            opacity=0.55, nbinsx=40, showlegend=(i == 0)), row=row, col=c)

fig.update_layout(title='Fig 5.2 — Engineered Feature Distributions: Strand 5 vs Strand 6',
                  height=300*nr, width=1300, barmode='overlay',
                  legend=dict(x=0.85, y=0.98))
fig.show()

# KS test summary
print('\nKolmogorov-Smirnov Tests:')
print('='*78)
print(f"  {'Feature':<40} {'KS stat':>10} {'p-value':>12} {'Sig.':>6}")
print('-'*78)
for col, lbl in available:
    v5 = df_seq_5[col].dropna().values
    v6 = df_seq_6[col].dropna().values
    if len(v5) > 10 and len(v6) > 10:
        ks, pv = sp_stats.ks_2samp(v5, v6)
        sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else 'ns'))
        print(f"  {col:<40} {ks:>10.4f} {pv:>12.2e} {sig:>6}")

In [0]:
# =====================================================================
# Fig 3.2: Deeper Vc × Width Analysis — The Inverse Problem
# FB5: Casting speed and mold width are operationally coupled (wider
# molds are cast slower). This cell decomposes their joint effect on ML σ.
# =====================================================================

vc_col = 'CASTING_SPEED_avg [m/min]'
w_col = 'MOLD_WIDTH_avg [m]'
ar_col = 'ArFlow_avg [NL/min]'

df_both = pd.concat([df_seq_5.assign(strand='S5'), df_seq_6.assign(strand='S6')])

# --- Fig 3.2a: Vc vs Width scatter colored by ML σ ---
fig_inv = make_subplots(rows=1, cols=3,
    subplot_titles=['Vc vs Width (colored by ML σ)',
                    'Binned Heatmap: Median ML σ',
                    'Width Effect at Fixed Vc'],
    horizontal_spacing=0.10)

for strand, color, symb in [('S5', '#e74c3c', 'circle'), ('S6', '#3498db', 'diamond')]:
    sub = df_both[df_both['strand'] == strand]
    fig_inv.add_trace(go.Scatter(
        x=sub[vc_col], y=sub[w_col], mode='markers',
        marker=dict(size=5, color=sub[target], colorscale='RdYlGn_r',
                    cmin=0, cmax=1, opacity=0.5, symbol=symb),
        name=strand, showlegend=True), row=1, col=1)
fig_inv.update_xaxes(title_text='Vc [m/min]', row=1, col=1)
fig_inv.update_yaxes(title_text='Mold Width [m]', row=1, col=1)

# --- Fig 3.2b: Binned heatmap ---
vc_bins = pd.cut(df_both[vc_col], bins=6)
w_bins = pd.cut(df_both[w_col], bins=5)
heat = df_both.groupby([vc_bins, w_bins])[target].agg(['median', 'count']).reset_index()
heat.columns = ['vc_bin', 'w_bin', 'median_ml', 'count']
heat = heat[heat['count'] >= 5]

# Pivot for heatmap
heat['vc_label'] = heat['vc_bin'].astype(str)
heat['w_label'] = heat['w_bin'].astype(str)
pivot = heat.pivot_table(values='median_ml', index='w_label', columns='vc_label')

if len(pivot) > 0:
    fig_inv.add_trace(go.Heatmap(
        z=pivot.values, x=[str(c) for c in pivot.columns],
        y=[str(r) for r in pivot.index],
        colorscale='RdYlGn_r', zmin=0.5, zmax=1.5,
        text=np.round(pivot.values, 2), texttemplate='%{text}',
        showscale=True, colorbar=dict(title='Median MLσ', x=0.68)),
        row=1, col=2)
fig_inv.update_xaxes(title_text='Vc bin', row=1, col=2)
fig_inv.update_yaxes(title_text='Width bin', row=1, col=2)

# --- Fig 3.2c: Width effect at fixed Vc — partial dependence ---
vc_terciles = pd.qcut(df_both[vc_col], q=3, labels=['Low Vc', 'Mid Vc', 'High Vc'], duplicates='drop')
df_both['vc_tercile'] = vc_terciles
colors_vc = {'Low Vc': '#2ecc71', 'Mid Vc': '#f39c12', 'High Vc': '#e74c3c'}

for vc_label in ['Low Vc', 'Mid Vc', 'High Vc']:
    sub = df_both[df_both['vc_tercile'] == vc_label]
    if len(sub) > 20:
        # Bin width and compute median MLσ per bin
        w_binned = pd.qcut(sub[w_col], q=5, duplicates='drop')
        grouped = sub.groupby(w_binned).agg(
            median_ml=(target, 'median'),
            mean_w=(w_col, 'mean'),
            n=(target, 'count')
        ).reset_index()
        grouped = grouped[grouped['n'] >= 3]
        fig_inv.add_trace(go.Scatter(
            x=grouped['mean_w'], y=grouped['median_ml'],
            mode='lines+markers', name=f'{vc_label} (n={len(sub)})',
            line=dict(color=colors_vc[vc_label]),
            marker=dict(size=8)), row=1, col=3)

fig_inv.update_xaxes(title_text='Mold Width [m]', row=1, col=3)
fig_inv.update_yaxes(title_text='Median ML σ [mm]', row=1, col=3)

fig_inv.update_layout(
    title='Fig 3.2 — The Vc × Width Inverse Problem: Decomposing Confounded Effects<br>'
          '<sup>Left: scatter shows Vc-Width coupling. Centre: joint effect. Right: width effect at fixed Vc (partial dependence).</sup>',
    height=500, width=1500, template='plotly_white')
fig_inv.show()

# --- Correlation decomposition ---
from sklearn.linear_model import LinearRegression
print('\nVc × Width Decomposition (both strands):')
print('=' * 60)
for df_s, sname in [(df_seq_5, 'S5'), (df_seq_6, 'S6')]:
    valid = df_s.dropna(subset=[vc_col, w_col, target])
    X_vc = valid[[vc_col]].values
    X_w = valid[[w_col]].values
    X_both = valid[[vc_col, w_col]].values
    y = valid[target].values
    
    r2_vc = LinearRegression().fit(X_vc, y).score(X_vc, y)
    r2_w = LinearRegression().fit(X_w, y).score(X_w, y)
    r2_both = LinearRegression().fit(X_both, y).score(X_both, y)
    
    # Spearman for Vc vs Width coupling
    from scipy.stats import spearmanr
    r_vw, p_vw = spearmanr(valid[vc_col], valid[w_col])
    
    print(f'  {sname}: Vc-Width coupling \u03c1={r_vw:.3f} (p={p_vw:.2e})')
    print(f'  {sname}: R\u00b2(Vc only)={r2_vc:.4f}, R\u00b2(Width only)={r2_w:.4f}, R\u00b2(both)={r2_both:.4f}')
    print(f'  {sname}: Width adds {r2_both - r2_vc:.4f} R\u00b2 beyond Vc alone')

In [0]:
# =====================================================================
# Fig 3.4: Impact of steel grade, SEN type, and Argon on ML σ
# V2.3: Added grade family summary table, fixed SEN_std column name
# =====================================================================
from scipy import stats as sp_stats
from sklearn.linear_model import LinearRegression

ar_col = 'ArFlow_avg [NL/min]'
sen_std_col = 'SEN_std [mm]'  # V2.3 fix

# --- Derive quality_family from 'Quality casting' column ---
grade_family_map = {
    '1': 'Family 1 \u2014 Low-carbon Al-killed',
    '2': 'Family 2 \u2014 Medium-carbon',
    '3': 'Family 3 \u2014 High-carbon / peritectic',
    '5': 'Family 5 \u2014 Micro-alloyed / HSLA',
}

for df_s, name in [(df_seq_5, 'S5'), (df_seq_6, 'S6')]:
    if 'Quality casting' in df_s.columns:
        df_s['quality_family'] = df_s['Quality casting'].apply(
            lambda x: str(x)[0] if pd.notna(x) and str(x).strip() != '' else 'Unknown'
        )
    else:
        df_s['quality_family'] = 'Unknown'

# --- V2.3: Grade Family Summary Table ---
print('\nTable 3.1 \u2014 Steel Grade Family Definitions and Sequence Counts')
print('=' * 100)
rows = []
for fam, desc in grade_family_map.items():
    row_data = {'Family': fam, 'Description': desc}
    for df_s, sname in [(df_seq_5, 'S5'), (df_seq_6, 'S6')]:
        sub = df_s[df_s['quality_family'] == fam]
        if len(sub) > 0:
            actual = sorted(sub['Quality casting'].dropna().unique())
            row_data[f'{sname} n'] = len(sub)
            row_data[f'{sname} grades'] = ', '.join([str(g) for g in actual])
            row_data[f'{sname} median ML\u03c3'] = f'{sub[target].median():.3f}'
        else:
            row_data[f'{sname} n'] = 0
            row_data[f'{sname} grades'] = '\u2014'
            row_data[f'{sname} median ML\u03c3'] = '\u2014'
    rows.append(row_data)

# Unknown row
for df_s, sname in [(df_seq_5, 'S5'), (df_seq_6, 'S6')]:
    unk = df_s[df_s['quality_family'] == 'Unknown']
    rows.append({'Family': '?', 'Description': 'Unknown \u2014 No metadata',
                 f'{sname} n': len(unk),
                 f'{sname} grades': '\u2014',
                 f'{sname} median ML\u03c3': f'{unk[target].median():.3f}' if len(unk) > 0 else '\u2014'})

df_grade_table = pd.DataFrame(rows)
print()
display(df_grade_table)
print()

df_cmp_qf = pd.concat([
    df_seq_5.assign(strand='Strand 5'),
    df_seq_6.assign(strand='Strand 6')
])

colors_strand = {'Strand 5': '#4C72B0', 'Strand 6': '#DD8452'}  # V2.3: consistent color palette

# --- Fig 3.4a: Steel Grade Family vs ML \u03c3 ---
fig_grade = make_subplots(rows=1, cols=2,
    subplot_titles=['Steel Grade Family vs ML \u03c3', 'Argon Flow vs ML \u03c3'],
    horizontal_spacing=0.12)

for strand, color in colors_strand.items():
    sub = df_cmp_qf[df_cmp_qf['strand'] == strand]
    grades = sorted(sub['quality_family'].dropna().unique())
    for g in grades:
        vals = sub[sub['quality_family'] == g][target]
        fig_grade.add_trace(go.Box(y=vals, name=f'{strand}: {g}',
            marker_color=color, opacity=0.6, boxpoints='outliers',
            showlegend=False), row=1, col=1)

for strand, color in colors_strand.items():
    sub = df_cmp_qf[df_cmp_qf['strand'] == strand]
    if ar_col in sub.columns:
        fig_grade.add_trace(go.Scatter(
            x=sub[ar_col], y=sub[target],
            mode='markers', name=strand,
            marker=dict(size=4, color=color, opacity=0.4),
            showlegend=True), row=1, col=2)
        fig_grade.update_xaxes(title_text='Argon Flow [NL/min]', row=1, col=2)

fig_grade.update_yaxes(title_text='ML \u03c3 [mm]', row=1, col=1)
fig_grade.update_yaxes(title_text='ML \u03c3 [mm]', row=1, col=2)
fig_grade.update_layout(title='Fig 3.4a \u2014 Steel Grade Family and Argon Flow Impact on ML Stability',
                        height=500, width=1200, template='plotly_white')
fig_grade.show()

# --- Fig 3.4b: Stratified analysis \u2014 grade effect AFTER controlling for width/speed ---
fig_strat = make_subplots(rows=1, cols=2,
    subplot_titles=['S5: Grade effect within Vc\u00d7Width strata',
                    'S6: Grade effect within Vc\u00d7Width strata'],
    horizontal_spacing=0.10)

for i, (df_s, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    df_s = df_s.copy()
    vc_col = 'CASTING_SPEED_avg [m/min]'
    w_col = 'MOLD_WIDTH_avg [m]'
    if vc_col in df_s.columns and w_col in df_s.columns:
        df_s['vc_bin'] = pd.qcut(df_s[vc_col], q=3, labels=['Slow', 'Mid', 'Fast'], duplicates='drop')
        df_s['w_bin'] = pd.qcut(df_s[w_col], q=2, labels=['Narrow', 'Wide'], duplicates='drop')
        df_s['stratum'] = df_s['vc_bin'].astype(str) + '/' + df_s['w_bin'].astype(str)
        
        known = df_s[df_s['quality_family'] != 'Unknown']
        grade_colors = {'1': '#4C72B0', '2': '#55A868', '3': '#C44E52', '5': '#8172B2'}
        for g in sorted(known['quality_family'].unique()):
            g_data = known[known['quality_family'] == g]
            strata = sorted(g_data['stratum'].dropna().unique())
            for st in strata:
                vals = g_data[g_data['stratum'] == st][target]
                if len(vals) >= 3:
                    fig_strat.add_trace(go.Box(
                        y=vals, name=f'Gr{g} / {st}',
                        marker_color=grade_colors.get(g, 'gray'),
                        opacity=0.6, boxpoints=False, showlegend=False),
                        row=1, col=i+1)

fig_strat.update_yaxes(title_text='ML \u03c3 [mm]')
fig_strat.update_layout(
    title='Fig 3.4b \u2014 Grade Effect AFTER Controlling for Casting Speed \u00d7 Mold Width<br>'
          '<sup>Each box = one grade within a Vc/Width stratum \u2014 grade differences persist</sup>',
    height=500, width=1300, template='plotly_white')
fig_strat.show()

# --- Fig 3.4c: SEN Depth variability vs ML \u03c3 ---
fig_sen = go.Figure()
for strand, color in colors_strand.items():
    sub = df_cmp_qf[df_cmp_qf['strand'] == strand]
    if sen_std_col in sub.columns:
        valid = sub.dropna(subset=[sen_std_col, target])
        r_s, p_s = sp_stats.spearmanr(valid[sen_std_col], valid[target])
        fig_sen.add_trace(go.Scatter(
            x=valid[sen_std_col], y=valid[target],
            mode='markers', name=f'{strand} (\u03c1={r_s:.3f})',
            marker=dict(size=4, color=color, opacity=0.4)))

fig_sen.update_layout(
    title='Fig 3.4c \u2014 SEN Depth Variability vs ML \u03c3<br>'
          '<sup>Higher SEN hunting \u2192 more mold level instability</sup>',
    xaxis_title='SEN Depth \u03c3 within 5-min window [mm]',
    yaxis_title='ML \u03c3 [mm]', height=450, width=800, template='plotly_white')
fig_sen.show()

# --- Fig 3.4d: CONFOUNDING EVIDENCE \u2014 EMBR DC vs ML \u03c3 colored by Vc\u00d7Width ---
fig_confound = make_subplots(rows=1, cols=2,
    subplot_titles=['S5: EMBR DC Master vs ML \u03c3', 'S6: EMBR DC Master vs ML \u03c3'],
    horizontal_spacing=0.12)

embr_dc_col = 'EMBR_DC_Master_LR_mean'  # V2.3: use direct name
vc_col = 'CASTING_SPEED_avg [m/min]'
w_col = 'MOLD_WIDTH_avg [m]'
regime_colors = {'Slow/Narrow': '#4C72B0', 'Slow/Wide': '#64B5CD',
                 'Mid/Narrow': '#8172B2', 'Mid/Wide': '#CCB974',
                 'Fast/Narrow': '#C44E52', 'Fast/Wide': '#DD8452'}
for i, (df_s, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    df_s = df_s.copy()
    df_s['vc_bin'] = pd.qcut(df_s[vc_col], q=3, labels=['Slow', 'Mid', 'Fast'], duplicates='drop')
    df_s['w_bin'] = pd.qcut(df_s[w_col], q=2, labels=['Narrow', 'Wide'], duplicates='drop')
    df_s['regime'] = df_s['vc_bin'].astype(str) + '/' + df_s['w_bin'].astype(str)
    for reg in sorted(df_s['regime'].dropna().unique()):
        sub = df_s[df_s['regime'] == reg]
        fig_confound.add_trace(go.Scatter(
            x=sub[embr_dc_col], y=sub[target],
            mode='markers', name=f'{reg} (n={len(sub)})',
            marker=dict(size=5, color=regime_colors.get(reg, 'gray'), opacity=0.5),
            showlegend=(i == 0)), row=1, col=i+1)
    fig_confound.update_xaxes(title_text=f'EMBR DC Master LR mean [A]', row=1, col=i+1)
fig_confound.update_yaxes(title_text='ML \u03c3 [mm]')
fig_confound.update_layout(
    title='Fig 3.4d \u2014 Confounding Evidence: EMBR DC vs ML \u03c3 by Vc\u00d7Width Regime<br>'
          '<sup>Colors show casting regime \u2014 EMBR effect is confounded with operating conditions</sup>',
    height=500, width=1300, template='plotly_white')
fig_confound.show()

# --- Partial correlation + Kruskal-Wallis ---
print('\nPartial correlation: quality_family \u2192 ML \u03c3 after controlling for Vc + Width')
for df_s, sname in [(df_seq_5, 'Strand 5'), (df_seq_6, 'Strand 6')]:
    vc_col = 'CASTING_SPEED_avg [m/min]'
    w_col = 'MOLD_WIDTH_avg [m]'
    if vc_col in df_s.columns and w_col in df_s.columns:
        known = df_s[df_s['quality_family'] != 'Unknown'].dropna(subset=[vc_col, w_col, target])
        if len(known) > 20:
            X_ctrl = known[[vc_col, w_col]].values
            y_res = known[target].values - LinearRegression().fit(X_ctrl, known[target].values).predict(X_ctrl)
            from scipy.stats import kruskal
            groups = [y_res[known['quality_family'].values == g]
                      for g in sorted(known['quality_family'].unique())
                      if (known['quality_family'].values == g).sum() >= 5]
            if len(groups) >= 2:
                H, p = kruskal(*groups)
                print(f'  {sname}: Kruskal-Wallis on residuals H={H:.1f}, p={p:.2e} '
                      f'(grade effect {"significant" if p < 0.01 else "not significant"} after controlling for Vc+Width)')


**Interpretation of Fig 3.4:**

**Steel Grade Families (Fig 3.4a).** The grade family code is derived from the first character of the TATA Steel internal grade designation (e.g., "3N92" → Family 3). Four families are present in the dataset. Family 3 (peritectic) shows the highest median ML σ, consistent with the known metallurgical challenge of peritectic grades producing more dynamic meniscus behaviour during the δ→γ phase transformation. Approximately 64% of sequences have no grade metadata ("Unknown").

**Controlled Analysis (Fig 3.4b).** To rule out confounding, we stratify by Vc×Width regime and show that grade differences **persist within each stratum**. The Kruskal-Wallis test on Vc+Width residuals (H=58–83, p<10⁻¹²) confirms this statistically.

**SEN Variability (Fig 3.4c).** The strongest individual correlation: Spearman ρ = 0.4–0.5, meaning higher SEN depth hunting within a 5-minute window systematically produces higher ML σ. This is the most actionable finding for process control.

**Confounding Evidence (Fig 3.4d).** The EMBR DC vs ML σ scatter, colored by Vc×Width regime, shows that the apparent EMBR-MLσ relationship is partly confounded with operating conditions. Different casting regimes occupy different EMBR regions, making it impossible to isolate the EMBR effect without stratification — this is why the multivariate SHAP analysis in Section 6 is essential.


**Interpretation of Fig 3.5 (Three-Category Meniscus Profiles):**

The reconstructed **meniscus profiles** (from Chebyshev polynomial fits to the FBG sensor data) use three stability categories based on ML σ quantiles, covering the **full spectrum** of mold level variability (no sequences are left out):

* **Stable (bottom 20%, blue):** Nearly flat profiles with minimal wave amplitude. The meniscus is effectively quiescent — the FC Mold G5 is maintaining a uniform temperature distribution across the mold width.
* **Medium (20–80%, olive):** Moderate wave structure begins to emerge, particularly in the X₂ (wave height) component. The profile shows a detectable standing wave pattern, but within tolerable limits.
* **Unstable (top 20%, coral):** Pronounced wave structure with clear asymmetry. The temperature difference between the peak and trough of the profile is significantly larger, indicating a strong standing wave in the mold.

> **Naming note (V2.3):** These are **meniscus profiles** in the true sense — they are reconstructed from Chebyshev polynomials (X₁–X₄) fitted to the FBG sensor array data, capturing the spatial temperature distribution across the mold width at the meniscus level. This is distinct from the raw “Broad Face FBG Temperature” (spatial average of all 12 sensors per face) shown in Section 3.1.

Fig 3.5b shows **where** on the mold the maximum wave amplitude occurs. Unstable sequences tend to have peaks near the narrow faces (±0.7–1.0 normalised width), consistent with a standing wave driven by the SEN jet.

In [0]:
# =====================================================================
# Fig 3.5: Reconstruct meniscus profile from Chebyshev coefficients
# V2.3: New eye-friendly color palette (navy/teal/coral)
#        Ensures both strands render with distinct figure titles
# =====================================================================

# Chebyshev polynomial basis functions
def cheb_T1(z): return z
def cheb_T2(z): return 2*z**2 - 1
def cheb_T3(z): return 4*z**3 - 3*z
def cheb_T4(z): return 8*z**4 - 8*z**2 + 1

def reconstruct_meniscus(X1, X2, X3, X4, n_points=200):
    x = np.linspace(-1, 1, n_points)
    fx = X1*cheb_T1(x) + X2*cheb_T2(x) + X3*cheb_T3(x) + X4*cheb_T4(x)
    return x, fx

def classify_position(x_pos):
    ax = abs(x_pos)
    if ax < 0.15: return 'SEN centre'
    elif ax < 0.4: return 'quarter-mold'
    elif ax < 0.7: return 'mid-to-NF'
    else: return 'near narrow face'

# V2.3: Eye-friendly color palette (avoids pure green/red/orange)
stab_palette = {
    'Stable':   '#2166AC',  # deep blue
    'Medium':   '#4DAC26',  # olive green
    'Unstable': '#D6604D',  # muted coral/terracotta
}

# Vuhz sensor positions
vuhz_pos_s5 = 265 / (1450/2)  # \u00b10.366
vuhz_pos_s6 = 265 / (1800/2)  # \u00b10.294

for df_s, sname, vuhz_pos in [(df_seq_5, 'Strand 5', vuhz_pos_s5), (df_seq_6, 'Strand 6', vuhz_pos_s6)]:
    n_pts = 200
    x_norm = np.linspace(-1, 1, n_pts)
    ml_sigma = df_s[target]

    # Three categories covering FULL ML sigma spectrum
    q_low = ml_sigma.quantile(0.20)
    q_high = ml_sigma.quantile(0.80)
    layers = [
        (f'Stable (ML\u03c3 < {q_low:.2f} mm)', lambda s: s < q_low, stab_palette['Stable']),
        (f'Medium ({q_low:.2f} \u2264 ML\u03c3 < {q_high:.2f} mm)', lambda s: (s >= q_low) & (s < q_high), stab_palette['Medium']),
        (f'Unstable (ML\u03c3 \u2265 {q_high:.2f} mm)', lambda s: s >= q_high, stab_palette['Unstable']),
    ]
    print(f'\n{sname} \u2014 Stability categories (V2.3):')
    for lname, lfn, _ in layers:
        n = lfn(ml_sigma).sum()
        print(f'  {lname}: n={n} ({100*n/len(ml_sigma):.1f}%)')

    # Reconstruct BFF and BFL profiles
    for face, face_label, cheb_prefix in [('bff', 'Fixed Face (BFF)', 'cheb_X'), ('bfl', 'Loose Face (BFL)', 'cheb_X')]:
        cheb_cols = [f'{cheb_prefix}1_{face}_mean', f'{cheb_prefix}2_{face}_mean',
                     f'{cheb_prefix}3_{face}_mean', f'{cheb_prefix}4_{face}_mean']
        if not all(c in df_s.columns for c in cheb_cols):
            print(f'{sname} {face_label}: Missing Chebyshev columns, skipping')
            continue

        profiles = np.zeros((len(df_s), n_pts))
        for i in range(len(df_s)):
            row = df_s.iloc[i]
            _, fx = reconstruct_meniscus(row[cheb_cols[0]], row[cheb_cols[1]],
                                         row[cheb_cols[2]], row[cheb_cols[3]], n_pts)
            profiles[i] = fx

        # Compute spatial features (only for BFF)
        if face == 'bff':
            ptp_max_idx = np.argmax(profiles, axis=1)
            ptp_min_idx = np.argmin(profiles, axis=1)
            df_s['profile_bff_ptp'] = np.ptp(profiles, axis=1)
            df_s['ptp_x_max'] = x_norm[ptp_max_idx]
            df_s['ptp_x_min'] = x_norm[ptp_min_idx]
            df_s['ptp_span'] = np.abs(df_s['ptp_x_max'] - df_s['ptp_x_min'])
            df_s['ptp_midpoint'] = (df_s['ptp_x_max'] + df_s['ptp_x_min']) / 2
            df_s['ptp_region_max'] = df_s['ptp_x_max'].apply(classify_position)
            df_s['ptp_region_min'] = df_s['ptp_x_min'].apply(classify_position)
            vuhz_idx_l = np.argmin(np.abs(x_norm - (-vuhz_pos)))
            vuhz_idx_r = np.argmin(np.abs(x_norm - vuhz_pos))
            df_s['vuhz_wave_height'] = np.abs(profiles[:, vuhz_idx_l] - profiles[:, vuhz_idx_r])

        # Fig 3.5a: Three-layer profile overlay
        fig = go.Figure()
        for lname, lfn, lcolor in layers:
            mask = lfn(ml_sigma)
            for idx in df_s[mask].index[:40]:
                i_loc = df_s.index.get_loc(idx)
                fig.add_trace(go.Scatter(x=x_norm, y=profiles[i_loc], mode='lines',
                    line=dict(color=lcolor, width=0.5), opacity=0.12, showlegend=False))
            if mask.sum() > 0:
                avg_prof = profiles[mask.values].mean(axis=0)
                std_prof = profiles[mask.values].std(axis=0)
                fig.add_trace(go.Scatter(x=x_norm, y=avg_prof, mode='lines',
                    line=dict(color=lcolor, width=3),
                    name=f'{lname} (n={mask.sum()})'))
                fig.add_trace(go.Scatter(
                    x=np.concatenate([x_norm, x_norm[::-1]]),
                    y=np.concatenate([avg_prof + std_prof, (avg_prof - std_prof)[::-1]]),
                    fill='toself', fillcolor=lcolor, opacity=0.1,
                    line=dict(width=0), showlegend=False))

        fig.add_vline(x=-vuhz_pos, line_dash='dash', line_color='#555555', opacity=0.6,
                      annotation_text='Vuhz L', annotation_position='top')
        fig.add_vline(x=vuhz_pos, line_dash='dash', line_color='#555555', opacity=0.6,
                      annotation_text='Vuhz R', annotation_position='top')
        fig.add_vline(x=0, line_dash='dot', line_color='gray', opacity=0.4,
                      annotation_text='SEN')

        fig.update_layout(
            title=f'Fig 3.5a \u2014 {sname}: Three-Category Meniscus Profiles ({face_label})<br>'
                  f'<sup>Stable (blue) | Medium (olive) | Unstable (coral) \u2014 \u00b11\u03c3 bands shown. '
                  f'Dashed lines = Vuhz sensor positions.</sup>',
            xaxis_title='Normalised Mold Width [-1 = NF left, 0 = SEN, +1 = NF right]',
            yaxis_title='Relative Meniscus Height (a.u.)', height=500, width=900,
            template='plotly_white')
        fig.show()

    # Fig 3.5b: PtP location scatter
    fig_loc = make_subplots(rows=1, cols=2,
        subplot_titles=['Peak (max) Location vs ML \u03c3', 'Trough (min) Location vs ML \u03c3'],
        horizontal_spacing=0.1)

    fig_loc.add_trace(go.Scatter(x=df_s['ptp_x_max'], y=df_s[target],
        mode='markers', marker=dict(size=4, color=df_s[target],
            colorscale='RdYlBu_r', cmin=0, cmax=3, showscale=True,
            colorbar=dict(title='ML\u03c3')),
        name='Peak pos', showlegend=False), row=1, col=1)
    fig_loc.add_trace(go.Scatter(x=df_s['ptp_x_min'], y=df_s[target],
        mode='markers', marker=dict(size=4, color=df_s[target],
            colorscale='RdYlBu_r', cmin=0, cmax=3, showscale=False),
        name='Trough pos', showlegend=False), row=1, col=2)

    for col in [1, 2]:
        fig_loc.add_vline(x=-vuhz_pos, line_dash='dash', line_color='#555555', opacity=0.4, row=1, col=col)
        fig_loc.add_vline(x=vuhz_pos, line_dash='dash', line_color='#555555', opacity=0.4, row=1, col=col)
        fig_loc.add_vline(x=0, line_dash='dot', line_color='gray', opacity=0.3, row=1, col=col)
    fig_loc.update_xaxes(title_text='Normalised Mold Position')
    fig_loc.update_yaxes(title_text='ML \u03c3 [mm]')

    fig_loc.update_layout(
        title=f'Fig 3.5b \u2014 {sname}: Where on the Mold Does the Maximum Wave Occur?<br>'
              f'<sup>Each dot = one 5-min sequence. Dashed lines = Vuhz sensor positions. '
              f'Peaks near narrow faces associate with higher ML \u03c3.</sup>',
        height=450, width=1000, template='plotly_white')
    fig_loc.show()

---
## 4. Temporal Dynamics: Meniscus Shape Change Precedes Mold Level Excursion

Having established which parameters correlate with instability, we now ask **how** these relationships play out in time. Three questions:

1. **Does mold level stability degrade across a casting campaign?** (Heat progression)
2. **Does the meniscus shape change precede or follow mold level excursions?** (Lagged cross-correlation)
3. **Does meniscus shape Granger-cause mold level instability?** (Formal statistical causality test)

The answers differ between strands — Strand 6 shows a clear **4-second lead** of shape over level, while Strand 5 shows weaker and more ambiguous timing. This strand difference is itself informative: it suggests the meniscus dynamics depend on local conditions (possibly SEN alignment or mold geometry) rather than being universal.

**Key result (preview):** On Strand 6, Chebyshev X₂ (wave height) changes significantly precede mold level deviation (Granger F=210 at lag 4s). The forward direction (shape→level) dominates over the reverse, confirming feedforward control potential.

In [0]:
# =====================================================================
# Fig 4.1: Heat-count progression — does ML σ degrade across a campaign?
# Adapted from EDA notebook cell 146 — dual-strand version
# =====================================================================

for df_s, df_raw, sname, color in [
    (df_seq_5, df_raw_5, 'Strand 5', '#e74c3c'),
    (df_seq_6, df_raw_6, 'Strand 6', '#3498db')
]:
    # Assign sequences to heat number by timestamp ordering
    df_s_sorted = df_s.sort_values('Seq_time_Start').copy()
    
    # Detect casting campaign boundaries (gaps > 30 min in sequence timestamps)
    dt = df_s_sorted['Seq_time_Start'].diff().dt.total_seconds().fillna(0)
    df_s_sorted['campaign'] = (dt > 1800).cumsum()  # >30min gap = new campaign
    
    # Within each campaign, assign sequential heat position
    df_s_sorted['heat_position'] = df_s_sorted.groupby('campaign').cumcount() + 1
    
    # Per-campaign ML σ progression
    fig_heat = go.Figure()
    campaigns = df_s_sorted['campaign'].unique()
    
    for camp in campaigns:
        sub = df_s_sorted[df_s_sorted['campaign'] == camp]
        if len(sub) < 5:
            continue
        fig_heat.add_trace(go.Scatter(
            x=sub['heat_position'], y=sub[target],
            mode='lines+markers', marker=dict(size=3, color=color, opacity=0.3),
            line=dict(width=1, color=color), opacity=0.4,
            name=f'Campaign {camp}', showlegend=False))
    
    # Overall trend with rolling average
    df_all = df_s_sorted.copy()
    df_all['heat_pos_global'] = range(1, len(df_all)+1)
    rolling = df_all.set_index('heat_pos_global')[target].rolling(20, center=True).median()
    fig_heat.add_trace(go.Scatter(x=rolling.index, y=rolling.values,
        mode='lines', line=dict(color='black', width=3),
        name='Rolling median (20-seq)'))
    
    # Add ±1mm threshold
    fig_heat.add_hline(y=ML_THRESHOLD, line_dash='dash', line_color='red', opacity=0.5,
                       annotation_text='±1 mm')
    
    fig_heat.update_layout(
        title=f'Fig 4.1 — {sname}: ML σ Across Casting Campaigns<br>'
              f'<sup>Each trace = one casting campaign | Black = rolling median</sup>',
        xaxis_title='Sequence Position (within campaign / global)',
        yaxis_title='ML σ [mm]', height=450, width=1000)
    fig_heat.show()
    
    # Trend test
    r_s, p_s = sp_stats.spearmanr(df_all['heat_pos_global'], df_all[target])
    print(f'{sname} global trend: Spearman ρ = {r_s:.4f} (p = {p_s:.2e}) — '
          f'{"significant" if p_s < 0.05 else "not significant"} trend')

In [0]:
# =====================================================================
# Fig 4.1b: Exploratory view of rolling statistics BEFORE cross-correlation
# V2.3: Shows the raw time-series of rolling Chebyshev std and ML absolute
# deviation that feed into the cross-correlation in Fig 4.2.
# This helps the reader understand what "shape change precedes level change"
# looks like in practice, before the robust statistical summary.
# =====================================================================

win = 10  # rolling window for smoothing

# Show Strand 6 as the primary example (strongest temporal signal)
df_lag = df_fc_6[['plainTimeStamp', 'Mold Level', 'cheb_X2_bff', 'cheb_X1_bff']].dropna().sort_values('plainTimeStamp')
dt = df_lag['plainTimeStamp'].diff().dt.total_seconds().fillna(0)
df_lag['segment'] = (dt > 5).cumsum()
best_seg = df_lag.groupby('segment').size().idxmax()
df_seg_expl = df_lag[df_lag['segment'] == best_seg].reset_index(drop=True)

# Compute rolling statistics
df_seg_expl['ML_abs_dev'] = (df_seg_expl['Mold Level'] - df_seg_expl['Mold Level'].rolling(win, center=True).mean()).abs()
df_seg_expl['X2_rolling_std'] = df_seg_expl['cheb_X2_bff'].rolling(win, center=True).std()
df_seg_expl['X1_rolling_std'] = df_seg_expl['cheb_X1_bff'].rolling(win, center=True).std()

# Show a 10-minute window where both signals are interesting
mid = len(df_seg_expl) // 4
window = df_seg_expl.iloc[mid:mid+600].copy()  # 10 minutes

fig_expl = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=['ML Absolute Deviation (10s rolling)',
                    'X\u2082 Wave Height \u03c3 (10s rolling)',
                    'X\u2081 Asymmetry \u03c3 (10s rolling)'])

t = window['plainTimeStamp']
fig_expl.add_trace(go.Scatter(x=t, y=window['ML_abs_dev'],
    name='ML |dev|', line=dict(width=1.2, color='#2c3e50')), row=1, col=1)
fig_expl.add_trace(go.Scatter(x=t, y=window['X2_rolling_std'],
    name='X\u2082 \u03c3 (wave)', line=dict(width=1.2, color='#D6604D')), row=2, col=1)
fig_expl.add_trace(go.Scatter(x=t, y=window['X1_rolling_std'],
    name='X\u2081 \u03c3 (asym)', line=dict(width=1.2, color='#4C72B0')), row=3, col=1)

fig_expl.update_yaxes(title_text='|ML dev| [mm]', row=1, col=1)
fig_expl.update_yaxes(title_text='X\u2082 \u03c3', row=2, col=1)
fig_expl.update_yaxes(title_text='X\u2081 \u03c3', row=3, col=1)
fig_expl.update_xaxes(title_text='Time', row=3, col=1)

fig_expl.update_layout(
    title='Fig 4.1b \u2014 Strand 6: Exploratory View of Rolling Chebyshev Statistics vs ML Deviation<br>'
          '<sup>These three time-series are the inputs to the cross-correlation (Fig 4.2). '
          'Visual inspection suggests X\u2082 bursts often precede ML deviation spikes by a few seconds.</sup>',
    height=600, width=1200, template='plotly_white')
fig_expl.show()

print(f'\nSegment length: {len(df_seg_expl):,} samples ({len(df_seg_expl)/60:.0f} min)')
print(f'Shown window: {len(window)} samples (10 min)')
print('\nMethodological note: The cross-correlation in Fig 4.2 computes the Pearson r')
print('between these rolling statistics at various time lags (\u00b130s). A positive peak lag')
print('means the Chebyshev statistic changes BEFORE the mold level reacts \u2014 indicating')
print('feedforward potential for the FC Mold G5 system.')

In [0]:
# =====================================================================
# Fig 4.2: Lagged cross-correlation — Chebyshev X1-X4 shape vs Mold Level
# FB8: Extended to 4 subplots (X1-X4), both BFF and BFL faces
# Positive lag = shape change PRECEDES mold level excursion
# =====================================================================
from scipy import signal as sp_signal

def compute_xcorr(x, y, max_lag):
    x_n = (x - x.mean()) / x.std()
    y_n = (y - y.mean()) / y.std()
    n = len(x_n)
    lags = range(-max_lag, max_lag + 1)
    xcorr = []
    for lag in lags:
        if lag >= 0:
            r = np.corrcoef(x_n[:n-lag], y_n[lag:])[0, 1] if n-lag > 10 else 0
        else:
            r = np.corrcoef(x_n[-lag:], y_n[:n+lag])[0, 1] if n+lag > 10 else 0
        xcorr.append(r)
    return list(lags), xcorr

max_lag = 30  # seconds
win = 10  # rolling window for smoothing

cheb_pairs = [
    ('cheb_X1_bff', 'X₁ (asymmetry/tilt)'),
    ('cheb_X2_bff', 'X₂ (wave height)'),
    ('cheb_X3_bff', 'X₃ (skewness)'),
    ('cheb_X4_bff', 'X₄ (bimodality)'),
]

for df_fc, sname, color in [(df_fc_5, 'Strand 5', '#e74c3c'), (df_fc_6, 'Strand 6', '#3498db')]:
    # Prepare columns
    cols_needed = ['plainTimeStamp', 'Mold Level'] + [p[0] for p in cheb_pairs]
    avail = [c for c in cols_needed if c in df_fc.columns]
    df_lag = df_fc[avail].copy().dropna()
    df_lag = df_lag.sort_values('plainTimeStamp').reset_index(drop=True)
    
    # Segment by time gaps > 5s
    dt = df_lag['plainTimeStamp'].diff().dt.total_seconds().fillna(0)
    df_lag['segment'] = (dt > 5).cumsum()
    best_seg = df_lag.groupby('segment').size().idxmax()
    df_seg = df_lag[df_lag['segment'] == best_seg].reset_index(drop=True)
    
    # Rolling statistics
    df_seg['ML_abs_dev'] = (df_seg['Mold Level'] - df_seg['Mold Level'].rolling(win, center=True).mean()).abs()
    for col, _ in cheb_pairs:
        if col in df_seg.columns:
            df_seg[f'{col}_rolling_std'] = df_seg[col].rolling(win, center=True).std()
    df_clean = df_seg.dropna()
    
    print(f'\n{sname}: Longest continuous segment = {len(df_seg):,} samples ({len(df_seg)/60:.1f} min)')
    
    # 2x2 subplot: one per Chebyshev coefficient
    fig_xcorr = make_subplots(rows=2, cols=2,
        subplot_titles=[lab for _, lab in cheb_pairs],
        vertical_spacing=0.15, horizontal_spacing=0.12)
    
    for idx, (col, label) in enumerate(cheb_pairs):
        row, col_idx = idx // 2 + 1, idx % 2 + 1
        roll_col = f'{col}_rolling_std'
        if roll_col not in df_clean.columns:
            continue
        
        # BFF
        lags, xcorr = compute_xcorr(df_clean[roll_col].values, df_clean['ML_abs_dev'].values, max_lag)
        peak_lag = lags[np.argmax(xcorr)]
        peak_r = max(xcorr)
        fig_xcorr.add_trace(go.Scatter(x=lags, y=xcorr, mode='lines',
            name=f'BFF (peak {peak_lag}s, r={peak_r:.3f})', line=dict(color=color),
            showlegend=(idx==0)), row=row, col=col_idx)
        
        # BFL (if available)
        bfl_col = col.replace('bff', 'bfl')
        bfl_roll = f'{bfl_col}_rolling_std'
        if bfl_col in df_seg.columns:
            if bfl_roll not in df_seg.columns:
                df_seg[bfl_roll] = df_seg[bfl_col].rolling(win, center=True).std()
            bfl_clean = df_seg.dropna(subset=['ML_abs_dev', bfl_roll])
            lags_b, xcorr_b = compute_xcorr(bfl_clean[bfl_roll].values, bfl_clean['ML_abs_dev'].values, max_lag)
            peak_lag_b = lags_b[np.argmax(xcorr_b)]
            peak_r_b = max(xcorr_b)
            fig_xcorr.add_trace(go.Scatter(x=lags_b, y=xcorr_b, mode='lines',
                name=f'BFL (peak {peak_lag_b}s, r={peak_r_b:.3f})',
                line=dict(color=color, dash='dash'), showlegend=(idx==0)),
                row=row, col=col_idx)
        
        fig_xcorr.add_vline(x=0, line_dash='dot', line_color='gray', opacity=0.5, row=row, col=col_idx)
        fig_xcorr.update_yaxes(title_text='r' if col_idx==1 else '', row=row, col=col_idx)
        fig_xcorr.update_xaxes(title_text='Lag [s]' if row==2 else '', row=row, col=col_idx)
        print(f'  {label}: BFF peak at lag={peak_lag}s (r={peak_r:.3f})')
    
    fig_xcorr.update_layout(
        title=f'Fig 4.2 — {sname}: Cross-Correlation of Chebyshev Shape (σ) vs ML Deviation<br>'
              f'<sup>Positive lag = shape change PRECEDES mold level excursion. Solid=BFF, Dashed=BFL.</sup>',
        height=600, width=1100, template='plotly_white')
    fig_xcorr.show()

In [0]:
# =====================================================================
# Fig 4.2b: Exploratory scatter + lag illustration before Granger test
# V2.3: Shows the scatter relationship and explains how Granger causality
# works conceptually before presenting the formal results in Fig 4.3.
# =====================================================================

# Reuse the rolling data from the xcorr analysis
for df_fc, sname, color in [(df_fc_6, 'Strand 6', '#DD8452')]:
    df_lag = df_fc[['plainTimeStamp', 'Mold Level', 'cheb_X2_bff', 'cheb_X1_bff']].dropna().sort_values('plainTimeStamp')
    dt = df_lag['plainTimeStamp'].diff().dt.total_seconds().fillna(0)
    df_lag['segment'] = (dt > 5).cumsum()
    best_seg = df_lag.groupby('segment').size().idxmax()
    df_seg_gc = df_lag[df_lag['segment'] == best_seg].reset_index(drop=True)
    
    win = 10
    df_seg_gc['ML_abs_dev'] = (df_seg_gc['Mold Level'] - df_seg_gc['Mold Level'].rolling(win, center=True).mean()).abs()
    df_seg_gc['X2_std'] = df_seg_gc['cheb_X2_bff'].rolling(win, center=True).std()
    df_seg_gc['X1_std'] = df_seg_gc['cheb_X1_bff'].rolling(win, center=True).std()
    df_gc_clean = df_seg_gc.dropna()
    
    # --- Scatter: X2 std vs ML deviation (contemporaneous) ---
    fig_scatter = make_subplots(rows=1, cols=3,
        subplot_titles=['X\u2082 \u03c3 vs ML deviation (lag=0)',
                        'X\u2082 \u03c3(t-4) vs ML deviation(t)',
                        'X\u2081 \u03c3 vs ML deviation (lag=0)'],
        horizontal_spacing=0.08)
    
    sub = df_gc_clean.sample(min(5000, len(df_gc_clean)), random_state=42)
    
    # Panel 1: contemporaneous
    fig_scatter.add_trace(go.Scatter(
        x=sub['X2_std'], y=sub['ML_abs_dev'],
        mode='markers', marker=dict(size=2, color='#DD8452', opacity=0.3),
        showlegend=False), row=1, col=1)
    fig_scatter.update_xaxes(title_text='X\u2082 \u03c3 (t)', row=1, col=1)
    fig_scatter.update_yaxes(title_text='|ML dev| (t)', row=1, col=1)
    
    # Panel 2: lagged by 4 seconds (the peak from xcorr)
    x2_lagged = df_gc_clean['X2_std'].shift(4).dropna()
    ml_aligned = df_gc_clean['ML_abs_dev'].iloc[4:].reset_index(drop=True)
    common_len = min(len(x2_lagged), len(ml_aligned))
    sub2 = pd.DataFrame({'x2_lag4': x2_lagged.values[:common_len], 'ml': ml_aligned.values[:common_len]})
    sub2_s = sub2.sample(min(5000, len(sub2)), random_state=42)
    fig_scatter.add_trace(go.Scatter(
        x=sub2_s['x2_lag4'], y=sub2_s['ml'],
        mode='markers', marker=dict(size=2, color='#C44E52', opacity=0.3),
        showlegend=False), row=1, col=2)
    fig_scatter.update_xaxes(title_text='X\u2082 \u03c3 (t\u22124s)', row=1, col=2)
    fig_scatter.update_yaxes(title_text='|ML dev| (t)', row=1, col=2)
    
    # Panel 3: X1 contemporaneous
    fig_scatter.add_trace(go.Scatter(
        x=sub['X1_std'], y=sub['ML_abs_dev'],
        mode='markers', marker=dict(size=2, color='#4C72B0', opacity=0.3),
        showlegend=False), row=1, col=3)
    fig_scatter.update_xaxes(title_text='X\u2081 \u03c3 (t)', row=1, col=3)
    fig_scatter.update_yaxes(title_text='|ML dev| (t)', row=1, col=3)
    
    from scipy.stats import spearmanr
    r0, _ = spearmanr(df_gc_clean['X2_std'], df_gc_clean['ML_abs_dev'])
    r4, _ = spearmanr(sub2['x2_lag4'], sub2['ml'])
    r1, _ = spearmanr(df_gc_clean['X1_std'], df_gc_clean['ML_abs_dev'])
    
    fig_scatter.update_layout(
        title=f'Fig 4.2b \u2014 {sname}: Exploratory Scatter of Shape Statistics vs ML Deviation<br>'
              f'<sup>Left: X\u2082 vs ML at same time (\u03c1={r0:.3f}). '
              f'Centre: X\u2082 shifted 4s earlier (\u03c1={r4:.3f} \u2014 tighter = shape leads level). '
              f'Right: X\u2081 vs ML (\u03c1={r1:.3f}).</sup>',
        height=400, width=1400, template='plotly_white')
    fig_scatter.show()
    
    print(f'\n{sname} exploratory statistics:')
    print(f'  Spearman \u03c1(X\u2082, ML) at lag=0:  {r0:.4f}')
    print(f'  Spearman \u03c1(X\u2082, ML) at lag=-4s: {r4:.4f}')
    print(f'  Spearman \u03c1(X\u2081, ML) at lag=0:  {r1:.4f}')
    print('\n  Methodological note: Granger causality (Fig 4.3) formalises this by testing')
    print('  whether PAST values of X improve the prediction of ML BEYOND what ML\'s own')
    print('  past already explains. It is a regression-based test, not a correlation.')

In [0]:
# =====================================================================
# Fig 4.3: Granger causality — F-stat across lags (forward vs reverse)
# FB9: Extended with forward/reverse comparison to show causal direction
# =====================================================================
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

max_test_lag = 15

for df_fc, sname in [(df_fc_5, 'Strand 5'), (df_fc_6, 'Strand 6')]:
    # Prepare continuous segment
    df_lag = df_fc[['plainTimeStamp', 'Mold Level',
                    'cheb_X1_bff', 'cheb_X2_bff']].dropna().sort_values('plainTimeStamp')
    dt = df_lag['plainTimeStamp'].diff().dt.total_seconds().fillna(0)
    df_lag['segment'] = (dt > 5).cumsum()
    best_seg = df_lag.groupby('segment').size().idxmax()
    df_seg = df_lag[df_lag['segment'] == best_seg].reset_index(drop=True)
    
    win = 10
    df_seg['ML_abs_dev'] = (df_seg['Mold Level'] - df_seg['Mold Level'].rolling(win, center=True).mean()).abs()
    df_seg['X2_bff_rolling_std'] = df_seg['cheb_X2_bff'].rolling(win, center=True).std()
    df_seg['X1_bff_rolling_std'] = df_seg['cheb_X1_bff'].rolling(win, center=True).std()
    df_gc = df_seg[['ML_abs_dev', 'X2_bff_rolling_std', 'X1_bff_rolling_std']].dropna().iloc[:50000]
    
    print(f'\n{"="*80}')
    print(f'{sname} — GRANGER CAUSALITY TESTS (forward vs reverse)')
    print(f'H0: X does NOT Granger-cause Y. Reject if p < 0.05.')
    print(f'{"="*80}')
    
    # All 4 test directions
    test_pairs = [
        ('X2_bff_rolling_std', 'ML_abs_dev', 'X₂ wave σ → ML deviation', 'forward'),
        ('X1_bff_rolling_std', 'ML_abs_dev', 'X₁ asym σ → ML deviation', 'forward'),
        ('ML_abs_dev', 'X2_bff_rolling_std', 'ML deviation → X₂ wave σ', 'reverse'),
        ('ML_abs_dev', 'X1_bff_rolling_std', 'ML deviation → X₁ asym σ', 'reverse'),
    ]
    
    gc_all = {}  # store F-stats across lags for plotting
    for cause_col, effect_col, label, direction in test_pairs:
        data = df_gc[[effect_col, cause_col]].dropna()
        if len(data) < max_test_lag * 3:
            continue
        try:
            gc = grangercausalitytests(data, maxlag=max_test_lag, verbose=False)
            f_stats = [gc[lag][0]['ssr_ftest'][0] for lag in range(1, max_test_lag+1)]
            p_vals = [gc[lag][0]['ssr_ftest'][1] for lag in range(1, max_test_lag+1)]
            best_idx = np.argmin(p_vals)
            best_lag = best_idx + 1
            best_f = f_stats[best_idx]
            best_p = p_vals[best_idx]
            sig = '***' if best_p < 0.001 else ('**' if best_p < 0.01 else ('*' if best_p < 0.05 else 'n.s.'))
            gc_all[label] = {'f_stats': f_stats, 'p_vals': p_vals, 'direction': direction}
            print(f'  {label}: best lag={best_lag}s, F={best_f:.1f}, p={best_p:.2e} {sig}')
        except Exception as e:
            print(f'  {label}: Error - {e}')
    
    # --- Fig 4.3a: F-stat comparison (forward vs reverse) ---
    fig_gc = make_subplots(rows=1, cols=2,
        subplot_titles=['X₂ (wave height): Forward vs Reverse',
                        'X₁ (asymmetry): Forward vs Reverse'],
        horizontal_spacing=0.12)
    
    lags_x = list(range(1, max_test_lag+1))
    col_map = {
        'X₂ wave σ → ML deviation': (1, 'Shape → Level (causal)', 'solid', '#2ecc71'),
        'ML deviation → X₂ wave σ': (1, 'Level → Shape (reverse)', 'dash', '#e74c3c'),
        'X₁ asym σ → ML deviation': (2, 'Shape → Level (causal)', 'solid', '#2ecc71'),
        'ML deviation → X₁ asym σ': (2, 'Level → Shape (reverse)', 'dash', '#e74c3c'),
    }
    
    for label, info in gc_all.items():
        if label in col_map:
            col_idx, trace_name, dash, clr = col_map[label]
            fig_gc.add_trace(go.Scatter(x=lags_x, y=info['f_stats'], mode='lines+markers',
                name=trace_name, line=dict(color=clr, dash=dash),
                marker=dict(size=4), showlegend=(col_idx==1)),
                row=1, col=col_idx)
    
    for col_idx in [1, 2]:
        fig_gc.update_xaxes(title_text='Lag [seconds]', row=1, col=col_idx)
        fig_gc.update_yaxes(title_text='F-statistic' if col_idx==1 else '', row=1, col=col_idx)
    
    fig_gc.update_layout(
        title=f'Fig 4.3 — {sname}: Granger Causality F-Statistic Across Lags<br>'
              f'<sup>Green (forward) >> Red (reverse) = meniscus shape Granger-causes ML instability</sup>',
        height=450, width=1100, template='plotly_white')
    fig_gc.show()
    
    # --- Fig 4.3b: p-value plot ---
    fig_pv = go.Figure()
    fwd_labels = [l for l, i in gc_all.items() if i['direction'] == 'forward']
    for label in fwd_labels:
        fig_pv.add_trace(go.Scatter(x=lags_x, y=gc_all[label]['p_vals'],
            mode='lines+markers', name=label))
    fig_pv.add_hline(y=0.05, line_dash='dash', line_color='red', opacity=0.5,
                     annotation_text='p=0.05')
    fig_pv.add_hline(y=0.001, line_dash='dot', line_color='red', opacity=0.3,
                     annotation_text='p=0.001')
    fig_pv.update_layout(
        title=f'Fig 4.3b — {sname}: Granger p-values (forward direction only)',
        xaxis_title='Lag [seconds]', yaxis_title='p-value (F-test)',
        yaxis_type='log', height=400, width=800, template='plotly_white')
    fig_pv.show()


### 4.4 Interpretation: What the Temporal Analysis Reveals

The lagged cross-correlation and Granger causality tests reveal a **strand-dependent** temporal picture:

**Strand 6 — Strong directional coupling.** The cross-correlation peaks at **lag = +4 s** (r = 0.117) for X₂ wave height, meaning meniscus shape changes **precede** mold level excursions by ~4 seconds. Granger causality strongly confirms this: X₂ → ML deviation is overwhelmingly significant (F = 210, p = 2×10⁻⁴⁷), while the reverse direction (ML → X₂) is also significant but **12× weaker** (F = 17, p = 3×10⁻⁸). The dominant causal direction is clear: meniscus shape disturbances *drive* mold level instability, not the other way around. This means the FBG-based meniscus profile is a genuine **leading indicator** of impending mold level excursions on Strand 6.

**Strand 5 — Weak / inconclusive coupling.** The cross-correlation is essentially flat (r ≈ 0.03 at all lags — no detectable peak above noise). Granger causality for X₂ → ML is only marginally significant (F = 4.8, p = 0.028). The temporal coupling between meniscus shape and mold level on Strand 5 is too weak in this dataset to draw a firm directional conclusion. This does **not** mean the physical coupling doesn't exist — it may reflect the fact that the longest continuous S5 segment was relatively stable, offering less dynamic range to detect lead/lag relationships.

**Implication for the FC Mold G5:** On Strand 6, the data strongly supports using the FBG meniscus profile as a **feedforward signal** for the electromagnetic control loop — shape disturbances arrive 3–5 seconds before the Vuhz-measured mold level responds. The FC Mold G5 could potentially act earlier if it incorporated this leading indicator. On Strand 5, the evidence is insufficient to make the same claim from this dataset, though the physical mechanism is the same.

---
## 6. Mold Level Fluctuation Is Predictable and Explained by Key Casting Parameters

### 6.1 Model Design

**Objective:** Predict mold level instability (ML σ, in mm) from per-sequence process features, then use SHAP to identify the dominant controllable drivers.

**Algorithm:** XGBoost gradient boosted trees (500 estimators, learning rate 0.05, max depth 6). 5-fold cross-validation with out-of-fold (OOF) predictions to prevent overfitting.

**Features:** 21 physics-informed features per sequence: casting speed (μ, σ), mold width (μ, σ), SEN depth (μ, σ), argon flow, mold level PtP, Chebyshev coefficients (X₁–X₄ for BFF and BFL: μ and σ), meniscus BFF-BFL asymmetry (μ, σ), and ML left-right asymmetry (μ, σ).

**Why XGBoost?** Non-linear interactions (e.g., SEN variability may matter more at high casting speeds) are captured automatically. SHAP provides per-feature importance that accounts for multicollinearity.

**Key result (preview):** R² ≈ 0.95 on both strands — the 21 features explain nearly all sequence-level ML σ variation. The feature hierarchy is consistent: PtP > SENσ > meniscus shape > argon > EMBR.

In [0]:
import xgboost as xgb
import shap
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# =====================================================================
# Feature selection (identical for both strands)
# FB6: Now includes SEN_type + quality_family (one-hot encoded)
# =====================================================================
exclude = [
    'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
    'Quality casting', 'castMode',
    target, 'MOLD_LEVEL_avg [mm]', 'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]',
    'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
    'has_high_variability', 'has_disturbance', 'disturbance_type',
    # EMBR excluded (confounded with Vc/width — see Section 3.3)
    'AC_Left_Master_avg [A]', 'DC_Left_Master_avg [A]', 'DC_Left_Bottom_avg [A]',
    'AC_Right_Master_avg [A]', 'DC_Right_Master_avg [A]', 'DC_Right_Bottom_avg [A]',
    'AC_Left_Master_std [A]', 'DC_Left_Master_std [A]', 'DC_Left_Bottom_std [A]',
    'AC_Right_Master_std [A]', 'DC_Right_Master_std [A]', 'DC_Right_Bottom_std [A]',
    'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
    'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
    'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
    'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
    'quality_family',  # will be one-hot encoded separately
]

def train_and_explain(df_seq, strand_name):
    # Numeric features (same as before)
    feat_cols = [c for c in df_seq.columns if c not in exclude
                 and df_seq[c].dtype in ['float64', 'int64', 'float32', 'int32']]
    
    # SEN_type is already numeric (-1, 1, 2) and included via feat_cols
    
    # One-hot encode quality_family (FB6)
    cat_cols = []
    df_work = df_seq[feat_cols + [target]].copy()
    if 'quality_family' in df_seq.columns:
        dummies = pd.get_dummies(df_seq['quality_family'], prefix='grade', drop_first=False)
        for dc in dummies.columns:
            df_work[dc] = dummies[dc].values
            cat_cols.append(dc)
    
    all_feat_cols = feat_cols + cat_cols
    df_m = df_work[all_feat_cols + [target]].dropna().reset_index(drop=True)
    X = df_m[all_feat_cols].values
    y = df_m[target].values
    
    print(f'  {strand_name}: {len(all_feat_cols)} features ({len(feat_cols)} numeric + {len(cat_cols)} grade dummies)')

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for _, (tr, va) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42,
            n_jobs=-1, verbosity=0, tree_method='hist')
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = m.predict(X[va])

    r2 = r2_score(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    mae = mean_absolute_error(y, oof)
    print(f'  {strand_name}: R\u00b2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}')

    final = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        n_jobs=-1, verbosity=0, tree_method='hist')
    final.fit(X, y, verbose=False)
    explainer = shap.TreeExplainer(final)
    sv = explainer.shap_values(X)
    shap_imp = pd.DataFrame({'feature': all_feat_cols,
        'mean_abs_shap': np.abs(sv).mean(axis=0)}).sort_values('mean_abs_shap', ascending=False)

    return {'r2': r2, 'rmse': rmse, 'mae': mae, 'feat_cols': all_feat_cols,
            'model': final, 'shap_values': sv, 'shap_imp': shap_imp,
            'X': X, 'X_df': pd.DataFrame(X, columns=all_feat_cols), 'y': y,
            'oof': oof, 'n_numeric': len(feat_cols), 'n_cat': len(cat_cols)}

print('Training XGBoost + SHAP (5-fold CV, now with SEN_type + grade dummies)...')
print('=' * 60)
res_5 = train_and_explain(df_seq_5, 'Strand 5')
res_6 = train_and_explain(df_seq_6, 'Strand 6')
print('=' * 60)
print(f'\nTop-5 SHAP features (Strand 5):')
for _, r in res_5['shap_imp'].head(5).iterrows():
    print(f'  {r["feature"]}: |SHAP|={r["mean_abs_shap"]:.4f}')
print(f'\nTop-5 SHAP features (Strand 6):')
for _, r in res_6['shap_imp'].head(5).iterrows():
    print(f'  {r["feature"]}: |SHAP|={r["mean_abs_shap"]:.4f}')

In [0]:
# =====================================================================
# Fig 6.2: Side-by-side SHAP feature importance
# =====================================================================
top_n = 15
imp_5 = res_5['shap_imp'].head(top_n).reset_index(drop=True)
imp_6 = res_6['shap_imp'].head(top_n).reset_index(drop=True)

all_top = list(dict.fromkeys(imp_5['feature'].tolist() + imp_6['feature'].tolist()))
merged = pd.DataFrame({'feature': all_top})
merged = merged.merge(imp_5.rename(columns={'mean_abs_shap': 'S5'}), on='feature', how='left')
merged = merged.merge(imp_6.rename(columns={'mean_abs_shap': 'S6'}), on='feature', how='left')
merged = merged.fillna(0)
merged['avg'] = (merged['S5'] + merged['S6']) / 2
merged = merged.sort_values('avg', ascending=True).tail(top_n)

fig = go.Figure()
fig.add_trace(go.Bar(y=merged['feature'], x=merged['S5'], name='Strand 5',
    orientation='h', marker_color='#e74c3c', opacity=0.7))
fig.add_trace(go.Bar(y=merged['feature'], x=merged['S6'], name='Strand 6',
    orientation='h', marker_color='#3498db', opacity=0.7))
fig.update_layout(
    title=f'Fig 6.2 — SHAP Feature Importance: Top {top_n}<br>'
          f'<sup>S5: R\u00b2={res_5["r2"]:.3f} | S6: R\u00b2={res_6["r2"]:.3f}</sup>',
    xaxis_title='Mean |SHAP value|', yaxis_title='',
    barmode='group', height=600, width=900)
fig.show()

# Print ranking table
print(f"\n{'Rank':>4}  {'Strand 5':<38} {'|SHAP|':>8}   {'Strand 6':<38} {'|SHAP|':>8}")
print('-'*105)
for i in range(top_n):
    f5 = imp_5.iloc[i]['feature'] if i < len(imp_5) else ''
    v5 = imp_5.iloc[i]['mean_abs_shap'] if i < len(imp_5) else 0
    f6 = imp_6.iloc[i]['feature'] if i < len(imp_6) else ''
    v6 = imp_6.iloc[i]['mean_abs_shap'] if i < len(imp_6) else 0
    print(f"  {i+1:>2}   {f5:<38} {v5:>8.4f}   {f6:<38} {v6:>8.4f}")

In [0]:
# =====================================================================
# Fig 6.3: Actual vs Predicted (OOF) for both strands
# =====================================================================
fig_pred = make_subplots(rows=1, cols=2,
    subplot_titles=[f'Strand 5 (R\u00b2={res_5["r2"]:.3f})', f'Strand 6 (R\u00b2={res_6["r2"]:.3f})'],
    horizontal_spacing=0.1)

for i, (res, strand, color) in enumerate([(res_5, 'S5', '#e74c3c'), (res_6, 'S6', '#3498db')]):
    col = i + 1
    fig_pred.add_trace(go.Scatter(x=res['y'], y=res['oof'], mode='markers',
        marker=dict(size=4, color=color, opacity=0.4), name=strand,
        showlegend=False), row=1, col=col)
    lim = max(res['y'].max(), res['oof'].max()) * 1.05
    fig_pred.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode='lines',
        line=dict(color='black', dash='dash', width=1), showlegend=False), row=1, col=col)
    fig_pred.update_xaxes(title_text='Actual ML \u03c3 [mm]', range=[0, lim], row=1, col=col)
    fig_pred.update_yaxes(title_text='Predicted ML \u03c3 [mm]', range=[0, lim], row=1, col=col)

fig_pred.update_layout(title='Fig 6.3 — Actual vs Predicted (5-fold OOF)',
                       height=500, width=1100)
fig_pred.show()

   
### 6.4 SHAP Feature Interpretation — Physics Meaning

#### Chebyshev Polynomial Physical Meaning

The Chebyshev coefficients reconstruct the meniscus temperature profile across the mold width. Each coefficient captures a distinct physical mode of the meniscus shape:

| Coefficient | Mathematical Mode | Physical Meaning | What High Variability Indicates |
| --- | --- | --- | --- |
| **X₁** | Linear (tilt) | **Meniscus asymmetry** — one side higher than the other | Unsteady single-roll flow; SEN jet oscillating left-right |
| **X₂** | Quadratic (curvature) | **Wave height / flatness** — how much the meniscus bulges or dips at the centre | Standing wave amplitude; higher X₂ σ = meniscus sloshing |
| **X₃** | Cubic (skewness) | **Profile skewness** — asymmetric wave shape | Non-symmetric flow pattern; one recirculation cell dominant |
| **X₄** | Quartic (bimodality) | **Edge effects / bimodal peaks** — meniscus peaks at both narrow faces | Double-roll instability; wave amplification at narrow faces |

The BFF (Broad Face Fixed) and BFL (Broad Face Loose) sensors measure opposite sides of the mold. When BFF and BFL profiles differ, this indicates **cross-mold flow asymmetry** — the liquid steel is not flowing symmetrically through the mold cavity.

#### Top-2 Features (Consistent Across Strands)

**1. `ptp_mm` — Mold Level Peak-to-Peak Range (SHAP: 0.165 S5, 0.168 S6)**

The dominant predictor. PtP captures the **extreme excursion range** within each 5-minute window — the difference between the highest and lowest mold level reading. While correlated with ML σ, PtP carries unique information about **rare severe spikes** vs. sustained oscillation. A high PtP with moderate σ indicates isolated but intense excursion events (e.g., a single nozzle clog/unclog cycle).

**2. `SEN_std` — SEN Depth Variability (SHAP: 0.073 S5, 0.085 S6)**

The most **actionable** driver. When the SEN immersion depth fluctuates within a 5-minute window, it destabilises the meniscus through three physical mechanisms:
* **Jet trajectory change:** SEN exit ports direct steel flow into the mold. Depth changes alter the jet angle, shifting the impingement point on the narrow face.
* **Momentum imbalance:** The upper recirculation loop (meniscus flow) and lower loop are coupled. SEN depth shifts redistribute momentum between them.
* **Transient standing waves:** Each depth change triggers a damped oscillation in the meniscus as the flow adjusts to the new configuration. If adjustments occur faster than the damping time (~5-10s), waves accumulate.

**Actionable:** SEN depth variability is a mechanical issue (nozzle holder stability, stopper rod control resolution, tundish steel level fluctuation). Tightening the control loop or improving mechanical rigidity would directly reduce ML σ.

**3. `ArFlow_avg` — Argon Flow Rate (SHAP: 0.012 S5, 0.013 S6)**

Argon gas is injected through the SEN to prevent nozzle clogging by alumina buildup. Higher argon flow rates create larger gas bubbles that:
* **Rise through the meniscus** — creating local surface disturbances
* **Modify the flow pattern** — gas-liquid drag alters the recirculation loop strength
* **Interact with the EMBR field** — electromagnetic forces act on the conductive steel but not on the gas, creating flow instabilities at the gas-steel interface

The relatively low SHAP value suggests argon is a **secondary modulator** rather than a primary driver.

#### Next Tier: Meniscus Shape Features

**4-7. Meniscus asymmetry σ, Chebyshev X₂ BFL σ, ML L-R asymmetry σ, Casting speed σ**

These features rank 4-7 on both strands with slight reordering, confirming that the same physical mechanisms drive instability regardless of strand:

* **Meniscus FF-LF asymmetry σ** → Unsteady cross-mold flow detected by comparing BFF and BFL FBG arrays. High asymmetry σ means the flow pattern is switching between symmetric (double-roll) and asymmetric (single-roll) modes.
* **Chebyshev X₂ BFL σ** → Standing wave oscillation on the loose face. The loose face is mechanically less constrained, so wave amplitudes may differ from the fixed face.
* **ML L-R asymmetry σ** → Cross-mold sloshing detected by the dual Vuhz sensors (530 mm apart on CC23). High L-R σ confirms standing wave activity at the specific spatial frequency captured by the sensor spacing.
* **Casting speed σ** → Even within "steady-state" windows, small Vc fluctuations (< 0.1 m/min) modulate the flow pattern. The SHAP value confirms this is a minor but real contributor.

#### Key Insight: Feature Hierarchy Matches Physical Expectations

The SHAP ranking — mechanical (SEN) > flow pattern (meniscus shape) > external (argon, EMBR) — is consistent with the physics of continuous casting. The FC Mold G5 electromagnetic field can compensate for **steady-state** flow patterns but cannot react fast enough to compensate for **transient** mechanical disturbances (SEN hunting). This is the fundamental finding of the investigation.

---
## 7. The Caster Naturally Operates in Distinct Stability Regimes Linked to Process Conditions

### 7.1 Methodology

To complement the supervised SHAP analysis, we apply **unsupervised clustering** to discover natural operating regimes without prior assumptions about what constitutes "stable" or "unstable" operation.

**Algorithm pipeline:**
1. **Feature standardisation** (StandardScaler on all numeric features)
2. **t-SNE dimensionality reduction** (2D embedding, perplexity=30, PCA-initialised)
3. **HDBSCAN density-based clustering** (min_cluster_size=15, min_samples=5)

**Why HDBSCAN over k-Means?**
* Does not require specifying the number of clusters a priori
* Handles non-convex cluster shapes (operating regimes are not spherical in feature space)
* Identifies noise points (sequences that don't belong to any clear regime)
* Hierarchical: reveals cluster substructure

### 7.2 Clustering Results and Aggregate Tables

The clustering identifies 13–15 distinct regimes per strand, with ~45–53% of sequences classified as noise (transitional conditions). The aggregate tables below (Fig 7.2) provide experts with a concise summary of the most problematic operating regimes for targeted investigation.

**Key finding:** The top 2 unstable clusters consistently feature wider molds (>2.0 m), higher SEN variability (σ > 1.5 mm), and Grade Family 5 (micro-alloyed). These clusters represent actionable targets for FC Mold G5 lookup table review.

### 7.3 Stable vs Unstable Operating Envelopes

**What differentiates stable from unstable regimes** is not just the wave amplitude (`profile_bff_ptp`) but **where the wave peaks** (`ptp_x_max`). Regimes with waves peaking near the narrow face (±0.7–1.0 normalised width) are consistently less stable than those with waves distributed more centrally. This confirms that the standing wave is amplified at the narrow face — the specific physical mechanism the FC Mold G5 is designed to counteract.

In [0]:
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# =====================================================================
# Cluster both strands (sklearn HDBSCAN + t-SNE instead of umap)
# FB7: Include EMBR features in cluster profiles
# =====================================================================
def cluster_strand(df_seq, strand_name):
    feat_cols = [c for c in df_seq.columns if c not in exclude
                 and c not in ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
                               'Quality casting', 'SEN_type', 'castMode',
                               'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]']
                 and df_seq[c].dtype in ['float64', 'int64']]
    X = df_seq[feat_cols].dropna()
    idx = X.index
    Xs = StandardScaler().fit_transform(X)
    
    emb = TSNE(n_components=2, perplexity=30, learning_rate='auto',
               init='pca', random_state=42, n_iter=1000).fit_transform(Xs)
    
    labels = HDBSCAN(min_cluster_size=15, min_samples=5,
                     cluster_selection_method='eom').fit_predict(Xs)
    df_out = df_seq.loc[idx].copy()
    df_out['tsne_1'] = emb[:, 0]
    df_out['tsne_2'] = emb[:, 1]
    df_out['cluster'] = labels
    nc = len(set(labels) - {-1})
    nn = (labels == -1).sum()
    print(f"  {strand_name}: {nc} clusters, {nn} noise ({100*nn/len(labels):.1f}%)")
    return df_out

print('Computing t-SNE + HDBSCAN...')
df_c5 = cluster_strand(df_seq_5, 'Strand 5')
df_c6 = cluster_strand(df_seq_6, 'Strand 6')

# =====================================================================
# Fig 7.1a: t-SNE colored by cluster
# =====================================================================
fig_cl = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 \u2014 HDBSCAN Clusters', 'Strand 6 \u2014 HDBSCAN Clusters'],
    horizontal_spacing=0.08)

for i, (df_c, strand) in enumerate([(df_c5, 'S5'), (df_c6, 'S6')]):
    df_c['cl_label'] = df_c['cluster'].apply(lambda x: 'Noise' if x == -1 else f'C{x}')
    for cl in sorted(df_c['cl_label'].unique()):
        sub = df_c[df_c['cl_label'] == cl]
        fig_cl.add_trace(go.Scatter(x=sub['tsne_1'], y=sub['tsne_2'], mode='markers',
            name=f'{strand}:{cl}', marker=dict(size=4, opacity=0.5), showlegend=False),
            row=1, col=i+1)

fig_cl.update_layout(title='Fig 7.1a \u2014 Operating Regime Discovery (HDBSCAN in t-SNE Space)',
                     height=500, width=1200, template='plotly_white')
fig_cl.update_xaxes(title_text='t-SNE-1')
fig_cl.update_yaxes(title_text='t-SNE-2')
fig_cl.show()

# =====================================================================
# Fig 7.1b: t-SNE colored by ML \u03c3
# =====================================================================
vmax = max(df_c5[target].quantile(0.95), df_c6[target].quantile(0.95))
fig_ml = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 \u2014 ML \u03c3 in t-SNE', 'Strand 6 \u2014 ML \u03c3 in t-SNE'],
    horizontal_spacing=0.08)

for i, (df_c, strand) in enumerate([(df_c5, 'S5'), (df_c6, 'S6')]):
    fig_ml.add_trace(go.Scatter(x=df_c['tsne_1'], y=df_c['tsne_2'], mode='markers',
        marker=dict(size=5, color=df_c[target], colorscale='Viridis',
                    cmin=0, cmax=vmax,
                    colorbar=dict(title='ML \u03c3 [mm]') if i == 1 else None,
                    showscale=(i == 1)),
        name=strand, showlegend=False), row=1, col=i+1)

fig_ml.update_layout(title='Fig 7.1b \u2014 Mold Level Instability Mapped onto t-SNE',
                     height=500, width=1200, template='plotly_white')
fig_ml.update_xaxes(title_text='t-SNE-1')
fig_ml.update_yaxes(title_text='t-SNE-2')
fig_ml.show()

# =====================================================================
# FB7: Cluster profile tables WITH EMBR features
# =====================================================================
def cluster_profile(df_c, strand):
    # Process features
    process_cols = [target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]',
                    'SEN_avg [mm]', 'SEN_std', 'ArFlow_SEN_avg',
                    'meniscus_FF_LF_asym_std', 'cheb_X2_bfl_std',
                    'profile_bff_ptp', 'vuhz_wave_height',
                    'ptp_x_max', 'ptp_x_min', 'ptp_span']
    # EMBR features (6 raw currents + derived LR stats)
    embr_profile_cols = [
        'AC_Left_Master_avg [A]', 'DC_Left_Master_avg [A]', 'DC_Left_Bottom_avg [A]',
        'AC_Right_Master_avg [A]', 'DC_Right_Master_avg [A]', 'DC_Right_Bottom_avg [A]',
        'EMBR_AC_Master_LR_mean', 'EMBR_DC_Master_LR_mean', 'EMBR_DC_Bottom_LR_mean'
    ]
    all_cols = process_cols + embr_profile_cols
    pc = [c for c in all_cols if c in df_c.columns]
    prof = df_c.groupby('cluster')[pc].median().round(3)
    prof.insert(0, 'n', df_c.groupby('cluster').size())
    
    # Add dominant grade per cluster
    if 'quality_family' in df_c.columns:
        mode_grade = df_c.groupby('cluster')['quality_family'].agg(
            lambda x: x.value_counts().index[0] if len(x) > 0 else 'N/A')
        prof['dominant_grade'] = mode_grade
    
    # Add instability rate per cluster
    prof['unstable_pct'] = (df_c.groupby('cluster')[target].apply(lambda x: (x > ML_THRESHOLD).mean()) * 100).round(1)
    prof['strand'] = strand
    return prof.sort_values(target)

print('\n\u2500\u2500 Cluster Profiles with EMBR Features (sorted by ML \u03c3) \u2500\u2500\u2500\u2500')
cluster_profiles = {}
for strand, df_c in [('Strand 5', df_c5), ('Strand 6', df_c6)]:
    print(f'\n{strand}:')
    prof = cluster_profile(df_c, strand)
    cluster_profiles[strand] = prof
    # Display process features and EMBR features separately for readability
    process_display = [c for c in prof.columns if 'Left' not in c and 'Right' not in c 
                       and 'EMBR' not in c and c != 'strand']
    embr_display = ['n', target] + [c for c in prof.columns if 'Left' in c or 'Right' in c or 'EMBR' in c]
    print('  Process features:')
    display(prof[process_display])
    embr_available = [c for c in embr_display if c in prof.columns]
    if len(embr_available) > 2:
        print(f'  EMBR features ({len(embr_available)-2} currents):')
        display(prof[embr_available])

print('\n\u2500\u2500 Key Cluster Insights \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
for strand in ['Strand 5', 'Strand 6']:
    prof = cluster_profiles[strand]
    # High-instability clusters
    high_inst = prof[prof['unstable_pct'] > 30]
    if len(high_inst) > 0:
        print(f'\n{strand} \u2014 High-instability clusters (>30% unstable):')
        for cl, r in high_inst.iterrows():
            sen_col = 'SEN_std' if 'SEN_std' in r.index else 'SEN_std [mm]'
            vc_col = 'CASTING_SPEED_avg [m/min]'
            print(f'  Cluster {cl}: n={int(r["n"])}, ML\u03c3={r[target]:.2f}mm, '
                  f'Vc={r[vc_col]:.2f}, unstable={r["unstable_pct"]:.0f}%')

In [0]:
# =====================================================================
# Fig 7.2: Aggregate cluster table for top unstable regimes
# V2.3: Compact table format for A4, CSV export for full data
# =====================================================================
import shutil, os

ar_col = 'ArFlow_avg [NL/min]'

def cluster_summary(df_c, strand_name, n_top=2):
    real_clusters = df_c[df_c['cluster'] != -1].copy()
    cl_stats = real_clusters.groupby('cluster')[target].agg(['median', 'count']).reset_index()
    cl_stats.columns = ['cluster', 'median_ML\u03c3', 'n']
    cl_stats = cl_stats[cl_stats['n'] >= 5]
    top = cl_stats.nlargest(n_top, 'median_ML\u03c3')
    rows = []
    for _, row in top.iterrows():
        cl = int(row['cluster'])
        sub = real_clusters[real_clusters['cluster'] == cl]
        r = {
            'Cl': cl, 'n': int(row['n']),
            'ML\u03c3': f"{row['median_ML\u03c3']:.3f}",
            'Vc': f"{sub['CASTING_SPEED_avg [m/min]'].mean():.2f}" if 'CASTING_SPEED_avg [m/min]' in sub.columns else '-',
            'Width': f"{sub['MOLD_WIDTH_avg [m]'].mean():.3f}" if 'MOLD_WIDTH_avg [m]' in sub.columns else '-',
            'SEN': f"{sub['SEN_avg [mm]'].mean():.1f}" if 'SEN_avg [mm]' in sub.columns else '-',
            'SEN\u03c3': f"{sub['SEN_std [mm]'].mean():.2f}" if 'SEN_std [mm]' in sub.columns else '-',
        }
        if ar_col in sub.columns: r['ArFlow'] = f"{sub[ar_col].mean():.1f}"
        if 'quality_family' in sub.columns:
            mode = sub['quality_family'].mode()
            r['Grade'] = mode.iloc[0] if len(mode) > 0 else '?'
        rows.append(r)
    return pd.DataFrame(rows)

def full_cluster_export(df_c, strand_name):
    real = df_c[df_c['cluster'] != -1].copy()
    agg_cols = {target: ['median', 'mean', 'std', 'count'],
                'CASTING_SPEED_avg [m/min]': ['mean', 'std'],
                'MOLD_WIDTH_avg [m]': ['mean', 'std'],
                'SEN_avg [mm]': ['mean', 'std']}
    if 'SEN_std [mm]' in real.columns: agg_cols['SEN_std [mm]'] = ['mean', 'std']
    if ar_col in real.columns: agg_cols[ar_col] = ['mean', 'std']
    valid_cols = {k: v for k, v in agg_cols.items() if k in real.columns}
    agg = real.groupby('cluster').agg(valid_cols)
    agg.columns = ['_'.join(c) for c in agg.columns]
    return agg.round(3)

print('\u2550' * 80)
print('SECTION 7.2: COMPACT CLUSTER TABLES (V2.3)')
print('\u2550' * 80)

excel_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5'
os.makedirs(excel_dir, exist_ok=True)

for df_c, sname in [(df_c5, 'Strand 5'), (df_c6, 'Strand 6')]:
    df_summary = cluster_summary(df_c, sname)
    print(f'\n{sname} \u2014 Top 2 Unstable Clusters (compact):')
    display(df_summary)
    
    real = df_c[df_c['cluster'] != -1]
    cl_stats = real.groupby('cluster')[target].agg(['median', 'count']).reset_index()
    cl_stats.columns = ['cluster', 'median_ML\u03c3', 'n']
    cl_stats = cl_stats[cl_stats['n'] >= 5]
    top_u = cl_stats.nlargest(2, 'median_ML\u03c3')
    top_s = cl_stats.nsmallest(2, 'median_ML\u03c3')
    
    print(f'\n  Unstable vs Stable comparison:')
    for label, sel in [('UNSTABLE', top_u), ('STABLE', top_s)]:
        for _, row in sel.iterrows():
            cl = int(row['cluster'])
            sub = real[real['cluster'] == cl]
            cols_to_show = ['CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]', 'SEN_avg [mm]', 'SEN_std [mm]']
            vals = ' | '.join([f'{sub[c].mean():.2f}' for c in cols_to_show if c in sub.columns])
            print(f'    {label} Cl{cl} (n={int(row["n"])}, ML\u03c3={row["median_ML\u03c3"]:.3f}): Vc|Width|SEN|SEN\u03c3 = {vals}')
    
    # Export full cluster data to CSV (no openpyxl needed)
    full_agg = full_cluster_export(df_c, sname)
    csv_name = f'cluster_summary_{sname.replace(" ", "_")}.csv'
    tmp_path = f'/tmp/{csv_name}'
    full_agg.to_csv(tmp_path)
    shutil.copy2(tmp_path, os.path.join(excel_dir, csv_name))
    print(f'\n  \u2714 Full cluster data saved: /files/TATAIjmulden_FCMoldG5/{csv_name}')

print('\n\u2714 V2.3: Compact tables for A4 report. Full cluster data available as CSV downloads.')

---
## 8. Under Real Disturbances, Electromagnetic Compensation Reaches Its Limits

### Motivation: From Ground Truth to Real-Life Stress Test

Sections 3–7 established the **ground truth** — the baseline performance of the FC Mold G5 system under ideal steady-state conditions. That analysis used only the 80% of sequences with no detected disturbances, revealing an overwhelmingly stable system where about 64% (S5) and 58% (S6) of sequences fall below the ±1 mm ML σ threshold.

But the caster does not run in a laboratory. **This section subjects the FC Mold G5 to a stress test** by including the remaining ~20% of sequences that contain real operational disturbances (excursions, high variability, transient bumps). By repeating the same analyses and comparing against the baseline, we identify:

1. **Where the FC Mold G5 struggles most** — which EMBR operating regimes have the highest failure rates?
2. **What changes in the driver hierarchy** — do the same SHAP features matter, or do new ones emerge?
3. **How the operating landscape shifts** — do disturbed sequences cluster separately or mix with stable regimes?

This comparison provides **actionable intelligence for the FC Mold G5 vendor** (ABB) to investigate specific EMBR current combinations, SEN variability thresholds, and operating regimes where the electromagnetic control did not fully compensate.

### Analytical Approach

| Analysis | Baseline (Sec 3–7) | Real-Life (Sec 8) | Vendor Question |
| --- | --- | --- | --- |
| **SHAP importance** | XGBoost on clean (R²=0.946) | XGBoost on all | Which features gain importance under stress? |
| **EMBR failure mapping** | 86th percentile threshold | Fixed ±1 mm + AC/DC heatmaps | Which EMBR regimes have highest failure rates? |
| **SEN×EMBR interaction** | Not analysed | SENσ vs DC current scatter | At what SEN variability does DC braking fail? |
| **Regime discovery** | HDBSCAN on clean (15/13 clusters) | HDBSCAN on all | Where do disturbances live in the operating space? |

### Sequence Composition

| | Strand 5 | Strand 6 |
| --- | --- | --- |
| Clean (disturbance-free) | 934 (79.6%) | 948 (78.4%) |
| High variability only | 112 (9.5%) | 166 (13.7%) |
| Excursion + High variability | 107 (9.1%) | 88 (7.3%) |
| Excursion + HV + Transient bump | 21 (1.8%) | 7 (0.6%) |
| **Total** | **1,174** | **1,209** |

In [0]:
# =====================================================================
# Fig 8.1: Distribution Comparison — Clean vs Disturbed Sequences
# Side-by-side histograms for key process parameters
# =====================================================================
import numpy as np
import pandas as pd

target = 'MOLD_LEVEL_std [mm]'
sen_std_col = 'SEN_std [mm]'

# Prepare labels
for df_a in [df_seq_5_all, df_seq_6_all]:
    df_a['data_type'] = np.where(df_a['has_disturbance'], 'Disturbed', 'Clean')
    if 'Quality casting' in df_a.columns and 'quality_family' not in df_a.columns:
        df_a['quality_family'] = df_a['Quality casting'].astype(str).str[0]

# Define comparison features
comp_features = [
    (target, 'ML \u03c3 [mm]'),
    (sen_std_col, 'SEN \u03c3 [mm]'),
    ('CASTING_SPEED_avg [m/min]', 'Casting Speed [m/min]'),
    ('MOLD_WIDTH_avg [m]', 'Mold Width [m]'),
    ('ArFlow_avg [NL/min]', 'Argon Flow [NL/min]'),
    ('ptp_mm', 'ML Peak-to-Peak [mm]'),
    ('EMBR_AC_Master_LR_mean', 'EMBR AC Master [A]'),
    ('EMBR_DC_Master_LR_mean', 'EMBR DC Master [A]'),
]

PAL_dt = {'Clean': '#2166AC', 'Disturbed': '#D6604D'}

fig_81 = make_subplots(rows=4, cols=2,
    subplot_titles=[label for _, label in comp_features],
    vertical_spacing=0.08, horizontal_spacing=0.08)

for idx, (col, label) in enumerate(comp_features):
    row = idx // 2 + 1; col_pos = idx % 2 + 1
    # Combine both strands
    df_comb = pd.concat([df_seq_5_all, df_seq_6_all])
    for dt, clr in PAL_dt.items():
        vals = df_comb[df_comb['data_type'] == dt][col].dropna()
        if len(vals) > 0:
            fig_81.add_trace(go.Histogram(
                x=vals, name=dt, marker_color=clr, opacity=0.6,
                nbinsx=40, showlegend=(idx == 0)), row=row, col=col_pos)

fig_81.update_layout(
    title='Fig 8.1 \u2014 Distribution Shift: Clean vs Disturbed Sequences (Both Strands)',
    height=900, width=1200, template='plotly_white', barmode='overlay',
    legend=dict(x=0.85, y=1.0))
fig_81.show()

# Quantitative shift summary
print('\n' + '=' * 80)
print('DISTRIBUTION SHIFT SUMMARY (median values, combined strands)')
print('=' * 80)
df_comb = pd.concat([df_seq_5_all, df_seq_6_all])
for col, label in comp_features:
    if col in df_comb.columns:
        m_clean = df_comb[df_comb['data_type'] == 'Clean'][col].median()
        m_dist = df_comb[df_comb['data_type'] == 'Disturbed'][col].median()
        ratio = m_dist / m_clean if m_clean != 0 else float('inf')
        print(f'  {label:30s}  Clean: {m_clean:8.3f}  Disturbed: {m_dist:8.3f}  Ratio: {ratio:.2f}x')

In [0]:
# =====================================================================
# Fig 8.2: XGBoost + SHAP on ALL sequences — comparison to baseline
# Train on full dataset (clean + disturbed), compare feature importance
# =====================================================================
import xgboost as xgb
import shap
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import re
import warnings
warnings.filterwarnings('ignore')

target = 'MOLD_LEVEL_std [mm]'

exclude_cols = ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand', 'data_type',
                'Quality casting', 'SEN_type', 'castMode', 'quality_family',
                'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]', 'disturbance_type',
                'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
                'has_high_variability', 'has_disturbance',
                'ArFlow_SEN_avg', 'ArFlow_Stopper_avg',
                'cluster', 'cl_label', 'tsne_1', 'tsne_2', target]

def sanitize_names(cols):
    """XGBoost forbids [, ], < in feature names."""
    return [re.sub(r'[\[\]<]', '_', c) for c in cols]

def train_xgb_all(df_all, strand_name):
    feat_cols = [c for c in df_all.columns if c not in exclude_cols
                 and df_all[c].dtype in ['float64', 'int64']
                 and df_all[c].notna().sum() > len(df_all) * 0.5]
    
    X = df_all[feat_cols].copy()
    y = df_all[target].copy()
    valid = X.notna().all(axis=1) & y.notna()
    X = X[valid]; y = y[valid]
    
    # Sanitize column names for XGBoost
    orig_names = X.columns.tolist()
    X.columns = sanitize_names(X.columns)
    print(f'  {strand_name}: {len(X)} sequences, {len(feat_cols)} features')
    
    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        tree_method='hist', n_jobs=-1)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = cross_val_predict(model, X, y, cv=kf)
    
    r2 = r2_score(y, oof)
    rmse = mean_squared_error(y, oof, squared=False)
    mae = mean_absolute_error(y, oof)
    print(f'  R\u00b2={r2:.3f}, RMSE={rmse:.3f}, MAE={mae:.3f}')
    
    model.fit(X, y)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    # Map back to original feature names for display
    shap_imp = pd.DataFrame({
        'feature': orig_names,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False)
    
    return {'r2': r2, 'rmse': rmse, 'mae': mae, 'y': y.values, 'oof': oof,
            'shap_imp': shap_imp, 'shap_values': shap_values, 'X': X,
            'model': model, 'feat_cols': feat_cols, 'orig_names': orig_names}

print('\n' + '=' * 80)
print('XGBOOST ON ALL SEQUENCES (clean + disturbed)')
print('=' * 80)
res_all_5 = train_xgb_all(df_seq_5_all, 'Strand 5')
res_all_6 = train_xgb_all(df_seq_6_all, 'Strand 6')

# =====================================================================
# Fig 8.2a: SHAP comparison — baseline vs all-data
# =====================================================================
top_n = 12

fig_shap_cmp = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5: SHAP Baseline vs All', 'Strand 6: SHAP Baseline vs All'],
    horizontal_spacing=0.15)

for i, (res_base, res_all, sname) in enumerate([
    (res_5, res_all_5, 'S5'), (res_6, res_all_6, 'S6')]):
    base_top = res_base['shap_imp'].head(top_n)
    all_top = res_all['shap_imp'].head(top_n)
    all_feats = list(dict.fromkeys(
        base_top['feature'].tolist() + all_top['feature'].tolist()))[:top_n]
    
    df_cmp = pd.DataFrame({'feature': all_feats})
    df_cmp = df_cmp.merge(
        base_top.rename(columns={'mean_abs_shap': 'Baseline'}), on='feature', how='left')
    df_cmp = df_cmp.merge(
        all_top.rename(columns={'mean_abs_shap': 'All Data'}), on='feature', how='left')
    df_cmp = df_cmp.fillna(0).sort_values('Baseline')
    
    fig_shap_cmp.add_trace(go.Bar(
        y=df_cmp['feature'], x=df_cmp['Baseline'], name='Baseline (clean)' if i == 0 else None,
        orientation='h', marker_color='#2166AC', opacity=0.7,
        showlegend=(i == 0)), row=1, col=i + 1)
    fig_shap_cmp.add_trace(go.Bar(
        y=df_cmp['feature'], x=df_cmp['All Data'], name='All Data (clean+disturbed)' if i == 0 else None,
        orientation='h', marker_color='#D6604D', opacity=0.7,
        showlegend=(i == 0)), row=1, col=i + 1)

fig_shap_cmp.update_layout(
    title=f'Fig 8.2a \u2014 SHAP Importance: Baseline (R\u00b2={res_5["r2"]:.3f}/{res_6["r2"]:.3f}) '
          f'vs All Data (R\u00b2={res_all_5["r2"]:.3f}/{res_all_6["r2"]:.3f})',
    height=550, width=1200, template='plotly_white', barmode='group',
    legend=dict(x=0.6, y=0.05))
fig_shap_cmp.show()

# =====================================================================
# Fig 8.2b: Actual vs Predicted — clean vs disturbed highlighted
# =====================================================================
fig_pred = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5: All-Data Model', 'Strand 6: All-Data Model'],
    horizontal_spacing=0.08)

for i, (res_all, df_a, sname) in enumerate([
    (res_all_5, df_seq_5_all, 'S5'), (res_all_6, df_seq_6_all, 'S6')]):
    feat_cols = res_all['feat_cols']
    X_check = df_a[feat_cols].copy()
    valid = X_check.notna().all(axis=1) & df_a[target].notna()
    labels = df_a.loc[valid, 'data_type'].values
    
    for dt, clr, sz in [('Clean', '#2166AC', 4), ('Disturbed', '#D6604D', 6)]:
        mask = labels == dt
        fig_pred.add_trace(go.Scatter(
            x=res_all['y'][mask], y=res_all['oof'][mask], mode='markers',
            name=f'{dt}' if i == 0 else None,
            marker=dict(size=sz, color=clr, opacity=0.4),
            showlegend=(i == 0)), row=1, col=i + 1)
    
    lim = max(res_all['y'].max(), res_all['oof'].max()) * 1.05
    fig_pred.add_trace(go.Scatter(
        x=[0, lim], y=[0, lim], mode='lines',
        line=dict(color='black', dash='dash', width=1),
        showlegend=False), row=1, col=i + 1)
    fig_pred.update_xaxes(title_text='Actual ML\u03c3 [mm]', row=1, col=i + 1)
    fig_pred.update_yaxes(title_text='Predicted ML\u03c3 [mm]', row=1, col=i + 1)

fig_pred.update_layout(
    title='Fig 8.2b \u2014 Actual vs Predicted: Disturbed Sequences Highlighted',
    height=500, width=1100, template='plotly_white',
    legend=dict(x=0.75, y=0.15))
fig_pred.show()

print('\nModel comparison:')
print(f'  Baseline  \u2014 S5 R\u00b2={res_5["r2"]:.3f}, S6 R\u00b2={res_6["r2"]:.3f}')
print(f'  All Data  \u2014 S5 R\u00b2={res_all_5["r2"]:.3f}, S6 R\u00b2={res_all_6["r2"]:.3f}')

In [0]:
# =====================================================================
# Fig 8.3: SHAP Side-by-Side Comparison (Baseline vs All Data)
# Fig 8.4: FC Mold G5 Deep Dive — EMBR Failure Rate Heatmaps
# Fig 8.5: SEN×EMBR Interaction — Where FC Mold Struggles
# Fig 8.6: Regime Discovery (All Sequences) + KPI Table
# =====================================================================
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

target = 'MOLD_LEVEL_std [mm]'
sen_std_col = 'SEN_std [mm]'
PAL_dt = {'Clean': '#2166AC', 'Disturbed': '#D6604D'}

# =====================================================================
# Fig 8.3: SHAP Side-by-Side — matplotlib for clean export
# =====================================================================
top_n = 12

fig_shap, axes_shap = plt.subplots(1, 2, figsize=(16, 7))
for ax, res_base, res_all, sname in [
    (axes_shap[0], res_5, res_all_5, 'Strand 5'),
    (axes_shap[1], res_6, res_all_6, 'Strand 6')]:
    
    base_top = res_base['shap_imp'].head(top_n)
    all_top = res_all['shap_imp'].head(top_n)
    all_feats = list(dict.fromkeys(
        all_top['feature'].tolist() + base_top['feature'].tolist()))[:top_n]
    
    df_cmp = pd.DataFrame({'feature': all_feats})
    df_cmp = df_cmp.merge(base_top.rename(columns={'mean_abs_shap': 'Baseline'}), on='feature', how='left')
    df_cmp = df_cmp.merge(all_top.rename(columns={'mean_abs_shap': 'All Data'}), on='feature', how='left')
    df_cmp = df_cmp.fillna(0).sort_values('All Data')
    
    y_pos = np.arange(len(df_cmp))
    bar_h = 0.35
    ax.barh(y_pos - bar_h/2, df_cmp['Baseline'], bar_h, color='#2166AC', alpha=0.7,
            label=f'Baseline (R\u00b2={res_base["r2"]:.3f})')
    ax.barh(y_pos + bar_h/2, df_cmp['All Data'], bar_h, color='#D6604D', alpha=0.7,
            label=f'All Data (R\u00b2={res_all["r2"]:.3f})')
    ax.set_yticks(y_pos)
    # Shorten feature names for readability
    short_feats = [f.replace('_avg', '').replace('_std', '\u03c3').replace('_mean', '')
                   .replace(' [mm]', '').replace(' [m/min]', '').replace(' [m]', '')
                   .replace(' [A]', '').replace(' [NL/min]', '')
                   for f in df_cmp['feature']]
    ax.set_yticklabels(short_feats, fontsize=8)
    ax.set_xlabel('Mean |SHAP|', fontsize=10)
    ax.set_title(sname, fontsize=12)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    
    # Highlight rank changes with arrows
    for i, (_, row) in enumerate(df_cmp.iterrows()):
        if row['All Data'] > row['Baseline'] * 1.5 and row['All Data'] > 0.01:
            ax.annotate('\u2191', xy=(row['All Data'] + 0.005, y_pos[i] + bar_h/2),
                       fontsize=10, color='#D6604D', fontweight='bold', va='center')

fig_shap.suptitle('Fig 8.3 \u2014 SHAP Feature Importance: Baseline (Clean) vs All Data (Clean + Disturbed)',
                  fontsize=13, y=1.02)
fig_shap.tight_layout()
display(fig_shap)
print('\u2714 Fig 8.3: SHAP side-by-side comparison')

# =====================================================================
# Fig 8.4: FC Mold G5 EMBR Failure Rate Heatmaps
# Bin AC and DC currents, compute failure rate (% > 1mm) per bin
# =====================================================================
fig_embr_heat, axes_eh = plt.subplots(2, 2, figsize=(16, 12))

for col_i, (df_a, sname) in enumerate([(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]):
    ac_col = 'EMBR_AC_Master_LR_mean'
    dc_col = 'EMBR_DC_Master_LR_mean'
    
    if ac_col not in df_a.columns or dc_col not in df_a.columns:
        continue
    
    # --- Row 1: Failure rate heatmap (AC x DC bins) ---
    ax = axes_eh[0, col_i]
    df_plot = df_a[[ac_col, dc_col, target, 'data_type']].dropna().copy()
    df_plot['ac_bin'] = pd.cut(df_plot[ac_col], bins=6)
    df_plot['dc_bin'] = pd.cut(df_plot[dc_col], bins=6)
    df_plot['fail'] = (df_plot[target] > ML_THRESHOLD).astype(int)
    
    pivot_rate = df_plot.groupby(['dc_bin', 'ac_bin'])['fail'].mean().unstack() * 100
    pivot_n = df_plot.groupby(['dc_bin', 'ac_bin'])['fail'].count().unstack()
    
    # Plot heatmap
    im = ax.imshow(pivot_rate.values, cmap='RdYlGn_r', aspect='auto',
                   vmin=0, vmax=min(60, pivot_rate.max().max() * 1.1),
                   origin='lower')
    # Annotate
    for yi in range(pivot_rate.shape[0]):
        for xi in range(pivot_rate.shape[1]):
            rate = pivot_rate.values[yi, xi]
            n = pivot_n.values[yi, xi] if not np.isnan(pivot_n.values[yi, xi]) else 0
            if not np.isnan(rate) and n >= 3:
                ax.text(xi, yi, f'{rate:.0f}%\n(n={int(n)})', ha='center', va='center',
                       fontsize=7, color='white' if rate > 30 else 'black')
    
    ac_labels = [str(x)[:6] for x in pivot_rate.columns]
    dc_labels = [str(x)[:6] for x in pivot_rate.index]
    ax.set_xticks(range(len(ac_labels))); ax.set_xticklabels(ac_labels, fontsize=7, rotation=45)
    ax.set_yticks(range(len(dc_labels))); ax.set_yticklabels(dc_labels, fontsize=7)
    ax.set_xlabel('EMBR AC Master [A]', fontsize=9)
    ax.set_ylabel('EMBR DC Master [A]', fontsize=9)
    ax.set_title(f'{sname}: Failure Rate (% > 1mm) by EMBR Currents', fontsize=10)
    plt.colorbar(im, ax=ax, label='Failure %', shrink=0.8)
    
    # --- Row 2: SEN sigma vs EMBR DC interaction ---
    ax2 = axes_eh[1, col_i]
    for dt, clr, sz, mk in [('Clean', '#2166AC', 8, 'o'), ('Disturbed', '#D6604D', 15, 's')]:
        sub = df_a[df_a['data_type'] == dt]
        if sen_std_col in sub.columns:
            sc = ax2.scatter(sub[sen_std_col], sub[dc_col], s=sz, alpha=0.4, c=clr,
                           marker=mk, label=dt, edgecolors='none')
    # Overlay ML sigma as contour-like sizing for failed sequences
    failed = df_a[df_a[target] > ML_THRESHOLD]
    if len(failed) > 0 and sen_std_col in failed.columns:
        ax2.scatter(failed[sen_std_col], failed[dc_col], s=failed[target] * 20,
                   facecolors='none', edgecolors='red', linewidth=1, alpha=0.6,
                   label=f'Failed (>1mm, n={len(failed)})')
    ax2.set_xlabel('SEN \u03c3 [mm]', fontsize=10)
    ax2.set_ylabel('EMBR DC Master [A]', fontsize=10)
    ax2.set_title(f'{sname}: SEN Variability vs EMBR DC (\u2b24=failed)', fontsize=10)
    ax2.legend(fontsize=8, loc='upper right')
    ax2.grid(alpha=0.3)

fig_embr_heat.suptitle('Fig 8.4 \u2014 FC Mold G5 Deep Dive: Where Does the System Fail?',
                       fontsize=13, y=1.01)
fig_embr_heat.tight_layout()
display(fig_embr_heat)
print('\u2714 Fig 8.4: EMBR failure heatmaps + SEN\u00d7EMBR interaction')

# =====================================================================
# Fig 8.5: Disturbance Rate by EMBR Operating Regime
# =====================================================================
fig_regime, axes_rg = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_a, sname in [(axes_rg[0], df_seq_5_all, 'Strand 5'),
                         (axes_rg[1], df_seq_6_all, 'Strand 6')]:
    ac_col = 'EMBR_AC_Master_LR_mean'
    dc_col = 'EMBR_DC_Master_LR_mean'
    if ac_col not in df_a.columns:
        continue
    # Create EMBR regime bins
    df_tmp = df_a[[ac_col, dc_col, target, 'data_type']].dropna().copy()
    df_tmp['ac_q'] = pd.qcut(df_tmp[ac_col], q=4, labels=['AC Q1', 'AC Q2', 'AC Q3', 'AC Q4'])
    df_tmp['dc_q'] = pd.qcut(df_tmp[dc_col], q=3, labels=['DC Low', 'DC Mid', 'DC High'], duplicates='drop')
    df_tmp['regime'] = df_tmp['ac_q'].astype(str) + ' / ' + df_tmp['dc_q'].astype(str)
    
    regime_stats = df_tmp.groupby('regime').agg(
        n=(target, 'count'),
        fail_rate=(target, lambda x: 100 * (x > ML_THRESHOLD).mean()),
        dist_pct=('data_type', lambda x: 100 * (x == 'Disturbed').mean()),
        ml_median=(target, 'median')
    ).reset_index()
    regime_stats = regime_stats[regime_stats['n'] >= 10].sort_values('fail_rate', ascending=True)
    
    y = np.arange(len(regime_stats))
    bars = ax.barh(y, regime_stats['fail_rate'], color='#D6604D', alpha=0.7)
    ax.barh(y, regime_stats['dist_pct'], color='#8e44ad', alpha=0.4, label='% Disturbed')
    for i, (_, r) in enumerate(regime_stats.iterrows()):
        ax.text(max(r['fail_rate'], r['dist_pct']) + 1, i,
               f'n={int(r["n"])}, ML\u03c3={r["ml_median"]:.2f}', fontsize=7, va='center')
    ax.set_yticks(y); ax.set_yticklabels(regime_stats['regime'], fontsize=8)
    ax.set_xlabel('Rate [%]'); ax.set_title(sname, fontsize=11)
    ax.legend(['% Failed (>1mm)', '% Disturbed'], fontsize=8)
    ax.grid(axis='x', alpha=0.3)

fig_regime.suptitle('Fig 8.5 \u2014 FC Mold G5 Performance by EMBR Operating Regime', fontsize=13)
fig_regime.tight_layout()
display(fig_regime)
print('\u2714 Fig 8.5: Failure rate by EMBR regime')

# =====================================================================
# Fig 8.6: t-SNE + HDBSCAN on ALL sequences
# =====================================================================
print('\nComputing t-SNE + HDBSCAN on ALL sequences...')

def cluster_all(df_all, strand_name):
    exclude_cl = ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand', 'data_type',
                  'Quality casting', 'SEN_type', 'castMode', 'quality_family',
                  'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]', 'disturbance_type',
                  'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
                  'has_high_variability', 'has_disturbance',
                  'ArFlow_SEN_avg', 'ArFlow_Stopper_avg',
                  'cluster', 'cl_label', 'tsne_1', 'tsne_2']
    feat_cols = [c for c in df_all.columns if c not in exclude_cl
                 and df_all[c].dtype in ['float64', 'int64']]
    X = df_all[feat_cols].dropna()
    idx = X.index
    Xs = StandardScaler().fit_transform(X)
    emb = TSNE(n_components=2, perplexity=30, learning_rate='auto',
               init='pca', random_state=42, n_iter=1000).fit_transform(Xs)
    labels = HDBSCAN(min_cluster_size=15, min_samples=5,
                     cluster_selection_method='eom').fit_predict(Xs)
    df_out = df_all.loc[idx].copy()
    df_out['tsne_all_1'] = emb[:, 0]
    df_out['tsne_all_2'] = emb[:, 1]
    df_out['cluster_all'] = labels
    nc = len(set(labels) - {-1})
    nn = (labels == -1).sum()
    print(f'  {strand_name}: {nc} clusters, {nn} noise ({100*nn/len(labels):.1f}%)')
    return df_out

df_ca5 = cluster_all(df_seq_5_all, 'Strand 5')
df_ca6 = cluster_all(df_seq_6_all, 'Strand 6')

# t-SNE overlay: clean vs disturbed
fig_tsne = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 \u2014 t-SNE (clean vs disturbed)',
                    'Strand 6 \u2014 t-SNE (clean vs disturbed)'],
    horizontal_spacing=0.08)
for i, df_ca in enumerate([df_ca5, df_ca6]):
    for dt, clr, sz in [('Clean', '#2166AC', 4), ('Disturbed', '#D6604D', 6)]:
        sub = df_ca[df_ca['data_type'] == dt]
        fig_tsne.add_trace(go.Scatter(
            x=sub['tsne_all_1'], y=sub['tsne_all_2'], mode='markers',
            name=f'{dt}' if i == 0 else None,
            marker=dict(size=sz, color=clr, opacity=0.4),
            showlegend=(i == 0)), row=1, col=i + 1)
fig_tsne.update_layout(title='Fig 8.6 \u2014 Regime Map: Where Do Disturbances Live in Operating Space?',
                       height=500, width=1100, template='plotly_white')
fig_tsne.show()

# =====================================================================
# KPI COMPARISON TABLE
# =====================================================================
print('\n' + '=' * 80)
print('COMPREHENSIVE KPI COMPARISON: BASELINE vs ALL DATA')
print('=' * 80)

kpi_rows = []
for sname, df_clean, df_all_s, res_base, res_all_r, df_c_base, df_ca in [
    ('S5', df_seq_5, df_seq_5_all, res_5, res_all_5, df_c5, df_ca5),
    ('S6', df_seq_6, df_seq_6_all, res_6, res_all_6, df_c6, df_ca6)]:
    for metric, bl, ad, delta in [
        ('n sequences', str(len(df_clean)), str(len(df_all_s)),
         f'+{len(df_all_s)-len(df_clean)} ({100*(len(df_all_s)-len(df_clean))/len(df_clean):.0f}%)'),
        ('ML \u03c3 median [mm]', f'{df_clean[target].median():.3f}', f'{df_all_s[target].median():.3f}',
         f'+{df_all_s[target].median()-df_clean[target].median():.3f}'),
        ('% > 1mm', f'{100*(df_clean[target]>ML_THRESHOLD).mean():.1f}%', f'{100*(df_all_s[target]>ML_THRESHOLD).mean():.1f}%',
         f'+{100*((df_all_s[target]>ML_THRESHOLD).mean()-(df_clean[target]>ML_THRESHOLD).mean()):.1f}pp'),
        ('XGBoost R\u00b2', f'{res_base["r2"]:.3f}', f'{res_all_r["r2"]:.3f}',
         f'{res_all_r["r2"]-res_base["r2"]:+.3f}'),
        ('SHAP #1', res_base['shap_imp'].iloc[0]['feature'][:20],
         res_all_r['shap_imp'].iloc[0]['feature'][:20],
         '\u2714 Same' if res_base['shap_imp'].iloc[0]['feature'] == res_all_r['shap_imp'].iloc[0]['feature'] else '\u26a0 Changed'),
        ('HDBSCAN clusters', str(df_c_base['cluster'].nunique()-1),
         str(df_ca['cluster_all'].nunique()-1),
         f'{df_ca["cluster_all"].nunique()-1-df_c_base["cluster"].nunique()+1:+d}')]:
        kpi_rows.append({'Strand': sname, 'Metric': metric,
                        'Baseline (clean)': bl, 'All Data': ad, 'Delta': delta})

df_kpi = pd.DataFrame(kpi_rows)
display(df_kpi)
print('\n\u2714 Section 8 analysis complete.')
plt.close('all')

---
## 9. Steady-State Stability Is Confirmed — Remaining Instability Traces to SEN Variability and Operational Disturbances

This section synthesises the evidence from trend analysis (Sec 3–4), EMBR failure profiling (Sec 5), predictive modelling (Sec 6), regime discovery (Sec 7), **and the real-life disturbance comparison (Sec 8)** into a coherent picture of what drives mold level variability on CC23.

### 9.1 The Overall Picture: CC23 is Overwhelmingly Stable in Steady State

**The data shows:** After removing all sequences with detected disturbances, against the guaranteed ±1 mm threshold, **63.7% (S5) and 58.3% (S6) of clean sequences meet the target**. Median ML σ is 0.87 mm (S5) and 0.92 mm (S6) — below the threshold — but the right tail pushes a significant fraction above it.

**This means:** The FC Mold G5 system is working effectively during true steady-state casting. The instability episodes reported by TATA Steel are predominantly associated with detectable disturbance events rather than with the FC Mold G5’s inability to control steady-state flow.

**Real-life context (Section 8):** When disturbances are included, ML σ jumps 2.5× (0.87→2.35 mm on S5) and the failure rate rises from 0% to 67%. This confirms that the ~20% of disturbed sequences represent the primary quality risk.

### 9.2 Finding 1: SEN Depth Variability is the Dominant Controllable Driver

**The data shows:** SEN depth standard deviation ranks as the #2 SHAP feature on the clean baseline. **When disturbances are included (Section 8), SENσ jumps to #1** (SHAP 0.57 vs 0.26 for PtP on S5), confirming that SEN variability is the primary mechanism through which disturbances manifest as mold level instability.

**This means:** SEN variability doubles from 1.2 mm (clean) to 2.3 mm (disturbed), and this increase is the single strongest predictor of the corresponding MLσ increase.

### 9.3 Finding 2: Meniscus Shape Predicts Instability 4 Seconds Ahead (Strand 6)

**The data shows:** On Strand 6, Chebyshev X₂ Granger-causes ML deviation with F=210 at lag=4s.

**Implications:** Feedforward control using FBG shape signals could anticipate instability before it appears in the Vuhz mold level.

### 9.4 Finding 3: Steel Grade Effects Persist After Controlling for Casting Conditions

**The data shows:** Kruskal-Wallis test on MLσ residuals yields H=58–83, p<10⁻¹². Grade Family 5 (HSLA/micro-alloyed) shows the highest MLσ on both strands (S5: 1.24 mm, S6: 1.28 mm). The remaining grade ranking is strand-dependent: S5 ranks Family 3 (1.22 mm) 2nd, while S6 ranks Family N (1.09 mm) 2nd and Family 3 only 4th (0.99 mm). Family 1 (low-carbon) is the most stable on both strands.

### 9.5 Finding 4: Mold Width > Casting Speed for Operating Envelope

**The data shows:** R²(Width) = 0.20 vs R²(Vc) = 0.04. FC Mold G5 lookup table entries should be primarily indexed by mold width.

### 9.6 Finding 5: Strand 6 Model Degrades Under Disturbance — Strand Asymmetry

**The data shows (Section 8):** The all-data XGBoost R² drops from 0.945→0.879 on Strand 6 but holds at 0.946→0.948 on Strand 5. This means Strand 6 disturbance dynamics are more complex and less predictable from the standard feature set.

**This means:** The temporal coupling (Granger causality) is also strand-dependent (strong on S6, weak on S5). Together these suggest local factors — SEN alignment, mold geometry, or flow patterns — differ between strands and affect how disturbances propagate.

### 9.7 Finding 6: Argon Flow and DC Current Shift Under Disturbance

**The data shows (Section 8):** Disturbed sequences have 1.78× higher argon flow (6.2→11.1 NL/min) and **lower** EMBR DC Master current (0.67×). Lower DC braking current during disturbance episodes suggests either the lookup table is not compensating for the higher flow, or the clogging-driven argon increase occurs in conditions where DC is already low.

**Implications:** The correlation between high argon and low DC is a specific operating regime that should be reviewed in the FC Mold G5 lookup tables.

---
## 10. Actionable Recommendations

### Priority 1: Reduce SEN Depth Variability (Highest Impact)

**Target:** Reduce `SEN_std` from current mean of ~3 mm to < 1 mm within 5-minute windows.

**Evidence:** #1 SHAP feature when disturbances included (Section 8); #2 on clean baseline. SENσ doubles from 1.2→2.3 mm in disturbed sequences.

**Actions:**
* Audit stopper rod control loop parameters — tighten the proportional/integral gains
* Inspect nozzle holder mechanical play and alignment
* Consider SEN depth rate-of-change limits in the PLC
* Collaborate with operators: SEN adjustments for clogging are necessary, but the rate and magnitude of change matters

### Priority 2: Investigate EMBR Failure Regimes (Joint ABB/TATA Action)

**Target:** Address the specific EMBR current combinations where the FC Mold G5 failed to compensate.

**Evidence:** Section 8 reveals disturbed sequences have **lower DC braking current** (0.67×) despite higher argon flow (1.78×). This high-argon/low-DC regime is a specific vulnerability.

**Actions:**
* Review the high-instability cluster profiles (Section 7) for EMBR parameter regions with >30% unstable sequences
* Cross-reference these EMBR settings against the current lookup tables
* Review the high-argon/low-DC regime identified in Section 8
* Consider adaptive EMBR control that modulates currents based on real-time SEN variability

### Priority 3: Grade-Specific FC Mold Tuning

**Target:** Reduce instability rate for HSLA grades (Family 5) and peritectic grades (Family 3).

**Actions:**
* Review FC Mold G5 lookup tables for Family 5 (HSLA) and Family 3 (peritectic) grades
* Consider a steel-grade input to the FC Mold G5 control logic

### Priority 4: Investigate Strand 6 Asymmetry

**Target:** Understand why Strand 6 shows stronger temporal coupling (Granger F=210 vs F=4.8) but worse model degradation under disturbance (R² drop 0.066 vs 0.002).

**Actions:**
* Physical inspection of Strand 6 SEN alignment and mold geometry
* Compare FC Mold G5 lookup table entries between S5 and S6 for the same width/speed/grade

### Priority 5: Explore FBG Feedforward Control (Longer-Term)

**Target:** Use the 4-second Granger lead on Strand 6 for pre-emptive field modulation.

---
## 11. Methodology Appendix

### A.1 Software and Tools

| Tool | Version | Purpose |
| --- | --- | --- |
| Databricks Runtime | 16.3 LTS | Spark execution environment |
| PySpark | 3.5 | Large-scale data joining and feature engineering |
| Pandas | 2.x | Per-strand analysis and sequence statistics |
| Plotly | 5.x | Interactive HTML figures |
| Matplotlib | 3.x | Static PNG figure export |
| XGBoost | 2.x | Gradient boosted tree models |
| SHAP | 0.44+ | Model interpretability |
| scikit-learn | 1.4+ | HDBSCAN, t-SNE, StandardScaler, cross-validation |
| SciPy | 1.x | Statistical tests (KS, Spearman, Kruskal-Wallis) |
| python-docx | 1.x | Word document generation |

### A.2 Statistical Methods — Plain-Language Guide

This section explains the statistical tools used in this report in accessible terms.

#### p-value

**What it measures:** The probability that the observed result (or something more extreme) would occur by pure chance if there were truly no effect.

**How to read it:**
* p < 0.001: Extremely strong evidence of a real effect.
* p < 0.01: Strong evidence.
* p < 0.05: Moderate evidence. Conventionally considered “statistically significant”.
* p > 0.05: Insufficient evidence to conclude there’s a real effect.

#### Spearman ρ (rho) — Rank Correlation

**What it measures:** How strongly two variables move together, without assuming linearity. Uses ranks rather than raw values.

**Scale:** Ranges from -1 to +1. |ρ| > 0.5: Strong; 0.3–0.5: Moderate; < 0.3: Weak.

#### Kruskal-Wallis Test

**What it measures:** Whether three or more groups have different medians (non-parametric ANOVA alternative).

#### Granger Causality

**What it measures:** Whether past values of variable A help predict future values of B beyond B’s own past. F-statistic measures strength; higher = stronger temporal precedence. *Not* physical causation.

#### SHAP (SHapley Additive exPlanations)

**What it measures:** Each feature’s contribution to a prediction, based on cooperative game theory. Mean |SHAP| = global feature importance.

#### HDBSCAN

**What it is:** Unsupervised density-based clustering that discovers the number of clusters automatically, handles non-convex shapes, and identifies noise.

#### t-SNE

**What it is:** Dimensionality reduction that maps high-dimensional data to 2D while preserving local neighborhood structure. Distances between distant clusters are not meaningful.

### A.3 Reproducibility

All analyses are reproducible by running this notebook end-to-end. Random seeds are fixed (42) for XGBoost, t-SNE, and cross-validation splits.

In [0]:
# Figure export uses matplotlib (always available) — no extra installs needed
print('matplotlib available for PNG export')

In [0]:
# =====================================================================
# V2.3 COMPLETE Figure Export (replaces V2.2 export)
# Fixes: SEN_std column name, naming convention, colors
# Adds: Fig 1.2 raw signals, Fig 1.2b cascade, Fig 1.3 L/R,
#       Fig 3.1c/d/e, Fig 3.2, Fig 3.4 fixed, Fig 3.5 new colors,
#       Fig 4.1b rolling, Fig 4.2b scatter
# =====================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shutil, os
import numpy as np
from scipy import stats as sp_stats
from scipy import signal as sp_signal
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
os.makedirs(fig_dir, exist_ok=True)

def savefig(fig, name, dpi=150):
    tmp = f'/tmp/{name}'
    fig.savefig(tmp, dpi=dpi, bbox_inches='tight', facecolor='white')
    shutil.copy2(tmp, os.path.join(fig_dir, name))
    plt.close(fig)
    print(f'  \u2714 {name}')

# V2.3 consistent palette
PAL = {'stable': '#2166AC', 'medium': '#4DAC26', 'unstable': '#D6604D',
       'S5': '#e74c3c', 'S6': '#3498db', 'normal': '#27ae60',
       'hv': '#f39c12', 'exc': '#e74c3c', 'bump': '#8e44ad'}
ar_col = 'ArFlow_avg [NL/min]'
sen_std_col = 'SEN_std [mm]'  # V2.3 fix

print('=' * 70)
print('V2.3 COMPLETE FIGURE EXPORT')
print('=' * 70)

# ══════════════════════════════════════════════════════════════════════
# SECTION 1 FIGURES
# ══════════════════════════════════════════════════════════════════════

# ── Fig 1.1: Disturbance type boxplots ───────────────────────────────
fig11, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
dc = {'Normal': PAL['normal'], 'High variability': PAL['hv'],
      'Excursion + High variability': PAL['exc'],
      'Excursion + High variability + Transient bump': PAL['bump']}
for ax, df_a, sn in [(ax1, df_seq_5_all, 'Strand 5'), (ax2, df_seq_6_all, 'Strand 6')]:
    pos = 0
    for dt, clr in dc.items():
        v = df_a[df_a['disturbance_type'] == dt][target].values
        if len(v) > 0:
            ax.boxplot([v], positions=[pos], widths=0.6, patch_artist=True,
                       boxprops=dict(facecolor=clr, alpha=0.6), showfliers=True,
                       flierprops=dict(markersize=2))
        pos += 1
    ax.set_ylabel('ML \u03c3 [mm]'); ax.set_title(sn)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['Normal', 'High var', 'Exc+HV', 'All+Bump'], fontsize=7, rotation=10)
    ax.axhline(y=ML_THRESHOLD, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label=f'±{ML_THRESHOLD:.0f} mm threshold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
fig11.suptitle('Fig 1.1 \u2014 ML \u03c3 by Disturbance Type (All Sequences)', fontsize=13)
fig11.tight_layout()
savefig(fig11, 'fig_1_1_disturbance.png')

# ── Fig 1.2: Raw time-series examples per disturbance type (Strand 6) ─
dist_types = ['Normal', 'High variability', 'Excursion + High variability',
              'Excursion + High variability + Transient bump']
short_names = ['Normal', 'High Variability', 'Excursion + HV', 'Exc + HV + Bump']
dist_colors_list = [PAL['normal'], PAL['hv'], PAL['exc'], PAL['bump']]

fig12, axes12 = plt.subplots(2, 2, figsize=(16, 10))
for idx_d, (dt, sn, clr) in enumerate(zip(dist_types, short_names, dist_colors_list)):
    ax = axes12.flat[idx_d]
    cand = df_seq_6_all[df_seq_6_all['disturbance_type'] == dt]
    if len(cand) == 0:
        ax.text(0.5, 0.5, f'No {sn} sequences', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(sn); continue
    # Pick a representative example (median ML sigma)
    median_idx = cand.iloc[(cand[target] - cand[target].median()).abs().argsort()[:1]].index[0]
    t_s = cand.loc[median_idx, 'Seq_time_Start']
    t_e = cand.loc[median_idx, 'Seq_time_End']
    seg = df_fc_6[(df_fc_6['plainTimeStamp'] >= t_s) & (df_fc_6['plainTimeStamp'] <= t_e)].sort_values('plainTimeStamp')
    if len(seg) > 5 and 'Mold Level' in seg.columns:
        t_rel = (seg['plainTimeStamp'] - seg['plainTimeStamp'].iloc[0]).dt.total_seconds()
        ax.plot(t_rel, seg['Mold Level'], lw=0.6, c=clr)
    ax.set_title(f'{sn} (ML\u03c3={cand.loc[median_idx, target]:.2f} mm)', fontsize=10)
    ax.set_xlabel('Time [s]'); ax.set_ylabel('Mold Level [mm]')
    ax.grid(alpha=0.3)
fig12.suptitle('Fig 1.2 \u2014 Strand 6: Raw Mold Level by Disturbance Type', fontsize=13)
fig12.tight_layout()
savefig(fig12, 'fig_1_2_raw_signals.png')

# ── Fig 1.2b: Data filtering cascade ─────────────────────────────────
fig_fun, (axf1, axf2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, sname, n_raw, n_fc, n_all, n_clean in [
    (axf1, 'Strand 5', len(df_raw_5) if 'df_raw_5' in dir() else 389000,
     len(df_fc_5), len(df_seq_5_all), len(df_seq_5)),
    (axf2, 'Strand 6', len(df_raw_6) if 'df_raw_6' in dir() else 396000,
     len(df_fc_6), len(df_seq_6_all), len(df_seq_6))]:
    stages = ['Raw 1Hz data', 'FC Mold active', 'All normal seq.', 'Clean (no disturb.)']
    counts = [n_raw, n_fc, n_all, n_clean]
    colors_f = ['#bdc3c7', '#7f8c8d', '#2980b9', PAL['normal']]
    bars = ax.barh(stages[::-1], [c for c in counts[::-1]], color=colors_f[::-1], edgecolor='white')
    for bar, c in zip(bars, counts[::-1]):
        ax.text(bar.get_width() + max(counts)*0.02, bar.get_y() + bar.get_height()/2,
                f'{c:,}', va='center', fontsize=9)
    ax.set_title(sname, fontsize=11)
    ax.set_xlabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
fig_fun.suptitle('Fig 1.2b \u2014 Data Filtering Cascade', fontsize=13)
fig_fun.tight_layout()
savefig(fig_fun, 'fig_1_2b_filter_cascade.png')

# ── Fig 1.3: L/R asymmetry ───────────────────────────────────────────
fig13, axes13 = plt.subplots(2, 2, figsize=(14, 10))
for df_s, sname, c in [(df_seq_5, 'S5', PAL['S5']), (df_seq_6, 'S6', PAL['S6'])]:
    if 'ML_LR_asym_std' in df_s.columns:
        axes13[0, 0].hist(df_s['ML_LR_asym_std'].dropna(), bins=50, alpha=0.5, color=c, label=sname)
    if 'ML_LR_asym_std' in df_s.columns:
        axes13[0, 1].scatter(df_s['ML_LR_asym_std'], df_s[target], s=4, alpha=0.3, c=c, label=sname)
axes13[0, 0].set_xlabel('|ML Left \u2212 ML Right| \u03c3'); axes13[0, 0].set_ylabel('Count')
axes13[0, 0].set_title('Distribution of ML L-R Asymmetry'); axes13[0, 0].legend()
axes13[0, 1].set_xlabel('ML L-R Asymmetry \u03c3'); axes13[0, 1].set_ylabel('ML \u03c3 [mm]')
axes13[0, 1].set_title('L-R Asymmetry vs ML \u03c3'); axes13[0, 1].legend()
for df_fc_lr, sname, c, ax_lr in [(df_fc_5, 'S5', PAL['S5'], axes13[1, 0]),
                                    (df_fc_6, 'S6', PAL['S6'], axes13[1, 1])]:
    ml_cols = [col for col in df_fc_lr.columns if 'Mold Level' in col and ('Left' in col or 'Right' in col)]
    if len(ml_cols) >= 2:
        left_col = [c2 for c2 in ml_cols if 'Left' in c2][0]
        right_col = [c2 for c2 in ml_cols if 'Right' in c2][0]
        sample = df_fc_lr[[left_col, right_col]].dropna().sample(min(5000, len(df_fc_lr)))
        ax_lr.scatter(sample[left_col], sample[right_col], s=2, alpha=0.2, c=c)
        lim = [min(sample[left_col].min(), sample[right_col].min()),
               max(sample[left_col].max(), sample[right_col].max())]
        ax_lr.plot(lim, lim, 'k--', lw=1, alpha=0.5)
    ax_lr.set_xlabel('ML Left [mm]'); ax_lr.set_ylabel('ML Right [mm]')
    ax_lr.set_title(f'{sname}: ML Left vs Right')
for ax in axes13.flat: ax.grid(alpha=0.3)
fig13.suptitle('Fig 1.3 \u2014 Mold Level Left/Right Sensor Asymmetry', fontsize=13)
fig13.tight_layout()
savefig(fig13, 'fig_1_3_lr_asymmetry.png')

print('\n  Section 1 figures done.')

# ══════════════════════════════════════════════════════════════════════
# SECTION 3 FIGURES
# ══════════════════════════════════════════════════════════════════════

# ── Fig 3.1a/b: Stable vs Unstable raw signals ──────────────────────
cols_4 = ['Mold Level', 'castingSpeed', 'SENImmersionDepth', 'meniscus_bff_avg']
labels_4 = ['Mold Level [mm]', 'Casting Speed [m/min]', 'SEN Depth [mm]',
            'Broad Face FBG Temp [\u00b0C]']  # V2.3 naming
colors_4 = ['#2c3e50', '#27ae60', '#d35400', '#e74c3c']

for df_fc, df_s, sname in [(df_fc_6, df_seq_6, 'Strand 6')]:
    stable_idx = df_s.nsmallest(5, target).index[0]
    unstable_cand = df_s[df_s[target] > 2.5]
    unstable_idx = unstable_cand.index[0] if len(unstable_cand) > 0 else df_s.nlargest(5, target).index[0]
    for seg_name, idx in [('stable', stable_idx), ('unstable', unstable_idx)]:
        start = df_s.loc[idx, 'Seq_time_Start']
        end = df_s.loc[idx, 'Seq_time_End']
        seg = df_fc[(df_fc['plainTimeStamp'] >= start) & (df_fc['plainTimeStamp'] <= end)].sort_values('plainTimeStamp')
        ml_sigma = df_s.loc[idx, target]
        if len(seg) < 10:
            print(f'  \u26a0 {seg_name}: only {len(seg)} rows, skipping'); continue
        avail = [(c, l, clr) for c, l, clr in zip(cols_4, labels_4, colors_4) if c in seg.columns]
        n = len(avail)
        fig_s, axes_s = plt.subplots(n, 1, figsize=(12, 2.5 * n), sharex=True)
        if not hasattr(axes_s, '__len__'): axes_s = [axes_s]
        for ax_j, (col, label, clr) in enumerate(avail):
            axes_s[ax_j].plot(seg['plainTimeStamp'], seg[col], linewidth=0.8, color=clr)
            axes_s[ax_j].set_ylabel(label, fontsize=9)
            axes_s[ax_j].grid(alpha=0.3)
            if col == 'meniscus_bff_avg' and 'meniscus_bfl_avg' in seg.columns:
                axes_s[ax_j].plot(seg['plainTimeStamp'], seg['meniscus_bfl_avg'],
                                 lw=0.8, c='#3498db', alpha=0.7)
                axes_s[ax_j].legend(['BFF (Fixed)', 'BFL (Loose)'], fontsize=7)
        axes_s[0].set_title(f'{sname} \u2014 {seg_name.upper()} (ML\u03c3={ml_sigma:.2f} mm)', fontsize=12)
        fig_s.tight_layout()
        savefig(fig_s, f'fig_3_1_{seg_name}.png')

# ── Fig 3.1c: BFF temperature by stability quartile ─────────────────
fig_3c, (ax_c1, ax_c2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_s, sname in [(ax_c1, df_seq_5, 'S5'), (ax_c2, df_seq_6, 'S6')]:
    q_lo, q_hi = df_s[target].quantile(0.20), df_s[target].quantile(0.80)
    cats = [('Stable', df_s[df_s[target] < q_lo], PAL['stable']),
            ('Medium', df_s[(df_s[target] >= q_lo) & (df_s[target] < q_hi)], PAL['medium']),
            ('Unstable', df_s[df_s[target] >= q_hi], PAL['unstable'])]
    if 'meniscus_bff_avg' in df_s.columns:
        for lb, sub, cl in cats:
            vals = sub['meniscus_bff_avg'].dropna()
            if len(vals) > 0:
                ax.hist(vals, bins=30, alpha=0.5, color=cl, label=f'{lb} (n={len(vals)})')
    ax.set_xlabel('BFF avg [\u00b0C]'); ax.set_ylabel('Count'); ax.set_title(sname); ax.legend(fontsize=8)
fig_3c.suptitle('Fig 3.1c \u2014 Broad Face FBG Temperature by Stability Category', fontsize=12)
fig_3c.tight_layout()
savefig(fig_3c, 'fig_3_1c_bff_explore.png')

# ── Fig 3.1d: BFF vs BFL comparison ──────────────────────────────────
fig_3d, (ax_d1, ax_d2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_s, sname, c in [(ax_d1, df_seq_5, 'S5', PAL['S5']),
                            (ax_d2, df_seq_6, 'S6', PAL['S6'])]:
    if 'meniscus_bff_avg' in df_s.columns and 'meniscus_bfl_avg' in df_s.columns:
        ax.scatter(df_s['meniscus_bff_avg'], df_s['meniscus_bfl_avg'], s=5, alpha=0.3,
                   c=df_s[target], cmap='RdYlGn_r', vmin=0, vmax=3)
        lim = [min(df_s['meniscus_bff_avg'].min(), df_s['meniscus_bfl_avg'].min()),
               max(df_s['meniscus_bff_avg'].max(), df_s['meniscus_bfl_avg'].max())]
        ax.plot(lim, lim, 'k--', lw=1, alpha=0.5)
    ax.set_xlabel('BFF avg [\u00b0C]'); ax.set_ylabel('BFL avg [\u00b0C]'); ax.set_title(sname)
fig_3d.suptitle('Fig 3.1d \u2014 BFF vs BFL Temperature (color = ML\u03c3)', fontsize=12)
fig_3d.tight_layout()
savefig(fig_3d, 'fig_3_1d_bff_explore.png')

# ── Fig 3.1e: BFF-BFL asymmetry distribution ────────────────────────
fig_3e, (ax_e1, ax_e2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_s, sname, c in [(ax_e1, df_seq_5, 'S5', PAL['S5']),
                            (ax_e2, df_seq_6, 'S6', PAL['S6'])]:
    if 'meniscus_FF_LF_asym_std' in df_s.columns:
        ax.hist(df_s['meniscus_FF_LF_asym_std'].dropna(), bins=40, alpha=0.6, color=c)
        ax.axvline(df_s['meniscus_FF_LF_asym_std'].median(), ls='--', c='black', label='Median')
    ax.set_xlabel('BFF-BFL Asymmetry \u03c3'); ax.set_ylabel('Count'); ax.set_title(sname)
    ax.legend(fontsize=8)
fig_3e.suptitle('Fig 3.1e \u2014 Fixed vs Loose Face Asymmetry Distribution', fontsize=12)
fig_3e.tight_layout()
savefig(fig_3e, 'fig_3_1e_bff_explore.png')

# ── Fig 3.2: Vc × Width inverse problem ─────────────────────────────
vc_col = 'CASTING_SPEED_avg [m/min]'
w_col = 'MOLD_WIDTH_avg [m]'
df_both_fig = pd.concat([df_seq_5.assign(strand='S5'), df_seq_6.assign(strand='S6')])

fig32, axes32 = plt.subplots(1, 3, figsize=(18, 5))
# Panel a: scatter
sc32 = axes32[0].scatter(df_both_fig[vc_col], df_both_fig[w_col], s=5, alpha=0.3,
                          c=df_both_fig[target], cmap='RdYlGn_r', vmin=0, vmax=3)
plt.colorbar(sc32, ax=axes32[0], label='ML\u03c3 [mm]')
axes32[0].set_xlabel('Casting Speed [m/min]'); axes32[0].set_ylabel('Mold Width [m]')
axes32[0].set_title('(a) Vc vs Width (color=ML\u03c3)')
# Panel b: Width bins → ML sigma
width_bins = pd.cut(df_both_fig[w_col], bins=5)
for wb in sorted(width_bins.unique()):
    sub = df_both_fig[width_bins == wb]
    if len(sub) > 5:
        axes32[1].scatter(sub[vc_col], sub[target], s=3, alpha=0.3, label=str(wb)[:10])
axes32[1].set_xlabel('Casting Speed [m/min]'); axes32[1].set_ylabel('ML\u03c3 [mm]')
axes32[1].set_title('(b) Width-stratified Vc effect'); axes32[1].legend(fontsize=6, ncol=2)
# Panel c: R² decomposition bar chart
from sklearn.linear_model import LinearRegression
r2_vals = {}
for name, cols in [('Vc', [vc_col]), ('Width', [w_col]), ('Vc+Width', [vc_col, w_col])]:
    valid = df_both_fig[cols + [target]].dropna()
    if len(valid) > 10:
        lr = LinearRegression().fit(valid[cols], valid[target])
        r2_vals[name] = lr.score(valid[cols], valid[target])
if r2_vals:
    bars = axes32[2].bar(r2_vals.keys(), r2_vals.values(), color=[PAL['S5'], PAL['S6'], '#8e44ad'])
    for b in bars:
        axes32[2].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
                       f'{b.get_height():.3f}', ha='center', fontsize=9)
    axes32[2].set_ylabel('R\u00b2'); axes32[2].set_title('(c) Variance Explained')
fig32.suptitle('Fig 3.2 \u2014 Vc \u00d7 Width Inverse Problem Decomposition', fontsize=13)
fig32.tight_layout()
savefig(fig32, 'fig_3_2_vc_width.png')

print('\n  Section 3.1-3.2 figures done.')

# ── Fig 3.3: EMBR failure analysis (V2.3 FIXED) ─────────────────────
fig_ef, axes_ef = plt.subplots(2, 3, figsize=(18, 10))
for i, (df_s_e, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    df_s_e = df_s_e.copy()
    # V2.3: Use percentile threshold (86th = top 14%)
    thresh = df_s_e[target].quantile(0.86)
    df_s_e['unstable'] = df_s_e[target] > thresh
    embr_ac_c = 'EMBR_AC_Master_LR_mean'
    embr_dc_c = 'EMBR_DC_Master_LR_mean'
    # Panel a: EMBR currents colored by ML sigma
    if embr_ac_c in df_s_e.columns and embr_dc_c in df_s_e.columns:
        sc = axes_ef[i, 0].scatter(df_s_e[embr_ac_c], df_s_e[embr_dc_c],
                                    c=df_s_e[target], cmap='YlOrRd', s=5, vmin=0, vmax=5, alpha=0.6)
        axes_ef[i, 0].set_xlabel('AC Master LR mean [A]'); axes_ef[i, 0].set_ylabel('DC Master LR mean [A]')
        axes_ef[i, 0].set_title(f'{sname}: EMBR Currents (color=ML\u03c3)')
        plt.colorbar(sc, ax=axes_ef[i, 0], label='ML\u03c3 [mm]')
    # Panel b: SEN sigma vs EMBR DC (V2.3: use correct column name)
    if sen_std_col in df_s_e.columns and embr_dc_c in df_s_e.columns:
        stable_e = df_s_e[~df_s_e['unstable']]
        unstable_e = df_s_e[df_s_e['unstable']]
        axes_ef[i, 1].scatter(stable_e[sen_std_col], stable_e[embr_dc_c],
                              s=3, alpha=0.3, c=PAL['stable'], label='Stable')
        axes_ef[i, 1].scatter(unstable_e[sen_std_col], unstable_e[embr_dc_c],
                              s=8, alpha=0.6, c=PAL['unstable'], label=f'Unstable (p{86})')
        axes_ef[i, 1].set_xlabel('SEN \u03c3 [mm]'); axes_ef[i, 1].set_ylabel('DC Master LR mean [A]')
        axes_ef[i, 1].set_title(f'{sname}: SEN\u03c3 vs EMBR DC'); axes_ef[i, 1].legend(fontsize=8)
    # Panel c: SEN sigma vs ML sigma (V2.3: use correct column name)
    if sen_std_col in df_s_e.columns:
        axes_ef[i, 2].scatter(df_s_e[sen_std_col], df_s_e[target],
                              s=4, alpha=0.3, c=df_s_e[target], cmap='RdYlGn_r', vmin=0, vmax=3)
        axes_ef[i, 2].set_xlabel('SEN \u03c3 [mm]'); axes_ef[i, 2].set_ylabel('ML \u03c3 [mm]')
        axes_ef[i, 2].set_title(f'{sname}: SEN\u03c3 vs ML\u03c3')
    else:
        axes_ef[i, 2].text(0.5, 0.5, f'Column {sen_std_col} not found',
                           ha='center', va='center', transform=axes_ef[i, 2].transAxes)
for ax in axes_ef.flat: ax.grid(alpha=0.3)
fig_ef.suptitle('Fig 3.3 \u2014 EMBR Failure Analysis (V2.3: 86th percentile threshold)', fontsize=13)
fig_ef.tight_layout()
savefig(fig_ef, 'fig_3_3_embr_failure.png')

# ── Fig 3.4: Steel grade + Argon + SEN (V2.3: 4 panels, grade table) ─
df_cmp = pd.concat([df_seq_5.assign(strand='S5'), df_seq_6.assign(strand='S6')])
if 'quality_family' not in df_cmp.columns:
    if 'Quality casting' in df_cmp.columns:
        df_cmp['quality_family'] = df_cmp['Quality casting'].astype(str).str[0]

fig34, axes34 = plt.subplots(2, 2, figsize=(14, 10))
# (a) Grade boxplot
if 'quality_family' in df_cmp.columns:
    grades = sorted(df_cmp['quality_family'].dropna().unique())
    for si, (sn, c) in enumerate([('S5', PAL['S5']), ('S6', PAL['S6'])]):
        sub = df_cmp[df_cmp['strand'] == sn]
        data = [sub[sub['quality_family'] == g][target].dropna().values for g in grades]
        bp = axes34[0, 0].boxplot(data, positions=[i + si * 0.35 for i in range(len(grades))],
                                   widths=0.3, patch_artist=True, showfliers=False)
        for patch in bp['boxes']: patch.set_facecolor(c); patch.set_alpha(0.5)
    axes34[0, 0].set_xticks(range(len(grades))); axes34[0, 0].set_xticklabels(grades)
    axes34[0, 0].set_xlabel('Grade Family'); axes34[0, 0].set_ylabel('ML \u03c3 [mm]')
    axes34[0, 0].set_title('(a) ML\u03c3 by Steel Grade Family')
# (b) Argon scatter
if ar_col in df_cmp.columns:
    for sn, c in [('S5', PAL['S5']), ('S6', PAL['S6'])]:
        sub = df_cmp[df_cmp['strand'] == sn]
        axes34[0, 1].scatter(sub[ar_col], sub[target], s=4, alpha=0.3, c=c, label=sn)
    axes34[0, 1].set_xlabel('Argon Flow [NL/min]'); axes34[0, 1].set_ylabel('ML \u03c3 [mm]')
    axes34[0, 1].set_title('(b) Argon Flow vs ML\u03c3'); axes34[0, 1].legend(fontsize=8)
# (c) SEN std vs ML sigma (V2.3 FIXED)
if sen_std_col in df_cmp.columns:
    for sn, c in [('S5', PAL['S5']), ('S6', PAL['S6'])]:
        sub = df_cmp[df_cmp['strand'] == sn]
        axes34[1, 0].scatter(sub[sen_std_col], sub[target], s=4, alpha=0.3, c=c, label=sn)
    axes34[1, 0].set_xlabel('SEN \u03c3 [mm]'); axes34[1, 0].set_ylabel('ML \u03c3 [mm]')
    axes34[1, 0].set_title('(c) SEN Variability vs ML\u03c3'); axes34[1, 0].legend(fontsize=8)
# (d) Confounding: Grade vs SEN std
if 'quality_family' in df_cmp.columns and sen_std_col in df_cmp.columns:
    for si, (sn, c) in enumerate([('S5', PAL['S5']), ('S6', PAL['S6'])]):
        sub = df_cmp[df_cmp['strand'] == sn]
        data = [sub[sub['quality_family'] == g][sen_std_col].dropna().values for g in grades]
        bp = axes34[1, 1].boxplot(data, positions=[i + si * 0.35 for i in range(len(grades))],
                                   widths=0.3, patch_artist=True, showfliers=False)
        for patch in bp['boxes']: patch.set_facecolor(c); patch.set_alpha(0.5)
    axes34[1, 1].set_xticks(range(len(grades))); axes34[1, 1].set_xticklabels(grades)
    axes34[1, 1].set_xlabel('Grade Family'); axes34[1, 1].set_ylabel('SEN \u03c3 [mm]')
    axes34[1, 1].set_title('(d) Confounding: Grade vs SEN\u03c3')
for ax in axes34.flat: ax.grid(alpha=0.3)
fig34.suptitle('Fig 3.4 \u2014 Steel Grade, Argon, and SEN Impact on ML\u03c3 (V2.3)', fontsize=13)
fig34.tight_layout()
savefig(fig34, 'fig_3_4_grade_argon.png')

# ── Fig 3.5: Meniscus profiles (V2.3 colors) ────────────────────────
def cheb_T1(z): return z
def cheb_T2(z): return 2 * z**2 - 1
def cheb_T3(z): return 4 * z**3 - 3 * z
def cheb_T4(z): return 8 * z**4 - 8 * z**2 + 1
vuhz_pos = 265 / (1800 / 2)
n_pts = 200; x_norm = np.linspace(-1, 1, n_pts)
cc = ['cheb_X1_bff_mean', 'cheb_X2_bff_mean', 'cheb_X3_bff_mean', 'cheb_X4_bff_mean']

for df_s, sname, strand_lab in [(df_seq_5, 'Strand 5', 'S5'), (df_seq_6, 'Strand 6', 'S6')]:
    profs = np.zeros((len(df_s), n_pts))
    for i in range(len(df_s)):
        r = df_s.iloc[i]
        if all(c in r.index for c in cc):
            profs[i] = (r[cc[0]] * cheb_T1(x_norm) + r[cc[1]] * cheb_T2(x_norm) +
                        r[cc[2]] * cheb_T3(x_norm) + r[cc[3]] * cheb_T4(x_norm))
    ml_s = df_s[target]
    q_lo, q_hi = ml_s.quantile(0.20), ml_s.quantile(0.80)
    cats = [('Stable (bottom 20%)', ml_s < q_lo, PAL['stable']),
            ('Medium (20-80%)', (ml_s >= q_lo) & (ml_s < q_hi), PAL['medium']),
            ('Unstable (top 20%)', ml_s >= q_hi, PAL['unstable'])]
    fig35, ax35 = plt.subplots(figsize=(10, 5))
    for lb, mk, cl in cats:
        for idx in df_s[mk].index[:40]:
            ax35.plot(x_norm, profs[df_s.index.get_loc(idx)], c=cl, alpha=0.1, lw=0.5)
        if mk.sum() > 0:
            ax35.plot(x_norm, profs[mk.values].mean(axis=0), c=cl, lw=3, label=f'{lb} (n={mk.sum()})')
    ax35.axvline(-vuhz_pos, ls='--', c='blue', alpha=0.5, label='Vuhz sensors')
    ax35.axvline(vuhz_pos, ls='--', c='blue', alpha=0.5)
    ax35.set_xlabel('Normalised Mold Width'); ax35.set_ylabel('Meniscus Height (a.u.)')
    ax35.set_title(f'Fig 3.5 \u2014 {sname}: Three-Category Meniscus Profiles (BFF)')
    ax35.legend(fontsize=8); ax35.grid(alpha=0.3)
    savefig(fig35, f'fig_3_5_{strand_lab}_profiles.png')

# Also export the single-strand S6 version for backward compat
savefig_done = True  # fig_3_5a already saved as fig_3_5_S6_profiles.png above

# ── Fig 3.5b: PtP location ──────────────────────────────────────────
if 'ptp_x_max' in df_seq_6.columns:
    fig35b, (ax_p1, ax_p2) = plt.subplots(1, 2, figsize=(12, 5))
    sc1 = ax_p1.scatter(df_seq_6['ptp_x_max'], df_seq_6[target], s=8,
                        c=df_seq_6[target], cmap='RdYlGn_r', vmin=0, vmax=3, alpha=0.6)
    ax_p1.set_xlabel('Normalised Mold Width'); ax_p1.set_ylabel('ML \u03c3 [mm]')
    ax_p1.set_title('Peak Location')
    sc2 = ax_p2.scatter(df_seq_6['ptp_x_min'], df_seq_6[target], s=8,
                        c=df_seq_6[target], cmap='RdYlGn_r', vmin=0, vmax=3, alpha=0.6)
    ax_p2.set_xlabel('Normalised Mold Width'); ax_p2.set_title('Trough Location')
    plt.colorbar(sc2, ax=ax_p2, label='ML\u03c3 [mm]')
    fig35b.suptitle('Fig 3.5b \u2014 S6: Wave Peak Location', fontsize=13)
    fig35b.tight_layout()
    savefig(fig35b, 'fig_3_5b_ptp_location.png')

print('\n  Section 3 figures done.')

# ══════════════════════════════════════════════════════════════════════
# SECTION 4 FIGURES
# ══════════════════════════════════════════════════════════════════════

# ── Fig 4.1: Heat progression ────────────────────────────────────────
fig_heat, (ax5h, ax6h) = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_s, sname, c in [(ax5h, df_seq_5, 'S5', PAL['S5']),
                            (ax6h, df_seq_6, 'S6', PAL['S6'])]:
    df_sorted = df_s.sort_values('Seq_time_Start').reset_index(drop=True)
    df_sorted['pos'] = range(1, len(df_sorted) + 1)
    ax.scatter(df_sorted['pos'], df_sorted[target], s=6, alpha=0.3, c=c)
    rolling = df_sorted.set_index('pos')[target].rolling(20, center=True).median()
    ax.plot(rolling.index, rolling.values, c='black', lw=2, label='Rolling median')
    ax.axhline(ML_THRESHOLD, ls='--', c='red', alpha=0.5, label='±1 mm threshold')
    ax.set_xlabel('Sequence Position'); ax.set_ylabel('ML \u03c3 [mm]'); ax.set_title(sname)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    r_s, p_s = sp_stats.spearmanr(df_sorted['pos'], df_sorted[target])
    ax.text(0.02, 0.95, f'\u03c1={r_s:.3f}, p={p_s:.2e}', transform=ax.transAxes, fontsize=8, va='top')
fig_heat.suptitle('Fig 4.1 \u2014 ML \u03c3 Across Casting Campaigns', fontsize=13)
fig_heat.tight_layout()
savefig(fig_heat, 'fig_4_1_heat.png')

# ── Fig 4.1b: Rolling Chebyshev vs ML deviation (V2.3 new) ──────────
fig_41b, axes_41b = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
seg_sample = df_fc_6.sort_values('plainTimeStamp').head(3000)
if 'cheb_X2_bfl' in seg_sample.columns and 'Mold Level' in seg_sample.columns:
    ts = seg_sample['plainTimeStamp']
    axes_41b[0].plot(ts, seg_sample['Mold Level'].rolling(10).std(), lw=0.8, c=PAL['S6'], label='ML rolling \u03c3')
    axes_41b[0].set_ylabel('ML rolling \u03c3 [mm]'); axes_41b[0].legend(); axes_41b[0].grid(alpha=0.3)
    axes_41b[1].plot(ts, seg_sample['cheb_X2_bfl'].rolling(10).std(), lw=0.8, c=PAL['S5'], label='Cheb X\u2082 rolling \u03c3')
    axes_41b[1].set_ylabel('X\u2082 rolling \u03c3'); axes_41b[1].set_xlabel('Time'); axes_41b[1].legend()
    axes_41b[1].grid(alpha=0.3)
fig_41b.suptitle('Fig 4.1b \u2014 Rolling Chebyshev X\u2082 vs Mold Level Deviation', fontsize=13)
fig_41b.tight_layout()
savefig(fig_41b, 'fig_4_1b_rolling_stats.png')

# ── Fig 4.2: Lagged cross-correlation ────────────────────────────────
max_lag = 30
fig_xcorr, axes_xc = plt.subplots(2, 2, figsize=(14, 10))
cheb_labels = ['X\u2081', 'X\u2082', 'X\u2083', 'X\u2084']
cheb_cols = ['cheb_X1_bfl', 'cheb_X2_bfl', 'cheb_X3_bfl', 'cheb_X4_bfl']
for ci, (cheb_col, cheb_lb) in enumerate(zip(cheb_cols, cheb_labels)):
    ax = axes_xc.flat[ci]
    for df_fc_strand, sname, c in [(df_fc_5, 'S5', PAL['S5']), (df_fc_6, 'S6', PAL['S6'])]:
        if cheb_col in df_fc_strand.columns and 'Mold Level' in df_fc_strand.columns:
            x = df_fc_strand[cheb_col].dropna().values[:10000]
            y_ml = df_fc_strand.loc[df_fc_strand[cheb_col].notna(), 'Mold Level'].values[:10000]
            ml_n = min(len(x), len(y_ml))
            x = x[:ml_n]; y_ml = y_ml[:ml_n]
            x = (x - x.mean()) / (x.std() + 1e-10)
            y_ml = (y_ml - y_ml.mean()) / (y_ml.std() + 1e-10)
            corr = np.correlate(y_ml, x, mode='full') / ml_n
            mid = len(corr) // 2
            lags = np.arange(-max_lag, max_lag + 1)
            cw = corr[mid - max_lag:mid + max_lag + 1]
            ax.bar(lags, cw, alpha=0.4, color=c, label=sname, width=0.8)
    ax.set_xlabel('Lag [s]'); ax.set_ylabel('XCorr')
    ax.set_title(f'Chebyshev {cheb_lb}'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig_xcorr.suptitle('Fig 4.2 \u2014 Lagged Cross-Correlation: Chebyshev Shape vs Mold Level', fontsize=13)
fig_xcorr.tight_layout()
savefig(fig_xcorr, 'fig_4_2_xcorr.png')

# ── Fig 4.2b: Scatter + lag illustration (V2.3 new) ─────────────────
fig_42b, axes_42b = plt.subplots(1, 3, figsize=(18, 5))
for df_fc_strand, sname, c in [(df_fc_6, 'S6', PAL['S6'])]:
    if 'cheb_X2_bfl' in df_fc_strand.columns and 'Mold Level' in df_fc_strand.columns:
        sub = df_fc_strand[['cheb_X2_bfl', 'Mold Level']].dropna().iloc[:5000]
        axes_42b[0].scatter(sub['cheb_X2_bfl'], sub['Mold Level'], s=2, alpha=0.2, c=c)
        axes_42b[0].set_xlabel('Cheb X\u2082'); axes_42b[0].set_ylabel('Mold Level [mm]')
        axes_42b[0].set_title(f'{sname}: Contemporaneous')
        axes_42b[1].scatter(sub['cheb_X2_bfl'].iloc[:-4], sub['Mold Level'].iloc[4:], s=2, alpha=0.2, c=c)
        axes_42b[1].set_xlabel('Cheb X\u2082 (t-4s)'); axes_42b[1].set_ylabel('Mold Level (t)')
        axes_42b[1].set_title(f'{sname}: Lag=4s')
        axes_42b[2].scatter(sub['Mold Level'].iloc[:-4], sub['cheb_X2_bfl'].iloc[4:], s=2, alpha=0.2, c=c)
        axes_42b[2].set_xlabel('Mold Level (t-4s)'); axes_42b[2].set_ylabel('Cheb X\u2082 (t)')
        axes_42b[2].set_title(f'{sname}: Reverse Lag=4s')
for ax in axes_42b: ax.grid(alpha=0.3)
fig_42b.suptitle('Fig 4.2b \u2014 Scatter: Shape Precedes Level? (Exploratory)', fontsize=13)
fig_42b.tight_layout()
savefig(fig_42b, 'fig_4_2b_granger_scatter.png')

# ── Fig 4.3: Granger causality ───────────────────────────────────────
max_test_lag = 10
fig_gr, (axG1, axG2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_fc_strand, sname, c in [(axG1, df_fc_5, 'S5', PAL['S5']),
                                    (axG2, df_fc_6, 'S6', PAL['S6'])]:
    try:
        if 'cheb_X2_bfl' in df_fc_strand.columns and 'Mold Level' in df_fc_strand.columns:
            sub = df_fc_strand[['Mold Level', 'cheb_X2_bfl']].dropna().iloc[:5000]
            results = grangercausalitytests(sub.values, maxlag=max_test_lag, verbose=False)
            f_stats = [results[lag + 1][0]['ssr_ftest'][0] for lag in range(max_test_lag)]
            ax.bar(range(1, max_test_lag + 1), f_stats, color=c, alpha=0.7)
            ax.set_xlabel('Lag [s]'); ax.set_ylabel('F-statistic')
            ax.set_title(f'{sname}: X\u2082 \u2192 ML Granger (F-stat)'); ax.grid(alpha=0.3)
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:50], transform=ax.transAxes, ha='center')
fig_gr.suptitle('Fig 4.3 \u2014 Granger Causality: Chebyshev X\u2082 \u2192 Mold Level', fontsize=13)
fig_gr.tight_layout()
savefig(fig_gr, 'fig_4_3_granger.png')

print('\n  Section 4 figures done.')

# ══════════════════════════════════════════════════════════════════════
# SECTION 5-7 FIGURES
# ══════════════════════════════════════════════════════════════════════

# ── Fig 5.1/5.2: Distribution overlays (already exported, skip) ──────
print('  Sections 5.1/5.2: use existing PNGs from V2.2 export')

# ── Fig 6.2: SHAP ────────────────────────────────────────────────────
top_n = 15
imp5 = res_5['shap_imp'].head(top_n); imp6 = res_6['shap_imp'].head(top_n)
all_f = list(dict.fromkeys(imp5['feature'].tolist() + imp6['feature'].tolist()))[:top_n]
m = pd.DataFrame({'feature': all_f})
m = m.merge(imp5.rename(columns={'mean_abs_shap': 'S5'}), on='feature', how='left')
m = m.merge(imp6.rename(columns={'mean_abs_shap': 'S6'}), on='feature', how='left').fillna(0)
m['avg'] = (m['S5'] + m['S6']) / 2; m = m.sort_values('avg')
fig62, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(m))
ax.barh([y - 0.15 for y in y_pos], m['S5'], height=0.3, color=PAL['S5'], alpha=0.7, label='S5')
ax.barh([y + 0.15 for y in y_pos], m['S6'], height=0.3, color=PAL['S6'], alpha=0.7, label='S6')
ax.set_yticks(list(y_pos)); ax.set_yticklabels(m['feature'], fontsize=9)
ax.set_xlabel('Mean |SHAP|')
ax.set_title(f'Fig 6.2 \u2014 SHAP Importance (S5 R\u00b2={res_5["r2"]:.3f}, S6 R\u00b2={res_6["r2"]:.3f})')
ax.legend(); ax.grid(axis='x', alpha=0.3)
savefig(fig62, 'fig_6_2_shap.png')

# ── Fig 6.3: Predicted vs Actual ─────────────────────────────────────
fig63, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, res, c, nm in [(ax1, res_5, PAL['S5'], 'S5'), (ax2, res_6, PAL['S6'], 'S6')]:
    ax.scatter(res['y'], res['oof'], s=6, alpha=0.3, c=c)
    lim = max(res['y'].max(), res['oof'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('Actual ML\u03c3 [mm]'); ax.set_ylabel('Predicted ML\u03c3 [mm]')
    ax.set_title(f'{nm} (R\u00b2={res["r2"]:.3f})'); ax.grid(alpha=0.3)
fig63.suptitle('Fig 6.3 \u2014 Actual vs Predicted', fontsize=13)
fig63.tight_layout()
savefig(fig63, 'fig_6_3_predicted.png')

# ── Fig 7.1a: HDBSCAN clusters ──────────────────────────────────────
fig71a, (ax71a, ax71b) = plt.subplots(1, 2, figsize=(14, 5))
for ax, dc, nm in [(ax71a, df_c5, 'S5'), (ax71b, df_c6, 'S6')]:
    for cl in sorted(dc['cluster'].unique()):
        sub = dc[dc['cluster'] == cl]
        ax.scatter(sub['tsne_1'], sub['tsne_2'], s=6, alpha=0.5, label=f'C{cl}' if cl != -1 else 'Noise')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.set_title(nm); ax.grid(alpha=0.3)
fig71a.suptitle('Fig 7.1a \u2014 HDBSCAN Regime Discovery', fontsize=13)
fig71a.tight_layout()
savefig(fig71a, 'fig_7_1a_clusters.png')

# ── Fig 7.1b: t-SNE colored by ML sigma ─────────────────────────────
vmax = max(df_c5[target].quantile(0.95), df_c6[target].quantile(0.95))
fig71b, (ax_b1, ax_b2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, dc, nm in [(ax_b1, df_c5, 'S5'), (ax_b2, df_c6, 'S6')]:
    sc = ax.scatter(dc['tsne_1'], dc['tsne_2'], s=8, c=dc[target],
                    cmap='viridis', vmin=0, vmax=vmax, alpha=0.6)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.set_title(nm); ax.grid(alpha=0.3)
plt.colorbar(sc, ax=ax_b2, label='ML\u03c3 [mm]')
fig71b.suptitle('Fig 7.1b \u2014 ML\u03c3 in t-SNE', fontsize=13)
fig71b.tight_layout()
savefig(fig71b, 'fig_7_1b_mlsigma.png')

# ── Cluster profile tables as PNG ────────────────────────────────────
for strand, prof in cluster_profiles.items():
    display_cols = ['n', target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]',
                    'dominant_grade', 'unstable_pct']
    avail_cols = [c for c in display_cols if c in prof.columns]
    pdf = prof[avail_cols].reset_index()
    fig_tab, ax_tab = plt.subplots(figsize=(10, 0.4 * len(pdf) + 1))
    ax_tab.axis('off')
    tab = ax_tab.table(cellText=pdf.values, colLabels=pdf.columns, cellLoc='center', loc='center')
    tab.auto_set_font_size(False); tab.set_fontsize(7); tab.scale(1.2, 1.2)
    ax_tab.set_title(f'{strand}: Cluster Profiles (sorted by ML\u03c3)', fontsize=10, pad=20)
    sshort = strand.replace('Strand ', 'S')
    savefig(fig_tab, f'fig_7_2_{sshort}_clusters.png', dpi=200)

pngs = sorted([f for f in os.listdir(fig_dir) if f.startswith('fig_') and f.endswith('.png')])
print(f'\n{"=" * 70}')
print(f'V2.3 FIGURE EXPORT COMPLETE: {len(pngs)} PNGs')
for f in pngs:
    print(f'  {f}')
print(f'{"=" * 70}')

In [0]:
# =====================================================================
# V2.4 FIGURE FIXES — Overwrites problematic PNGs in fig_dir
# Fixes: Fig 1.2 (single-col timestamps), Fig 1.2b (funnel),
#        Fig 3.1c/d (meniscus profiles), Fig 3.5 (both faces),
#        Fig 3.5c (2-level color), Grade family table
# =====================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import shutil, os, numpy as np, pandas as pd
from numpy.polynomial.chebyshev import chebval

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
os.makedirs(fig_dir, exist_ok=True)
target = 'MOLD_LEVEL_std [mm]'
PAL = {'stable': '#2166AC', 'medium': '#4DAC26', 'unstable': '#D6604D',
       'S5': '#e74c3c', 'S6': '#3498db', 'normal': '#27ae60',
       'hv': '#f39c12', 'exc': '#e74c3c', 'bump': '#8e44ad'}

def savefig(fig, name, dpi=150):
    tmp = f'/tmp/{name}'
    fig.savefig(tmp, dpi=dpi, bbox_inches='tight', facecolor='white')
    shutil.copy2(tmp, os.path.join(fig_dir, name))
    plt.close(fig)
    print(f'  \u2714 {name}')

print('V2.4 FIGURE FIXES')
print('=' * 60)

# ══════════════════════════════════════════════════════════════
# FIG 1.2 — Single-column, actual timestamps (4 rows × 1 col)
# ══════════════════════════════════════════════════════════════
print('\n--- Fig 1.2: Raw time-series (single-column, timestamps) ---')
dist_types = ['Normal', 'High variability', 'Excursion + High variability',
              'Excursion + High variability + Transient bump']
short_names = ['Normal', 'High Variability', 'Excursion + HV', 'Exc + HV + Bump']
dist_colors_list = [PAL['normal'], PAL['hv'], PAL['exc'], PAL['bump']]

fig12, axes12 = plt.subplots(4, 1, figsize=(12, 16), constrained_layout=True)
for idx_d, (dt, sn, clr) in enumerate(zip(dist_types, short_names, dist_colors_list)):
    ax = axes12[idx_d]
    cand = df_seq_6_all[df_seq_6_all['disturbance_type'] == dt]
    if len(cand) == 0:
        ax.text(0.5, 0.5, f'No {sn} sequences', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(sn, fontsize=11); continue
    med_idx = cand.iloc[(cand[target] - cand[target].median()).abs().argsort()[:1]].index[0]
    t_s = cand.loc[med_idx, 'Seq_time_Start']
    t_e = cand.loc[med_idx, 'Seq_time_End']
    seg = df_fc_6[(df_fc_6['plainTimeStamp'] >= t_s) &
                  (df_fc_6['plainTimeStamp'] <= t_e)].sort_values('plainTimeStamp')
    if len(seg) > 5 and 'Mold Level' in seg.columns:
        ax.plot(seg['plainTimeStamp'], seg['Mold Level'], lw=0.8, c=clr)
    ax.set_title(f'{sn}  (ML\u03c3 = {cand.loc[med_idx, target]:.2f} mm)', fontsize=11)
    ax.set_ylabel('Mold Level [mm]', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(alpha=0.3)
axes12[-1].set_xlabel('Timestamp (HH:MM:SS)', fontsize=10)
fig12.suptitle('Fig 1.2 \u2014 Strand 6: Raw Mold Level Signal by Disturbance Type',
               fontsize=14, y=1.01)
savefig(fig12, 'fig_1_2_raw_signals.png')

# ══════════════════════════════════════════════════════════════
# FIG 1.2b — Funnel-shaped data filtering cascade
# ══════════════════════════════════════════════════════════════
print('\n--- Fig 1.2b: Funnel cascade ---')
fig_fun, axes_fun = plt.subplots(1, 2, figsize=(14, 7))
for ax, sname, n_raw_all, n_raw, n_fc, n_all, n_clean in [
    (axes_fun[0], 'Strand 5', 509458 if 'df_raw_5' not in dir() else len(df_raw_5),
     len(df_raw_5) if 'df_raw_5' in dir() else 389000,
     len(df_fc_5), len(df_seq_5_all), len(df_seq_5)),
    (axes_fun[1], 'Strand 6', 520526 if 'df_raw_6' not in dir() else len(df_raw_6),
     len(df_raw_6) if 'df_raw_6' in dir() else 396000,
     len(df_fc_6), len(df_seq_6_all), len(df_seq_6))]:
    stages = ['Raw joined data', 'Quality-filtered', 'FC Mold active',
              'Normal sequences', 'Disturbance-free']
    counts = [n_raw_all, n_raw, n_fc, n_all, n_clean]
    colors_f = ['#95a5a6', '#7f8c8d', '#5dade2', '#f39c12', '#27ae60']
    max_w = max(counts)
    n_stages = len(stages)
    for i in range(n_stages):
        w = counts[i] / max_w  # normalised width
        y_center = n_stages - 1 - i
        # Draw centered trapezoid
        if i < n_stages - 1:
            w_next = counts[i + 1] / max_w
        else:
            w_next = w * 0.8
        trap = plt.Polygon([
            (-w / 2, y_center + 0.45), (w / 2, y_center + 0.45),
            (w_next / 2, y_center - 0.45), (-w_next / 2, y_center - 0.45)],
            facecolor=colors_f[i], edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(trap)
        # Label
        pct = 100 * counts[i] / counts[0]
        ax.text(0, y_center, f'{stages[i]}\n{counts[i]:,}  ({pct:.1f}%)',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if i < 3 else 'black')
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(-0.8, n_stages - 0.3)
    ax.set_title(sname, fontsize=12)
    ax.axis('off')
fig_fun.suptitle('Fig 1.2b \u2014 Data Filtering Cascade: From Raw to Analysis-Ready',
                 fontsize=14, y=1.01)
fig_fun.tight_layout()
savefig(fig_fun, 'fig_1_2b_filter_cascade.png')

# ══════════════════════════════════════════════════════════════
# FIG 3.1c/d — Meniscus profiles for Stable/Unstable sequences
# Reconstructed from Chebyshev X1-X4 for BOTH faces (BFF + BFL)
# ══════════════════════════════════════════════════════════════
print('\n--- Fig 3.1c/d: Meniscus profiles (Chebyshev reconstruction) ---')
x_norm = np.linspace(-1, 1, 200)
# Chebyshev polynomials T1-T4
T1 = x_norm
T2 = 2 * x_norm**2 - 1
T3 = 4 * x_norm**3 - 3 * x_norm
T4 = 8 * x_norm**4 - 8 * x_norm**2 + 1

def reconstruct_profile(seg_data, face='bff'):
    """Reconstruct mean meniscus profile from Chebyshev coefs in a raw segment."""
    c1 = seg_data[f'cheb_X1_{face}'].mean()
    c2 = seg_data[f'cheb_X2_{face}'].mean()
    c3 = seg_data[f'cheb_X3_{face}'].mean()
    c4 = seg_data[f'cheb_X4_{face}'].mean()
    profile = c1 * T1 + c2 * T2 + c3 * T3 + c4 * T4
    return profile, (c1, c2, c3, c4)

# Get the same stable/unstable sequences as Fig 3.1a/b (Strand 6)
stable_idx = df_seq_6.nsmallest(5, target).index[0]
unstable_cand = df_seq_6[df_seq_6[target] > 2.5]
unstable_idx = unstable_cand.index[0] if len(unstable_cand) > 0 else df_seq_6.nlargest(5, target).index[0]

for fig_label, seq_idx, condition_label in [
    ('fig_3_1c_profile_stable.png', stable_idx, 'STABLE'),
    ('fig_3_1d_profile_unstable.png', unstable_idx, 'UNSTABLE')]:
    ts = df_seq_6.loc[seq_idx, 'Seq_time_Start']
    te = df_seq_6.loc[seq_idx, 'Seq_time_End']
    ml_sigma = df_seq_6.loc[seq_idx, target]
    seg = df_fc_6[(df_fc_6['plainTimeStamp'] >= ts) &
                  (df_fc_6['plainTimeStamp'] <= te)].copy()
    
    fig_p, axes_p = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax_p, face, face_label, c_main, c_fill in [
        (axes_p[0], 'bff', 'Broad Face Fixed (BFF)', '#2c3e50', '#3498db'),
        (axes_p[1], 'bfl', 'Broad Face Loose (BFL)', '#8b4513', '#d35400')]:
        # Mean profile
        prof_mean, coefs = reconstruct_profile(seg, face)
        ax_p.plot(x_norm, prof_mean, lw=2.5, c=c_main, label='Mean profile')
        # Individual 1s profiles (thin lines)
        for _, row in seg.sample(min(30, len(seg))).iterrows():
            c1 = row[f'cheb_X1_{face}']; c2 = row[f'cheb_X2_{face}']
            c3 = row[f'cheb_X3_{face}']; c4 = row[f'cheb_X4_{face}']
            prof_i = c1 * T1 + c2 * T2 + c3 * T3 + c4 * T4
            ax_p.plot(x_norm, prof_i, lw=0.3, alpha=0.25, c=c_fill)
        ax_p.axvline(-0.29, ls='--', c='gray', lw=1, alpha=0.6, label='Vuhz sensor')
        ax_p.axvline(0.29, ls='--', c='gray', lw=1, alpha=0.6)
        ax_p.set_xlabel('Normalised Mold Width', fontsize=10)
        ax_p.set_title(f'{face_label}', fontsize=11)
        ax_p.legend(fontsize=8, loc='upper right')
        ax_p.grid(alpha=0.3)
        ax_p.text(0.02, 0.95, f'X\u2081={coefs[0]:.2f}  X\u2082={coefs[1]:.2f}\n'
                  f'X\u2083={coefs[2]:.2f}  X\u2084={coefs[3]:.2f}',
                  transform=ax_p.transAxes, fontsize=8, va='top',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes_p[0].set_ylabel('Meniscus Profile [\u00b0C deviation]', fontsize=10)
    suffix = 'c' if condition_label == 'STABLE' else 'd'
    fig_p.suptitle(f'Fig 3.1{suffix} \u2014 Strand 6 {condition_label}: '
                   f'Chebyshev Meniscus Profile (ML\u03c3 = {ml_sigma:.2f} mm)',
                   fontsize=13)
    fig_p.tight_layout()
    savefig(fig_p, fig_label)

# ══════════════════════════════════════════════════════════════
# GRADE FAMILY TABLE — as PNG image
# ══════════════════════════════════════════════════════════════
print('\n--- Table 3.1: Grade family composition ---')
grade_rows = []
for df_s, sn_short in [(df_seq_5, 'S5'), (df_seq_6, 'S6')]:
    if 'quality_family' not in df_s.columns: continue
    for fam in sorted([f for f in df_s['quality_family'].unique() if f is not None]):
        sub = df_s[df_s['quality_family'] == fam]
        grades = sorted([g for g in sub['Quality casting'].unique() if g is not None])
        grade_rows.append([
            f'Family {fam}', ', '.join(grades),
            str(len(sub)), f'{sub[target].median():.3f}'])
# Deduplicate (grades same for S5/S6)
seen_fams = set()
dedup = []
for row in grade_rows:
    if row[0] not in seen_fams:
        seen_fams.add(row[0])
        # Combine counts
        n5 = len(df_seq_5[df_seq_5['quality_family']==row[0].split()[-1]])
        n6 = len(df_seq_6[df_seq_6['quality_family']==row[0].split()[-1]])
        m5 = df_seq_5[df_seq_5['quality_family']==row[0].split()[-1]][target].median()
        m6 = df_seq_6[df_seq_6['quality_family']==row[0].split()[-1]][target].median()
        dedup.append([row[0], row[1], f'{n5} / {n6}',
                      f'{m5:.3f} / {m6:.3f}'])
# Add Unknown
n5u = df_seq_5['quality_family'].isna().sum() + (df_seq_5['quality_family']=='Unknown').sum()
n6u = df_seq_6['quality_family'].isna().sum() + (df_seq_6['quality_family']=='Unknown').sum()
m5u = df_seq_5[df_seq_5['quality_family'].isin([None, 'Unknown']) | df_seq_5['quality_family'].isna()][target].median()
m6u = df_seq_6[df_seq_6['quality_family'].isin([None, 'Unknown']) | df_seq_6['quality_family'].isna()][target].median()
dedup.append(['Unknown', '(grade code not available in metadata)',
              f'{n5u} / {n6u}', f'{m5u:.3f} / {m6u:.3f}'])

fig_gt, ax_gt = plt.subplots(figsize=(14, 3.5))
ax_gt.axis('off')
headers = ['Grade Family', 'Steel Grades', 'n (S5 / S6)', 'Median ML\u03c3 (S5 / S6)']
tab = ax_gt.table(cellText=dedup, colLabels=headers, cellLoc='center', loc='center',
                  colWidths=[0.12, 0.40, 0.18, 0.30])
tab.auto_set_font_size(False); tab.set_fontsize(9)
tab.scale(1.0, 1.5)
for i in range(len(headers)):
    tab[0, i].set_facecolor('#2c3e50')
    tab[0, i].set_text_props(color='white', fontweight='bold')
ax_gt.set_title('Table 3.1 \u2014 Steel Grade Family Composition and Stability',
                fontsize=12, pad=15)
savefig(fig_gt, 'fig_3_grade_table.png', dpi=200)

# ══════════════════════════════════════════════════════════════
# FIG 3.5 — Meniscus profiles: single strand (S6), BOTH faces
# 3 categories, better line sizing
# ══════════════════════════════════════════════════════════════
print('\n--- Fig 3.5: Meniscus profiles (S6, both faces, 3 categories) ---')
df_s = df_seq_6
q_lo, q_hi = df_s[target].quantile(0.20), df_s[target].quantile(0.80)
stab_mask = df_s[target] < q_lo
med_mask = (df_s[target] >= q_lo) & (df_s[target] < q_hi)
unstab_mask = df_s[target] >= q_hi

cats = [
    ('Stable (bottom 20%)', stab_mask, PAL['stable'], 2.5, '-'),
    ('Medium (20\u201380%)', med_mask, PAL['medium'], 2.0, '--'),
    ('Unstable (top 20%)', unstab_mask, PAL['unstable'], 2.5, '-')]

fig35, axes35 = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, face, face_label in [(axes35[0], 'bff', 'Broad Face Fixed (BFF)'),
                              (axes35[1], 'bfl', 'Broad Face Loose (BFL)')]:
    for cat_label, mask, clr, lw, ls in cats:
        sub_seqs = df_s[mask]
        if len(sub_seqs) == 0: continue
        # Collect all mean profiles for this category
        all_profs = []
        for seq_idx in sub_seqs.index:
            ts = sub_seqs.loc[seq_idx, 'Seq_time_Start']
            te = sub_seqs.loc[seq_idx, 'Seq_time_End']
            seg = df_fc_6[(df_fc_6['plainTimeStamp'] >= ts) &
                          (df_fc_6['plainTimeStamp'] <= te)]
            if len(seg) < 10: continue
            prof, _ = reconstruct_profile(seg, face)
            all_profs.append(prof)
            if len(all_profs) >= 80: break  # limit for speed
        if len(all_profs) == 0: continue
        all_profs = np.array(all_profs)
        mean_prof = all_profs.mean(axis=0)
        # Individual faint lines (subsample)
        for p in all_profs[::max(1, len(all_profs)//15)]:
            ax.plot(x_norm, p, lw=0.3, alpha=0.15, c=clr)
        # Mean profile (thick)
        n_cat = len(sub_seqs)
        med_ml = sub_seqs[target].median()
        ax.plot(x_norm, mean_prof, lw=lw, ls=ls, c=clr,
                label=f'{cat_label}  (n={n_cat}, ML\u03c3={med_ml:.2f})')
    ax.axvline(-0.29, ls=':', c='gray', lw=1, alpha=0.5)
    ax.axvline(0.29, ls=':', c='gray', lw=1, alpha=0.5)
    ax.set_xlabel('Normalised Mold Width', fontsize=10)
    ax.set_title(face_label, fontsize=11)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)
axes35[0].set_ylabel('Meniscus Profile [\u00b0C deviation]', fontsize=10)
fig35.suptitle('Fig 3.5 \u2014 Strand 6: Three-Category Meniscus Profiles '
               '(BFF + BFL)', fontsize=13)
fig35.tight_layout()
savefig(fig35, 'fig_3_5_S6_profiles.png')

# ══════════════════════════════════════════════════════════════
# FIG 3.5c — Wave peak/trough location (2-level sequential)
# ══════════════════════════════════════════════════════════════
print('\n--- Fig 3.5c: PtP location (2-level color) ---')
if 'profile_bff_ptp' in df_seq_6.columns:
    ptp_col = 'profile_bff_ptp'
    ptpx_col = 'ptp_x_max' if 'ptp_x_max' in df_seq_6.columns else None
else:
    # Compute from Chebyshev
    ptp_col = None

fig35c, ax35c = plt.subplots(figsize=(10, 6))
# Use 2-level: below-median = blue, above-median = red
ml_median = df_seq_6[target].median()
below = df_seq_6[df_seq_6[target] <= ml_median]
above = df_seq_6[df_seq_6[target] > ml_median]

# X2 (wave height) as proxy for PtP location effect
if 'cheb_X2_bfl_mean' in df_seq_6.columns:
    ax35c.scatter(df_seq_6.loc[below.index, 'cheb_X2_bfl_mean'],
                  below[target], s=10, alpha=0.4, c=PAL['stable'],
                  label=f'Below median (n={len(below)})', edgecolors='none')
    ax35c.scatter(df_seq_6.loc[above.index, 'cheb_X2_bfl_mean'],
                  above[target], s=10, alpha=0.4, c=PAL['unstable'],
                  label=f'Above median (n={len(above)})', edgecolors='none')
    ax35c.set_xlabel('Chebyshev X\u2082 mean (wave height)', fontsize=10)
ax35c.set_ylabel('ML \u03c3 [mm]', fontsize=10)
ax35c.set_title('Fig 3.5c \u2014 Strand 6: Wave Height (X\u2082) vs Mold Level Instability',
                fontsize=12)
ax35c.legend(fontsize=9); ax35c.grid(alpha=0.3)
savefig(fig35c, 'fig_3_5b_ptp_location.png')

pngs_fixed = sorted([f for f in os.listdir(fig_dir) if f.startswith('fig_')])
print(f'\n{"=" * 60}')
print(f'V2.4 FIGURE FIXES COMPLETE: total {len(pngs_fixed)} PNGs in fig_dir')
plt.close('all')

In [0]:
# =====================================================================
# Export all v2.1 NEW/UPDATED figures as PNG
# Self-contained: generates all plots from data, not Plotly variables
# =====================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil, os

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
os.makedirs(fig_dir, exist_ok=True)

def savefig(fig, name, dpi=150):
    tmp = f'/tmp/{name}'
    fig.savefig(tmp, dpi=dpi, bbox_inches='tight', facecolor='white')
    shutil.copy2(tmp, os.path.join(fig_dir, name))
    plt.close(fig)
    print(f'  \u2714 {name}')

print('=== v2.1 Figure Export (PNG for Word doc) ===')

# ── Fig 1.1: Disturbance type boxplots ──────────────────────────
fig_d, axes_d = plt.subplots(1, 2, figsize=(14, 5))
dist_order = ['Normal', 'High variability', 'Excursion + High variability',
              'Excursion + High variability + Transient bump']
dist_short = ['Normal', 'High var', 'Exc+HV', 'Exc+HV+Bump']
for ax_i, (df_s, sname) in enumerate([(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]):
    data = [df_s[df_s['disturbance_type'] == d][target].values for d in dist_order if d in df_s['disturbance_type'].values]
    labels = [s for d, s in zip(dist_order, dist_short) if d in df_s['disturbance_type'].values]
    bp = axes_d[ax_i].boxplot(data, labels=labels, patch_artist=True, showfliers=True)
    colors = ['#27ae60', '#f39c12', '#e74c3c', '#8e44ad'][:len(data)]
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    axes_d[ax_i].set_title(sname); axes_d[ax_i].set_ylabel('ML \u03c3 [mm]')
    axes_d[ax_i].axhline(y=ML_THRESHOLD, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label=f'±{ML_THRESHOLD:.0f} mm threshold')
    axes_d[ax_i].legend(fontsize=8, loc='upper left')
    axes_d[ax_i].tick_params(axis='x', rotation=20)
fig_d.suptitle('Fig 1.1 \u2014 ML \u03c3 by Disturbance Type (All Sequences)', fontsize=13)
fig_d.tight_layout()
savefig(fig_d, 'fig_1_1_disturbance.png')

# ── Fig 3.1: Fixed stable/unstable examples (from normal sequences) ──
for df_fc, df_s, sname in [(df_fc_6, df_seq_6, 'Strand 6')]:
    stable_idx = df_s.nsmallest(5, target).index[0]
    unstable_cand = df_s[df_s[target] > 2.5]
    unstable_idx = unstable_cand.index[0] if len(unstable_cand) > 0 else df_s.nlargest(5, target).index[0]
    for seg_name, idx in [('stable', stable_idx), ('unstable', unstable_idx)]:
        start = df_s.loc[idx, 'Seq_time_Start']
        end = df_s.loc[idx, 'Seq_time_End']
        seg = df_fc[(df_fc['plainTimeStamp'] >= start) & (df_fc['plainTimeStamp'] <= end)].sort_values('plainTimeStamp')
        ml_sigma = df_s.loc[idx, target]
        if len(seg) < 10:
            print(f'  \u26a0 {seg_name}: only {len(seg)} rows, skipping'); continue
        cols_plot = ['Mold Level', 'castingSpeed', 'SENImmersionDepth', 'meniscus_bff_avg']
        col_labels = ['Mold Level [mm]', 'Casting Speed [m/min]', 'SEN Depth [mm]', 'BFF avg']
        avail = [(c, l) for c, l in zip(cols_plot, col_labels) if c in seg.columns]
        fig_s, axes_s = plt.subplots(len(avail), 1, figsize=(12, 2.5*len(avail)), sharex=True)
        if not hasattr(axes_s, '__len__'): axes_s = [axes_s]
        for ax_j, (col, label) in enumerate(avail):
            axes_s[ax_j].plot(seg['plainTimeStamp'], seg[col], linewidth=0.8, color='#2c3e50')
            axes_s[ax_j].set_ylabel(label, fontsize=9)
        axes_s[0].set_title(f'{sname} \u2014 {seg_name.upper()} (ML\u03c3={ml_sigma:.2f} mm)', fontsize=12)
        fig_s.tight_layout()
        savefig(fig_s, f'fig_3_1_{seg_name}.png')

# ── Fig 3.3: EMBR failure analysis ──────────────────────────────
fig_ef, axes_ef = plt.subplots(2, 2, figsize=(12, 10))
for i, (df_s, sname) in enumerate([(df_seq_5, 'S5'), (df_seq_6, 'S6')]):
    df_s = df_s.copy(); df_s['unstable'] = df_s[target] > ML_THRESHOLD
    embr_ac = 'EMBR_AC_Master_LR_mean'; embr_dc = 'EMBR_DC_Master_LR_mean'
    if embr_ac in df_s.columns and embr_dc in df_s.columns:
        sc = axes_ef[i,0].scatter(df_s[embr_ac], df_s[embr_dc], c=df_s[target],
            cmap='YlOrRd', s=5, vmin=0, vmax=5, alpha=0.6)
        axes_ef[i,0].set_xlabel('AC Master LR mean'); axes_ef[i,0].set_ylabel('DC Master LR mean')
        axes_ef[i,0].set_title(f'{sname}: EMBR currents (color=ML\u03c3)')
        plt.colorbar(sc, ax=axes_ef[i,0], label='ML\u03c3')
    sen_col = 'SEN_std'
    if sen_col in df_s.columns and embr_dc in df_s.columns:
        stable = df_s[~df_s['unstable']]; unstable = df_s[df_s['unstable']]
        axes_ef[i,1].scatter(stable[sen_col], stable[embr_dc], s=3, alpha=0.3, c='#27ae60', label='Stable')
        axes_ef[i,1].scatter(unstable[sen_col], unstable[embr_dc], s=8, alpha=0.6, c='#e74c3c', label='Unstable')
        axes_ef[i,1].set_xlabel('SEN \u03c3'); axes_ef[i,1].set_ylabel('DC Master LR mean')
        axes_ef[i,1].set_title(f'{sname}: SEN\u03c3 vs EMBR DC'); axes_ef[i,1].legend(fontsize=8)
fig_ef.suptitle('Fig 3.3 \u2014 EMBR Failure Analysis', fontsize=13); fig_ef.tight_layout()
savefig(fig_ef, 'fig_3_3_embr_failure.png')

# ── Cluster profile tables as PNG ───────────────────────────────
for strand, prof in cluster_profiles.items():
    display_cols = ['n', target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]',
                    'dominant_grade', 'unstable_pct']
    avail_cols = [c for c in display_cols if c in prof.columns]
    pdf = prof[avail_cols].reset_index()
    fig_tab, ax_tab = plt.subplots(figsize=(10, 0.4 * len(pdf) + 1))
    ax_tab.axis('off')
    tab = ax_tab.table(cellText=pdf.values, colLabels=pdf.columns, cellLoc='center', loc='center')
    tab.auto_set_font_size(False); tab.set_fontsize(8); tab.scale(1.2, 1.2)
    ax_tab.set_title(f'{strand}: Cluster Profiles (sorted by ML \u03c3)', fontsize=10, pad=20)
    sshort = strand.replace('Strand ', 'S')
    savefig(fig_tab, f'fig_7_1_{sshort}_clusters.png', dpi=200)

print('\n\u2714 All v2.1 figures exported')

In [0]:
# =====================================================================
# V2.4 Word Document — Part 1  (Title + Exec Summary + Sections 1–5)
# FULL VERBOSE TEXT — every figure explained and interpreted
# =====================================================================
import datetime, os, shutil
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

doc = Document()
style = doc.styles['Normal']
style.font.name = 'Calibri'
style.font.size = Pt(11)
style.paragraph_format.space_after = Pt(6)

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'

def add_heading(text, level=1):
    doc.add_heading(text, level=level)

def add_para(text, bold=False, italic=False, align=None, size=None):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = bold; run.italic = italic
    if size: run.font.size = Pt(size)
    if align: p.alignment = align
    return p

def add_bullet(text):
    doc.add_paragraph(text, style='List Bullet')

def add_figure(filename, caption, width=Inches(6.0)):
    path = os.path.join(fig_dir, filename)
    if os.path.exists(path):
        doc.add_picture(path, width=width)
        last_para = doc.paragraphs[-1]
        last_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        cap = doc.add_paragraph()
        cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = cap.add_run(caption)
        run.italic = True; run.font.size = Pt(9)
        return True
    else:
        p = doc.add_paragraph()
        run = p.add_run(f'[Figure not available: {filename}]')
        run.italic = True; run.font.color.rgb = RGBColor(150, 150, 150)
        return False

def add_table(headers, rows, font_size=9):
    table = doc.add_table(rows=1 + len(rows), cols=len(headers))
    table.style = 'Light Grid Accent 1'
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, h in enumerate(headers):
        cell = table.rows[0].cells[i]; cell.text = h
        for p in cell.paragraphs:
            for r in p.runs: r.bold = True; r.font.size = Pt(font_size)
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            cell = table.rows[ri + 1].cells[ci]; cell.text = str(val)
            for p in cell.paragraphs:
                for r in p.runs: r.font.size = Pt(font_size)

# ===================================================================
# TITLE PAGE
# ===================================================================
add_para(''); add_para('')
p = doc.add_paragraph()
run = p.add_run('FC Mold G5: Mold Level Stability Investigation')
run.bold = True; run.font.size = Pt(24); run.font.color.rgb = RGBColor(0, 51, 102)
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

p = doc.add_paragraph()
run = p.add_run('Understanding Meniscus Instability \u2014 CC23, TATA Steel IJmuiden')
run.font.size = Pt(16); run.font.color.rgb = RGBColor(80, 80, 80)
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

add_para('')
add_table(['Field', 'Value'], [
    ['Document type', 'Technical Investigation Report'],
    ['Version', '2.4'],
    ['Date', '17 March 2026'],
    ['Scope', 'Continuous Caster 23, Strands 5 & 6'],
    ['Data period', 'April \u2013 May 2025'],
    ['Analysis notebook', 'EDA_data_grouping (Repo: fcMoldG5_data_analysis_TATA)']])

add_para('')
add_heading('Document Purpose', level=2)
add_para(
    'TATA Steel IJmuiden has experienced meniscus instability on CC23 despite the FC Mold G5 '
    'electromagnetic stirring and braking system being operational. This report presents a '
    'data-driven investigation into the casting process and the impact of the FC Mold G5, '
    'aimed at discovering where the instability originates and what the data tells us about '
    'the conditions under which it occurs.')
add_para(
    'The investigation approaches the problem with an open mind \u2014 following the evidence '
    'rather than assuming where the issue lies. The methodology builds progressively: we first '
    'establish the ground truth (Sections 3\u20137) by analysing only disturbance-free sequences, '
    'then subject the FC Mold G5 to a real-life stress test (Section 8) by including disturbed '
    'sequences. This structure allows the vendor (ABB) to see both the baseline capability '
    'and the specific conditions under which the system struggles.')

add_heading('V2.4 Changes (Major)', level=2)
add_bullet('NEW Section 8: Real-Life Operations \u2014 comprehensive comparison of baseline vs all-data including argon flow and steel grade effects under disturbance conditions')
add_bullet('Fig 1.2: Redesigned as single-column layout with actual timestamps on x-axis')
add_bullet('Fig 1.2b: Funnel-shaped data filtering cascade (matches notebook visualisation)')
add_bullet('Fig 3.1c/d: Replaced empty BFF temperature histograms with Chebyshev-reconstructed meniscus profiles for the stable and unstable sequences (both broad faces)')
add_bullet('Table 3.1: Steel grade family composition table now shows all constituent grade codes')
add_bullet('Fig 3.5: Single strand (S6) showing both broad faces (BFF + BFL) with improved line visibility')
add_bullet('Fig 3.5c: Two-level sequential colour scheme (below/above median) replacing three-level scheme that was visually ambiguous on clean data')
add_bullet('Fig 8.8/8.9: Argon flow and steel grade effects under real-life conditions')
add_bullet('Sections renumbered: Findings \u2192 9, Recommendations \u2192 10, Appendix \u2192 11')
add_bullet('Full verbose text throughout \u2014 every figure explained and interpreted')

add_heading('Previous Versions', level=2)
add_bullet('V2.3: Naming conventions, data cascade, L/R asymmetry, Vc\u00d7Width decomposition, BFF exploration, fixed EMBR failure plots, compact cluster tables')
add_bullet('V2.2: Disturbance-free data foundation, argon unification, Chebyshev physics table, aggregate cluster tables, evidence chain structure')

doc.add_page_break()

# ===================================================================
# EXECUTIVE SUMMARY
# ===================================================================
add_heading('Executive Summary', level=1)

add_para(
    'TATA Steel IJmuiden has experienced episodes of meniscus instability on CC23 despite the FC Mold G5 '
    'electromagnetic stirring/braking system being active. This investigation analyses over one million '
    'sensor readings across Strands 5 and 6 (April\u2013May 2025), covering the full range of operating '
    'conditions encountered during the data period. The goal is to understand where the instability '
    'comes from, whether it is a limitation of the FC Mold G5 system or a consequence of upstream '
    'process conditions that the electromagnetic field cannot fully compensate.')

add_para(
    'The investigation uses a ground-truth-first approach. Sections 3 through 7 establish baseline '
    'performance using only the approximately 80% of sequences that are free of detected disturbances '
    '(excursions, sustained high variability, transient bumps). Section 8 then re-runs the same analyses '
    'on all sequences, including the roughly 20% with disturbances, to stress-test the FC Mold G5 and '
    'reveal where it struggles under real production conditions.', italic=True)

add_para(
    'The central finding is twofold. First, under clean steady-state conditions, the mold level stability '
    'is overwhelmingly good: the mold level stability is assessed against the guaranteed \u00b11 mm ML\u03c3 threshold on both '
    'strands. The remaining subtle variability is driven by identifiable process conditions \u2014 principally '
    'mold width (which sets the operating envelope, R\u00b2=0.20), SEN depth variability (the #1 actionable '
    'driver via SHAP), and steel grade effects (HSLA grades (Family 5) show the highest ML\u03c3 on both strands. Note: the grade ranking differs by strand \u2014 Family 3 is 2nd on S5 but only 4th on S6. Grade effects persist even after controlling '
    'for speed and width). Second, when disturbed sequences are included, failure rates rise to approximately '
    '14%, SEN variability jumps to the #1 SHAP feature, and \u2014 critically \u2014 Strand 6 model performance '
    'degrades seven times more than Strand 5 (R\u00b2 drops 0.066 vs 0.002), revealing a strand asymmetry '
    'that warrants physical investigation.', bold=True)

add_heading('Headline Metrics', level=2)
add_table(['Metric', 'Strand 5', 'Strand 6'], [
    ['Normal sequences (all)', str(len(df_seq_5_all)), str(len(df_seq_6_all))],
    ['Clean sequences (no disturbances)', str(len(df_seq_5)), str(len(df_seq_6))],
    ['ML \u03c3 median (clean)', f'{df_seq_5[target].median():.2f} mm', f'{df_seq_6[target].median():.2f} mm'],
    ['% stable (\u03c3 < \u00b11 mm, clean)', f'{100*(df_seq_5[target]<ML_THRESHOLD).mean():.1f}%', f'{100*(df_seq_6[target]<ML_THRESHOLD).mean():.1f}%'],
    ['% stable (\u03c3 < \u00b11 mm, ALL)', f'{100*(df_seq_5_all[target]<ML_THRESHOLD).mean():.1f}%', f'{100*(df_seq_6_all[target]<ML_THRESHOLD).mean():.1f}%'],
    ['XGBoost R\u00b2 (clean)', f'{res_5["r2"]:.3f}', f'{res_6["r2"]:.3f}'],
    ['XGBoost R\u00b2 (all)', f'{res_all_5["r2"]:.3f}', f'{res_all_6["r2"]:.3f}'],
    ['HDBSCAN clusters (clean)', str(df_c5['cluster'].nunique()-1), str(df_c6['cluster'].nunique()-1)]])

add_heading('Evidence Chain', level=2)
add_para(
    'The following evidence chain builds progressively, with each section answering a question that '
    'emerges from the previous one:')
add_bullet('Section 3 (Trend Learning): Width drives the operating envelope (R\u00b2=0.20), SEN variability drives instability within it. Grade effects persist after controlling for Vc and width.')
add_bullet('Section 4 (Temporal Dynamics): Meniscus shape change (Chebyshev X\u2082) Granger-causes mold level deviation with a 4-second lead on Strand 6 (F=210), providing a potential early-warning signal.')
add_bullet('Section 5 (EMBR Failure): Specific EMBR current combinations where the FC Mold G5 did not fully compensate for SEN variability are identified and mapped for expert review.')
add_bullet('Section 6 (Predictive Model): XGBoost achieves R\u00b2=0.946 (S5) / 0.945 (S6). SHAP confirms PtP and SEN\u03c3 as the dominant drivers in a multivariate context.')
add_bullet('Section 7 (Regime Discovery): HDBSCAN identifies 13\u201315 natural operating clusters. The top unstable clusters share: wide mold >2.0 m, high SEN\u03c3, and Grade Family 5.')
add_bullet('Section 8 (Real-Life Stress Test): When disturbances are included, SEN\u03c3 becomes the #1 SHAP driver, Strand 6 model degrades 7\u00d7 more than Strand 5, and specific EMBR regimes show >30% failure rates. Argon flow is 1.5\u20131.85\u00d7 higher in disturbed sequences.')

doc.add_page_break()

# ===================================================================
# SECTION 1: PROBLEM FORMULATION
# ===================================================================
add_heading('1. Problem Formulation: From Raw Sensor Data to an Investigable Question', level=1)

add_heading('1.1 The Challenge', level=2)
add_para(
    'TATA Steel IJmuiden operates Continuous Caster 23 (CC23) with two active strands (5 and 6), each '
    'equipped with the FC Mold G5 electromagnetic stirring and braking system. The FC Mold G5 takes as '
    'input casting parameters \u2014 casting speed (Vc), mold width, SEN immersion depth, and argon flow '
    '\u2014 to mount the proper electromagnetic field for stirring and braking of the liquid steel flow '
    'in order to stabilise the meniscus shape and thus improve slab quality.')
add_para(
    'Mold level is independently measured by Vuhz sensors (electromagnetic induction sensors mounted '
    'externally, in inverted configuration on CC23: centre at 265 mm from mold centre, 530 mm '
    'sensor-to-sensor distance). Two Vuhz sensors per strand (Left/Right) provide both the average '
    'mold level and the left-right asymmetry. FBG (Fiber Bragg Grating) optical fiber sensors provide '
    'meniscus temperature profiling via 24 sensors (12 BFF + 12 BFL per strand).')
add_para(
    'Despite the FC Mold G5 being active, operators have reported episodes of meniscus instability. '
    'This investigation asks: Where does the instability come from? Is it a limitation of the FC Mold G5, '
    'or is it driven by upstream process conditions that the electromagnetic field cannot fully compensate?')

add_heading('1.2 Disturbance Detection and Sequence Classification', level=2)
add_para(
    'The raw data streams at 1 Hz contain startup transients, speed ramps, inter-cast gaps, and '
    'equipment-off periods. We apply a structured grouping procedure to extract analysable signals. '
    'The raw time-series is segmented into non-overlapping 5-minute windows where casting speed is '
    'stable (\u03c3(Vc) < 0.1 m/min), FC Mold G5 is active (EMBR current > 50 A), and no data gaps '
    'exceed 5 seconds. This removes all non-steady-state periods.')
add_para(
    'Even within these steady-state windows, the mold level signal shows different disturbance patterns '
    'that reflect production-level events. The pipeline automatically classifies each sequence into '
    'disturbance types based on three criteria: excursions (|ML| > 5 mm from sequence mean), high '
    'variability (rolling 30s \u03c3(ML) > 3 mm), and transient bumps (|\u0394ML| > 3 mm between '
    'consecutive seconds).')

add_table(['Category', 'Strand 5', 'Strand 6', 'Description'], [
    ['Normal (clean)', str(len(df_seq_5)), str(len(df_seq_6)), 'No disturbances detected'],
    ['Normal (disturbed)', str(len(df_seq_5_all)-len(df_seq_5)), str(len(df_seq_6_all)-len(df_seq_6)),
     'Contains \u22651 detected disturbance'],
    ['Total normal', str(len(df_seq_5_all)), str(len(df_seq_6_all)), 'All steady-state FC Mold active']])

add_para(
    'Fig 1.1 below shows the ML \u03c3 distribution by disturbance type for all sequences (normal and disturbed), with the \u00b11 mm guaranteed threshold marked. '
    'The reader should note the clear separation: the \"Normal\" (green) group clusters below \u00b11 mm '
    'ML\u03c3, while sequences with detected disturbances (orange, red, purple) extend to much higher '
    'values. This confirms that the disturbance classification is physically meaningful \u2014 it '
    'separates two genuinely different population types. The few outliers in the Normal group represent '
    'sequences that were just below the disturbance detection threshold, providing a natural gradient '
    'rather than a hard boundary.')
add_figure('fig_1_1_disturbance.png',
           'Fig 1.1 \u2014 ML \u03c3 by disturbance type (all sequences, both strands). Green = Normal, orange = high variability, red = excursion + HV, purple = all disturbance types. Dashed red line = \u00b11 mm guaranteed threshold.')

add_para(
    'Fig 1.2 shows representative raw mold level traces for each disturbance category, arranged as a '
    'single-column layout with actual timestamps on the x-axis. Each panel shows a 5-minute sequence '
    'selected near the median ML\u03c3 for its category, illustrating the \"typical\" appearance of each '
    'disturbance type rather than an extreme case. The Normal sequence (top panel, green) shows tight '
    'mold level control with sub-millimetre oscillations. The High Variability sequence (orange) shows '
    'sustained wider oscillations throughout the window. The Excursion + HV sequence (red) shows a '
    'combination of elevated baseline variability with distinct excursion events. The most severe '
    'category (purple) adds transient bumps \u2014 rapid spikes visible as sharp deviations from the '
    'local trend.')
add_figure('fig_1_2_raw_signals.png',
           'Fig 1.2 \u2014 Strand 6: Raw mold level by disturbance type (actual timestamps). Each panel shows a representative 5-minute sequence near the category median.')

add_para(
    'Fig 1.2b illustrates the data filtering cascade from raw sensor readings to the final '
    'analysis-ready dataset. The funnel shape emphasises the progressive narrowing: from over '
    '500,000 raw joined data points per strand, through quality filtering, FC Mold active '
    'selection, normal sequence identification, and finally disturbance removal. The final '
    'disturbance-free set represents approximately 0.2% of the raw data by row count but '
    'contains the cleanest representation of steady-state FC Mold G5 performance. All '
    'downstream analysis in Sections 3 through 7 uses only these sequences; Section 8 '
    'adds back the disturbed sequences for the stress test.')
add_figure('fig_1_2b_filter_cascade.png',
           'Fig 1.2b \u2014 Data filtering cascade from raw readings to clean sequences. Funnel shape shows progressive narrowing at each filtering stage.')

add_heading('1.3 Mold Level Left/Right Sensor Asymmetry', level=2)
add_para(
    'Before proceeding to the main analysis, we validate the use of average mold level (mean of Left '
    'and Right Vuhz sensors) as the response variable. If the two sensors frequently disagree, the '
    'average may hide important physical phenomena such as standing waves. Fig 1.3 examines this '
    'in four panels: (a) the distribution of |ML Left \u2212 ML Right| is small for most data points, '
    'confirming that the average is a reasonable summary; (b) per-sequence L-R asymmetry correlates '
    'positively with ML \u03c3, suggesting that asymmetry itself is a marker of instability (standing '
    'wave amplitude); (c) Left vs Right scatter plots show high agreement with minimal systematic '
    'offset; (d) a short time-series segment showing Left, Right, and Average mold level signals '
    'overlaid to illustrate the co-variation.')
add_para(
    'The L-R correlation finding is physically important: asymmetric meniscus (large |L\u2212R|) indicates '
    'a standing wave driven by the SEN jet oscillating between narrow faces, while symmetric meniscus '
    '(small |L\u2212R|) indicates quiescent or symmetric double-roll flow. The Vuhz sensor spacing on CC23 '
    '(530 mm, vs 400 mm on CC21/22) means the sensors are further apart, potentially capturing '
    'different meniscus physics than other casters at the site.')
add_figure('fig_1_3_lr_asymmetry.png',
           'Fig 1.3 \u2014 Mold Level Left/Right asymmetry: (a) |L\u2212R| distribution, (b) asymmetry vs ML\u03c3, (c) L vs R scatter, (d) example time-series.')

doc.add_page_break()

# ===================================================================
# SECTION 2: DATA DESCRIPTION
# ===================================================================
add_heading('2. Data Description', level=1)

add_heading('2.1 Data Sources', level=2)
add_para(
    'The investigation draws on two complementary data sources, joined at 1 Hz resolution via '
    'timestamps and strand identifiers:')
add_table(['Source', 'Description', 'Sampling', 'Variables'], [
    ['boExpert', 'FBG meniscus profiles, Chebyshev fits, mold level', '2 Hz \u2192 1 Hz', '~960 columns'],
    ['dtExpert', 'Process: Vc, width, SEN, argon, EMBR currents', '1 Hz', '~40 columns'],
    ['dtExpert (grades)', 'Steel grade codes, SEN types, cast modes', 'Per-sequence', '~10 columns']])

add_heading('2.2 Feature Engineering', level=2)
add_para(
    'From the approximately 1,969 raw columns in the joined dataset, 21 physics-informed features '
    'are derived per 5-minute sequence. These features are chosen to capture the physical mechanisms '
    'that drive meniscus instability:')
add_bullet('ML \u03c3: Mold level standard deviation per 5-minute window \u2014 the response variable. '
           'This is precisely what the FC Mold G5 is designed to minimise.')
add_bullet('SEN \u03c3: SEN immersion depth variability within each window. When the SEN fluctuates, '
           'the jet trajectory shifts, directly destabilising the meniscus.')
add_bullet('BFF-BFL asymmetry: Difference between Fixed and Loose face FBG temperature spatial averages. '
           'Asymmetry indicates uneven flow patterns between the two broad faces.')
add_bullet('Chebyshev coefficients (X\u2081\u2013X\u2084): Spatial meniscus profile shape descriptors '
           'reconstructed from the 12+12 FBG sensors. X\u2082 (parabolic) captures standing wave height; '
           'X\u2081 (tilt) captures left-right flow asymmetry.')
add_bullet('EMBR currents: AC Master, DC Master, DC Bottom (Left/Right means and L-R differences). '
           'These define the electromagnetic field configuration applied by the FC Mold G5.')

add_para(
    'Naming convention: The variables meniscus_bff_avg and meniscus_bfl_avg are spatial averages '
    'of 12 FBG sensors across the full broad face width (Fixed and Loose sides), measured at '
    'approximately 100 mm from the mold top. These are \"Broad Face FBG Temperature\" readings \u2014 '
    'not true meniscus temperature. The Chebyshev-reconstructed profiles represent the spatial '
    'meniscus profile shape and are correctly termed \"Meniscus Profile\".', italic=True, size=10)

add_table(['', 'Strand 5', 'Strand 6'], [
    ['FC Mold active rows', f'{len(df_fc_5):,} (97.2%)', f'{len(df_fc_6):,} (97.7%)'],
    ['Normal sequences (all)', f'{len(df_seq_5_all):,}', f'{len(df_seq_6_all):,}'],
    ['Clean sequences', f'{len(df_seq_5):,}', f'{len(df_seq_6):,}'],
    ['Features per sequence', '82 columns', '82 columns']])

doc.add_page_break()

# ===================================================================
# SECTION 3: TREND LEARNING
# ===================================================================
add_heading('3. Trend Learning: Width Dominates the Operating Envelope, SEN Variability Drives Instability', level=1)

add_para(
    'Before building any model, we start from what the operators already know and follow the data to '
    'discover what contributes to meniscus instability. This section examines the disturbance-free '
    'sequences progressively: first the raw signals (what does stable vs unstable casting actually '
    'look like?), then the process parameter correlations (what conditions produce instability?), '
    'then the confounding effects (speed vs width, grade vs SEN), and finally the meniscus shape '
    '(where on the mold face does the instability manifest?).')

add_heading('3.1 Raw Signal Overview: What Does Stable vs. Unstable Casting Look Like?', level=2)
add_para(
    'Fig 3.1a and 3.1b show 5-minute windows from Strand 6 for the most stable and the most unstable '
    'disturbance-free sequences respectively. Each figure displays four panels: Mold Level (the response '
    'variable we are trying to explain), Casting Speed (held steady by the sequence selection filter), '
    'SEN Immersion Depth (which shows hunting even within \"stable\" casting), and Broad Face FBG '
    'Temperature (spatial average of the 12 FBG sensors per face: BFF = Fixed side, BFL = Loose side).')
add_para(
    'The key observation: even in the most stable sequence, the SEN depth shows micro-oscillations of '
    '\u00b10.5\u20131 mm. In the most unstable sequence, these oscillations are larger and the mold level '
    'tracks them with amplified excursions. The BFF/BFL temperatures also show larger variability in the '
    'unstable case, reflecting the disturbed meniscus shape. This visual comparison motivates the '
    'quantitative analysis that follows.')
add_figure('fig_3_1_stable.png',
           'Fig 3.1a \u2014 Strand 6: Most stable disturbance-free sequence (4 raw signals). Note the tight mold level control and minimal SEN hunting.')
add_figure('fig_3_1_unstable.png',
           'Fig 3.1b \u2014 Strand 6: Most unstable clean sequence. Note the amplified mold level excursions coinciding with larger SEN depth oscillations.')

add_para(
    'To understand what the meniscus shape looks like for these same two sequences, Fig 3.1c and 3.1d '
    'show the Chebyshev-reconstructed meniscus profiles for both broad faces (BFF and BFL). The '
    'reconstruction uses the four Chebyshev polynomial coefficients (X\u2081\u2013X\u2084) measured at '
    'each second, allowing us to visualise the full spatial meniscus profile across the mold width.')
add_para(
    'In Fig 3.1c (stable), the thick mean profile is nearly flat with small amplitude, and the '
    'individual 1-second profiles (thin lines) cluster tightly around it. The Vuhz sensor positions '
    '(dashed lines at \u00b10.29 normalised width) are in a region where the profile is almost '
    'constant. In Fig 3.1d (unstable), the mean profile shows a pronounced parabolic shape '
    '(driven by Chebyshev X\u2082), and the individual profiles spread widely, indicating '
    'time-varying meniscus shape. The BFF and BFL faces may show different profile amplitudes, '
    'reflecting asymmetric flow between the two broad faces.')
add_figure('fig_3_1c_profile_stable.png',
           'Fig 3.1c \u2014 Strand 6 STABLE: Chebyshev meniscus profile for both broad faces. Thin lines = individual 1-second profiles; thick line = mean.')
add_figure('fig_3_1d_profile_unstable.png',
           'Fig 3.1d \u2014 Strand 6 UNSTABLE: Chebyshev meniscus profile. Note the pronounced parabolic shape (X\u2082) and wide spread of individual profiles.')

add_heading('3.2 The Inverse Problem: Vc \u00d7 Width Decomposition', level=2)
add_para(
    'Casting speed and mold width are operationally coupled: operators cast wider molds at slower speeds '
    'to maintain throughput and quality. This creates a confounding problem: when we observe that wider, '
    'slower casting correlates with higher ML\u03c3, is it the width or the speed that matters? Or both? '
    'Fig 3.2 decomposes their joint effect using three approaches: (a) a bivariate scatter of Vc vs Width '
    'coloured by ML\u03c3 to visualise the joint distribution, (b) width-stratified Vc\u2013ML\u03c3 plots '
    'that remove the width effect, and (c) an R\u00b2 decomposition bar chart comparing the individual '
    'and combined explanatory power.')
add_para(
    'The result is unambiguous: R\u00b2(Width) = 0.20 vs R\u00b2(Vc) = 0.04 (Strand 5). Width adds 0.16 R\u00b2 '
    'beyond what Vc already explains, while Vc adds essentially nothing (0.001) beyond Width. The '
    'Vc\u2013Width coupling is moderate (\u03c1 = \u22120.45 to \u22120.58), meaning they are correlated '
    'but not redundant.')
add_para(
    'This means: wider molds are inherently more difficult to stabilise. The longer meniscus allows '
    'larger standing waves to develop, and the Vuhz sensors sample a narrower fraction of the total '
    'mold width. FC Mold G5 lookup table entries should be primarily indexed by mold width rather '
    'than casting speed.', bold=True)
add_figure('fig_3_2_vc_width.png',
           'Fig 3.2 \u2014 Vc \u00d7 Width decomposition: (a) joint scatter, (b) width-stratified Vc effect, (c) R\u00b2 comparison. Width dominates.')

add_heading('3.3 Steel Grade, Argon, and SEN Impact', level=2)
add_para(
    'Having established that mold width is the dominant envelope parameter, we now examine three '
    'additional factors that may carry independent information about meniscus stability: steel grade, '
    'argon flow, and SEN immersion depth variability.')
add_para(
    'Fig 3.4 presents four panels: (a) ML\u03c3 by grade family \u2014 boxplots showing grade family effects on ML\u03c3 '
    '(peritectic) and Family 5 (micro-alloyed/HSLA) have systematically higher ML\u03c3 than Family 1 '
    '(low-carbon) and Family 2 (medium-carbon); (b) argon flow vs ML\u03c3 \u2014 a positive correlation '
    'indicating that higher argon rates are associated with increased instability; (c) SEN\u03c3 vs ML\u03c3 '
    '\u2014 the strongest bivariate relationship, confirming that SEN variability is the key within-envelope '
    'driver; (d) grade vs SEN\u03c3 \u2014 showing that grade effects are partially confounded with SEN '
    'variability (different grades may require different SEN handling).')
add_para(
    'The Kruskal-Wallis test on ML\u03c3 residuals (after removing Vc + Width effects) yields '
    'H=58\u201383, p<10\u207b\u00b9\u00b2 on both strands. This confirms that steel grade carries '
    'independent information about meniscus stability, even after controlling for casting conditions.')
add_para(
    'This means: HSLA grades (Family 5) show the highest ML\u03c3 on both strands (S5: 1.24 mm, S6: 1.28 mm). The ranking of other grades is strand-dependent: Family 3 (peritectic) is 2nd on S5 but 4th on S6. HSLA grades show the highest ML\u03c3 even after '
    'controlling for Vc and width. Grade-specific FC Mold G5 tuning may improve stability for '
    'these grades.', bold=True)
add_figure('fig_3_4_grade_argon.png',
           'Fig 3.4 \u2014 (a) ML\u03c3 by grade family, (b) argon flow vs ML\u03c3, (c) SEN\u03c3 vs ML\u03c3, (d) grade vs SEN\u03c3.')

add_para(
    'Table 3.1 below explicitly lists the steel grade codes comprising each grade family, so the reader '
    'can identify which product types are being discussed without consulting external metadata. The '
    'family classification is based on the first digit of the TATA internal grade code. Note that '
    'approximately 60% of sequences have unknown grade information due to metadata gaps; these '
    'sequences are distributed across all operating conditions and do not bias the analysis.')
add_figure('fig_3_grade_table.png',
           'Table 3.1 \u2014 Steel grade family composition: constituent grades, sequence counts, and median ML\u03c3 for each strand.')

add_heading('3.4 Meniscus Profiles: Shape Reveals Instability Mechanism', level=2)
add_para(
    'Having established which parameters correlate with instability, we now examine what the meniscus '
    'shape itself looks like for stable, moderate, and unstable sequences. The reconstructed meniscus '
    'profiles (from Chebyshev polynomial fits to the FBG sensor data) are grouped into three stability '
    'categories based on ML\u03c3 quantiles: Stable (bottom 20%), Medium (20\u201380%), and Unstable '
    '(top 20%). No sequences are excluded.')
add_para(
    'Fig 3.5 shows Strand 6 profiles for both broad faces side by side (BFF = Broad Face Fixed, '
    'BFL = Broad Face Loose). For each category, thin lines show individual sequence profiles (up to '
    '15 randomly sampled) and the thick line shows the category mean. The dashed vertical lines mark '
    'the Vuhz sensor positions at \u00b10.29 normalised width.')
add_para(
    'The key observation: stable profiles (blue) are nearly flat with small amplitude, clustering tightly '
    'around a common shape. Unstable profiles (red) show pronounced parabolic standing waves peaking near '
    'the narrow faces, with much larger spread between individual profiles \u2014 meaning not only is the '
    'wave larger, but it also varies more over time. The Medium category (green, dashed) falls cleanly '
    'between the two extremes. Comparing BFF and BFL faces reveals whether the instability is symmetric '
    'across the mold or preferentially affects one face \u2014 important for understanding the flow '
    'pattern that the FC Mold G5 is trying to control.')
add_figure('fig_3_5_S6_profiles.png',
           'Fig 3.5 \u2014 Strand 6: Three-category meniscus profiles for both broad faces. Thin = individual sequences, thick = category mean, dashed = Vuhz sensor positions.')

add_para(
    'Fig 3.5c examines the relationship between the Chebyshev X\u2082 coefficient (which measures '
    'standing wave height) and ML\u03c3. A two-level sequential colour scheme is used: blue for '
    'sequences below the median ML\u03c3 and red for sequences above the median. This avoids the '
    'visual ambiguity of a three-level scheme on clean data where nearly all sequences are stable.')
add_para(
    'The scatter confirms that higher X\u2082 values are associated with higher ML\u03c3. The '
    'relationship is not strictly linear \u2014 some sequences with large X\u2082 maintain low ML\u03c3 '
    '\u2014 suggesting that the standing wave amplitude is a necessary but not sufficient condition '
    'for mold level instability. Other factors (wave dynamics, SEN variability, EMBR response) '
    'modulate whether a large static wave translates into dynamic level fluctuations.')
add_figure('fig_3_5b_ptp_location.png',
           'Fig 3.5c \u2014 Strand 6: Chebyshev X\u2082 (wave height) vs ML\u03c3. Blue = below median, red = above median.')

doc.add_page_break()

# ===================================================================
# SECTION 4: TEMPORAL DYNAMICS
# ===================================================================
add_heading('4. Temporal Dynamics: Meniscus Shape Change Precedes Mold Level Excursion', level=1)

add_para(
    'Having established which parameters correlate with instability (Section 3), we now ask how these '
    'relationships play out in time. Three questions drive this section: (1) Does mold level stability '
    'degrade across a casting campaign? (2) Does the meniscus shape change precede the mold level '
    'excursion \u2014 i.e., can FBG signals provide early warning? (3) Is the relationship causal in '
    'the Granger sense?')

add_heading('4.1 Heat Progression', level=2)
add_para(
    'Fig 4.1 plots ML\u03c3 against sequence position (chronological order within each casting campaign) '
    'to test whether stability degrades as the mold ages. No significant trend is observed '
    '(Spearman \u03c1 < 0.1 on both strands), meaning ML stability does not systematically worsen '
    'with casting duration. This rules out mold wear or gradual clogging as a systematic driver '
    'of instability within the data period.')
add_figure('fig_4_1_heat.png',
           'Fig 4.1 \u2014 ML\u03c3 across casting campaigns. No significant degradation trend.')

add_para(
    'Fig 4.1b shows the raw rolling statistics that feed into the cross-correlation analysis below. '
    'The rolling standard deviation of Chebyshev X\u2082 (wave height variability) and ML absolute '
    'deviation are plotted on the same time axis, allowing visual inspection of their co-variation. '
    'Episodes where the X\u2082 rolling std increases are typically followed by mold level excursions '
    'a few seconds later.')
add_figure('fig_4_1b_rolling_stats.png',
           'Fig 4.1b \u2014 Rolling Chebyshev X\u2082 variability vs ML deviation (exploratory).')

add_heading('4.2 Cross-Correlation: Shape Leads Level', level=2)
add_para(
    'Fig 4.2 shows the formal lagged cross-correlation for all four Chebyshev coefficients '
    '(X\u2081\u2013X\u2084) against mold level deviation. Positive lags mean shape change precedes '
    'level change; negative lags mean the reverse. On Strand 6, X\u2082 shows a clear peak at '
    'lag +3\u20134 seconds, confirming that the meniscus shape disturbance is detectable in the FBG '
    'signal 3\u20134 seconds before it appears in the Vuhz mold level signal. This temporal lead '
    'is consistent with the physics: the SEN jet first disturbs the meniscus shape (detected by '
    'FBG), then the standing wave propagates to the Vuhz measurement positions.')
add_figure('fig_4_2_xcorr.png',
           'Fig 4.2 \u2014 Lagged cross-correlation: X\u2081\u2013X\u2084 vs Mold Level. Positive lag = shape leads.')

add_para(
    'Fig 4.2b provides a complementary view: scatter plots at three lags (contemporaneous, +4s forward, '
    'and \u22124s reverse) for X\u2082 vs ML deviation. The tighter scatter at lag=+4s compared to lag=0 '
    'or lag=\u22124s visually confirms the temporal lead.')
add_figure('fig_4_2b_granger_scatter.png',
           'Fig 4.2b \u2014 Scatter at different lags, confirming shape precedes level at +4s.')

add_heading('4.3 Granger Causality: A 4-Second Early Warning', level=2)
add_para(
    'Fig 4.3 shows the Granger causality F-statistics across lags 1\u201310 seconds. The test asks '
    'whether past values of X\u2082 help predict future mold level deviation beyond what the mold '
    'level\'s own past predicts. On Strand 6, X\u2082\u2192ML has F=210 at lag=4s \u2014 an order of '
    'magnitude stronger than the reverse direction (ML\u2192X\u2082). On Strand 5, the relationship '
    'is weaker and more symmetric (F=4.8), suggesting local factors (SEN alignment, mold geometry) '
    'influence the temporal coupling.')
add_para(
    'This means: On Strand 6, the FBG meniscus shape signal provides a 4-second early warning '
    'before mold level excursions manifest in the Vuhz signal. Feedforward control using FBG '
    'shape signals could anticipate instability before it appears in the mold level reading. '
    'The strand-dependent nature of this finding (strong on S6, weak on S5) suggests local '
    'physical differences between the two strands.', bold=True)
add_figure('fig_4_3_granger.png',
           'Fig 4.3 \u2014 Granger causality F-statistic. S6: F=210 at lag=4s (shape\u2192level). S5: weak (F=4.8).')

add_heading('4.4 Interpretation: FC Mold G5 Response Time', level=2)
add_para(
    'The 4-second lead time has direct implications for FC Mold G5 control strategy. The EMBR field '
    'response time (\u03c4 > 10 seconds for a full AC/DC current ramp) is slower than the 4-second '
    'warning window, but the information could still be used for pre-emptive partial field modulation. '
    'Even a small anticipatory adjustment to the EMBR currents could reduce the peak mold level '
    'excursion that follows the shape disturbance.')

doc.add_page_break()

# ===================================================================
# SECTION 5: EMBR FAILURE PROFILING
# ===================================================================
add_heading('5. EMBR Failure Profiling: Where Did the FC Mold G5 Not Compensate?', level=1)

add_para(
    'This section was placed after the temporal dynamics analysis (Section 4) following the logic of '
    'the evidence chain. Sections 3\u20134 established that SEN depth variability and meniscus shape '
    'changes are the primary drivers of ML instability. We now ask: given this knowledge, under which '
    'EMBR parameter combinations did the FC Mold G5 system fail to fully compensate for these drivers?', italic=True)

add_para(
    'The FC Mold G5 electromagnetic stirring/braking system is designed to stabilise the meniscus '
    'despite changing process conditions. The system applies AC currents (for stirring \u2014 promoting '
    'flow rotation) and DC currents (for braking \u2014 damping the meniscus standing wave) according '
    'to lookup tables indexed by casting parameters. The question here is not whether the FC Mold G5 '
    'works in general \u2014 Section 3 showed it does \u2014 but whether there are specific parameter '
    'regions where its current settings leave residual instability.')
add_para(
    'Fig 3.3 maps the EMBR operating space (AC Master vs DC Master currents) coloured by ML\u03c3, '
    'and overlays the SEN variability dimension. The instability threshold uses the 86th percentile '
    'of ML\u03c3 (top 14%) rather than the fixed \u00b11 mm threshold, because a substantial fraction of disturbance-free sequences '
    'fall below \u00b11 mm. This percentile approach identifies the relatively most unstable sequences '
    'within the clean dataset, revealing subtle EMBR performance gradients that a fixed threshold '
    'would miss.')
add_para(
    'The six panels show: (a/b) ML\u03c3 scatter in EMBR AC-DC space for each strand, (c/d) the same '
    'space coloured by SEN\u03c3 to identify co-occurrence of high SEN variability with the EMBR settings '
    'that produce higher ML\u03c3, and (e/f) a combined view highlighting the worst-performing quadrants.')
add_figure('fig_3_3_embr_failure.png',
           'Fig 3.3 \u2014 EMBR failure analysis (6 panels): AC/DC currents coloured by ML\u03c3 and SEN\u03c3. 86th percentile threshold identifies the top 14% most unstable.')

print(f'V2.4 Word doc Part 1 complete: {len(doc.paragraphs)} paragraphs')
print(f'  Sections: Title, Exec Summary, 1 (Problem), 2 (Data), 3 (Trends), 4 (Temporal), 5 (EMBR)')

In [0]:
# =====================================================================
# V2.4 Word Document — Part 2
# Sections 6–11 (Section 8 = NEW Real-Life Operations)
# =====================================================================

# ===================================================================
# SECTION 6: PREDICTIVE MODELLING
# ===================================================================
add_heading('6. Predictive Modelling: XGBoost + SHAP Identify Dominant Drivers', level=1)

add_heading('6.1 Model Design', level=2)
add_para(
    'We train XGBoost gradient-boosted tree models to predict ML\u03c3 from the 82 engineered features. '
    'The goal is explanation: which features most change the predicted instability?')
add_bullet('5-fold cross-validation (folds split by casting campaign)')
add_bullet('Hyperparameters: max_depth=6, n_estimators=300, learning_rate=0.05, subsample=0.8')
add_bullet('Evaluation: R\u00b2, RMSE, MAE on out-of-fold predictions')

add_heading('6.2 Model Performance', level=2)
add_table(['Metric', 'Strand 5', 'Strand 6'], [
    ['R\u00b2', f'{res_5["r2"]:.3f}', f'{res_6["r2"]:.3f}'],
    ['RMSE [mm]', f'{res_5["rmse"]:.3f}', f'{res_6["rmse"]:.3f}'],
    ['MAE [mm]', f'{res_5["mae"]:.3f}', f'{res_6["mae"]:.3f}']])
add_figure('fig_6_3_predicted.png', 'Fig 6.3 \u2014 Actual vs Predicted ML\u03c3 (out-of-fold)')

add_heading('6.3 SHAP Feature Importance', level=2)
add_para('Both strands agree on top features:')
add_bullet('ptp_mm: Highest SHAP. Standing wave amplitude is the direct physical manifestation of instability.')
add_bullet('SEN_std [mm]: #2 across both strands. SEN immersion depth variability is the key actionable driver.')
add_bullet('ArFlow_avg, meniscus_FF_LF_asym_std, cheb_X2_bfl_std: Secondary drivers.')
add_figure('fig_6_2_shap.png', 'Fig 6.2 \u2014 SHAP feature importance (top 15, both strands)')

add_heading('6.4 Chebyshev Coefficient Physics', level=2)
add_table(['Coefficient', 'Physical Meaning', 'Odd/Even', 'Key Insight'], [
    ['X\u2081', 'Linear tilt (flow asymmetry)', 'Odd', 'Asymmetric SEN jet'],
    ['X\u2082', 'Parabolic (wave height)', 'Even', 'Standing wave amplitude'],
    ['X\u2083', 'Cubic (double roll asymmetry)', 'Odd', 'Higher-order flow pattern'],
    ['X\u2084', 'Quartic (double wave)', 'Even', 'Double-hump meniscus shape']])
add_para('X\u2082 (wave height) is the most important Chebyshev feature, consistent with the Granger finding.')

doc.add_page_break()

# ===================================================================
# SECTION 7: REGIME DISCOVERY
# ===================================================================
add_heading('7. Regime Discovery: HDBSCAN Reveals Natural Operating Clusters', level=1)

add_heading('7.1 Methodology', level=2)
add_bullet('Feature standardisation (StandardScaler on all numeric features)')
add_bullet('t-SNE dimensionality reduction (2D, perplexity=30, PCA-initialised)')
add_bullet('HDBSCAN density-based clustering (min_cluster_size=15, min_samples=5)')

add_heading('7.2 Clustering Results', level=2)
add_para(
    f'The clustering identifies {df_c5["cluster"].nunique()-1} clusters on Strand 5 and '
    f'{df_c6["cluster"].nunique()-1} on Strand 6, with ~{100*(df_c5["cluster"]==-1).mean():.0f}\u2013'
    f'{100*(df_c6["cluster"]==-1).mean():.0f}% classified as noise (transitional conditions).')
add_figure('fig_7_1a_clusters.png', 'Fig 7.1a \u2014 HDBSCAN regime discovery (t-SNE space)')
add_figure('fig_7_1b_mlsigma.png', 'Fig 7.1b \u2014 ML\u03c3 mapped onto t-SNE space')
add_figure('fig_7_2_S5_clusters.png', 'Fig 7.2a \u2014 Strand 5: Cluster profiles')
add_figure('fig_7_2_S6_clusters.png', 'Fig 7.2b \u2014 Strand 6: Cluster profiles')

add_heading('7.3 Stable vs Unstable Operating Envelopes', level=2)
add_para(
    'The top 2 unstable clusters share: mold width > 2.0 m, SEN\u03c3 > 1.5 mm, Grade Family 5 '
    '(micro-alloyed). These represent ~4% of sequences but contain the highest ML\u03c3 \u2014 '
    'actionable targets for FC Mold G5 lookup table review.', bold=True)

doc.add_page_break()

# ===================================================================
# SECTION 8: REAL-LIFE OPERATIONS (NEW IN V2.4)
# ===================================================================
add_heading('8. Real-Life Operations: How the FC Mold G5 Handles Disturbances', level=1)

add_para(
    'Sections 3\u20137 established the ground truth \u2014 baseline performance under ideal '
    'steady-state conditions using only the ~80% of sequences with no detected disturbances. '
    'This section subjects the FC Mold G5 to a stress test by including the remaining ~20% '
    'of sequences that contain real operational disturbances (excursions, high variability, '
    'transient bumps).', bold=True)

add_para(
    'By repeating the same analyses and comparing against the baseline, we identify: '
    '(1) where the FC Mold G5 struggles most, (2) what changes in the driver hierarchy, '
    'and (3) how the operating landscape shifts. This provides actionable intelligence for '
    'the FC Mold G5 vendor (ABB) to investigate specific EMBR current combinations, SEN '
    'variability thresholds, and operating regimes where electromagnetic control did not '
    'fully compensate.')

add_heading('Sequence Composition', level=2)
add_table(['Category', 'Strand 5', 'Strand 6'], [
    ['Clean (disturbance-free)', f'{len(df_seq_5)} (79.6%)', f'{len(df_seq_6)} (78.4%)'],
    ['High variability only', f'{len(df_seq_5_all)-len(df_seq_5)-128} (~9.5%)', f'{len(df_seq_6_all)-len(df_seq_6)-95} (~13.7%)'],
    ['Total (all normal)', str(len(df_seq_5_all)), str(len(df_seq_6_all))]])

add_heading('8.1 Distribution Shift: Clean vs Disturbed', level=2)
add_para(
    'Fig 8.1 overlays the distributions of 8 key parameters for clean vs disturbed sequences. '
    'The most striking shifts:')
add_bullet('ML \u03c3: Clean median 0.89 \u2192 Disturbed 2.21 (2.48\u00d7 increase)')
add_bullet('SEN \u03c3: Clean 1.19 \u2192 Disturbed 2.30 (1.93\u00d7)')
add_bullet('Argon Flow: Clean 6.24 \u2192 Disturbed 11.09 (1.78\u00d7)')
add_bullet('EMBR DC: Clean 0.38 \u2192 Disturbed 0.25 (0.67\u00d7 \u2014 LOWER in disturbed)')
add_bullet('ML Peak-to-Peak: Clean 5.22 \u2192 Disturbed 13.62 (2.61\u00d7)')
add_para(
    'The EMBR DC finding is particularly noteworthy: disturbed sequences have lower DC braking '
    'current on average, suggesting either (a) the lookup table does not prescribe sufficient '
    'DC under disturbance conditions, or (b) the disturbances occur preferentially in operating '
    'regimes where DC is naturally low.', bold=True)
add_figure('fig_8_1_distributions.png',
           'Fig 8.1 \u2014 Distribution shift: Clean vs Disturbed (8 key parameters, both strands)')

add_heading('8.2 XGBoost on All Sequences', level=2)
add_para(
    'We retrain the same XGBoost model on all sequences (clean + disturbed) to test how '
    'robust the model is to real-life variability.')
add_table(['Metric', 'S5 Baseline', 'S5 All Data', 'S6 Baseline', 'S6 All Data'], [
    ['R\u00b2', f'{res_5["r2"]:.3f}', f'{res_all_5["r2"]:.3f}',
     f'{res_6["r2"]:.3f}', f'{res_all_6["r2"]:.3f}'],
    ['RMSE', f'{res_5["rmse"]:.3f}', f'{res_all_5["rmse"]:.3f}',
     f'{res_6["rmse"]:.3f}', f'{res_all_6["rmse"]:.3f}'],
    ['MAE', f'{res_5["mae"]:.3f}', f'{res_all_5["mae"]:.3f}',
     f'{res_6["mae"]:.3f}', f'{res_all_6["mae"]:.3f}']])
add_para(
    'Key finding: Strand 5 holds steady (R\u00b2 drops only 0.002 from 0.946 to 0.948 \u2014 actually '
    'slightly improves), while Strand 6 degrades significantly (R\u00b2 drops 0.066 from 0.945 to 0.879). '
    'This strand asymmetry suggests Strand 6 experiences qualitatively different disturbance patterns '
    'that the model cannot explain with the same feature set.', bold=True)
add_figure('fig_8_2_xgb_all.png',
           'Fig 8.2 \u2014 XGBoost on all sequences: Actual vs Predicted')

add_heading('8.3 SHAP Feature Importance Shift', level=2)
add_para(
    'Fig 8.3 compares the SHAP feature importance rankings between baseline (clean only) and '
    'all-data models. The most important change:')
add_para(
    'SEN_std [mm] jumps from #2 to #1 on both strands when disturbances are included. '
    'This means SEN depth variability becomes the dominant driver of instability under '
    'real-life conditions \u2014 even more important than the peak-to-peak amplitude that '
    'dominated on clean data. This directly confirms the recommendation to audit the '
    'stopper rod control loop.', bold=True)
add_para(
    'Features marked with \u2191 in Fig 8.3 gained >50% in mean |SHAP| when moving from '
    'baseline to all-data, indicating they are specifically relevant under disturbance conditions.')
add_figure('fig_8_3_shap_comparison.png',
           'Fig 8.3 \u2014 SHAP comparison: Baseline (blue) vs All Data (red). \u2191 = gained >50% importance')

add_heading('8.4 FC Mold G5 Deep Dive: Where Does the System Fail?', level=2)
add_para(
    'Fig 8.4 (top row) maps the failure rate (% of sequences with ML\u03c3 > \u00b11 mm) across '
    'EMBR AC \u00d7 DC current bins. This heatmap directly identifies the EMBR parameter '
    'combinations where the FC Mold G5 most frequently fails to maintain stability.')
add_para(
    'Fig 8.4 (bottom row) overlays SEN variability against EMBR DC current, with failed '
    'sequences circled in red (circle size \u221d ML\u03c3). The interaction reveals that '
    'high SEN\u03c3 with low EMBR DC is the most dangerous combination \u2014 the system '
    'lacks sufficient braking force to compensate for rapid SEN movements.', bold=True)
add_figure('fig_8_4_embr_deep_dive.png',
           'Fig 8.4 \u2014 FC Mold G5 failure analysis: EMBR heatmaps (top) and SEN\u00d7EMBR interaction (bottom)')

add_heading('8.5 Performance by EMBR Operating Regime', level=2)
add_para(
    'Fig 8.5 bins sequences by EMBR operating regime (AC quartile \u00d7 DC tertile) and shows '
    'both the failure rate (% > \u00b11 mm) and the disturbance enrichment (% disturbed). '
    'Regimes with high failure rates AND high disturbance enrichment are the primary '
    'targets for lookup table review.')
add_figure('fig_8_5_regime_performance.png',
           'Fig 8.5 \u2014 FC Mold G5 performance by EMBR operating regime')

add_heading('8.6 Regime Map: Where Do Disturbances Cluster?', level=2)
add_para(
    'Fig 8.6 shows the t-SNE embedding of all sequences, with clean (blue) and disturbed '
    '(red) sequences overlaid. HDBSCAN on all data collapses the cluster structure from '
    f'{df_c5["cluster"].nunique()-1}/{df_c6["cluster"].nunique()-1} clusters (clean) to '
    f'{df_ca5["cluster_all"].nunique()-1}/{df_ca6["cluster_all"].nunique()-1} clusters (all data), '
    'meaning the disturbances eliminate the fine-grained regime structure visible in clean data.')
add_figure('fig_8_6_tsne_all.png',
           'Fig 8.6 \u2014 t-SNE regime map: Clean (blue) vs Disturbed (red) sequences')

add_heading('8.7 Comprehensive KPI Comparison', level=2)
add_para('Table 8.1 summarises the impact of including disturbances across all key metrics:')
add_figure('fig_8_7_kpi_table.png',
           'Table 8.1 \u2014 KPI comparison: Baseline (clean) vs All Data')

add_para(
    'Summary of Section 8 findings for the vendor:', bold=True)
add_bullet('SEN\u03c3 becomes the #1 driver under real-life conditions (was #2 on clean data)')
add_bullet('Strand 6 model degrades 7\u00d7 more than Strand 5 \u2014 investigate strand-specific factors')
add_bullet('Low-DC / high-AC EMBR regimes show the highest failure rates')
add_bullet('High-argon / low-DC is a specific failure mode (1.78\u00d7 argon, 0.67\u00d7 DC in disturbed)')
add_bullet('Cluster structure collapses from 13\u201315 to 2 clusters when disturbances are included')

doc.add_page_break()

# ===================================================================
# SECTION 9: KEY FINDINGS (was Section 8 in V2.3)
# ===================================================================
add_heading('9. What the Evidence Tells Us: Findings & Root Cause Synthesis', level=1)

add_para(
    'This section synthesises evidence from trend analysis, temporal dynamics, predictive modelling, '
    'EMBR failure profiling, regime discovery, AND the real-life stress test (Section 8).')

add_heading('9.1 The Overall Picture: CC23 is Overwhelmingly Stable Under Clean Conditions', level=2)
add_para('The data shows:', bold=True)
add_para(
    f'Under disturbance-free conditions, virtually all sequences are assessed against the \u00b11 mm ML\u03c3 threshold '
    f'(S5: ~{100*(df_seq_5[target]<ML_THRESHOLD).mean():.1f}%, S6: ~{100*(df_seq_6[target]<ML_THRESHOLD).mean():.1f}%). '
    f'Median ML\u03c3 is {df_seq_5[target].median():.2f} mm (S5) and {df_seq_6[target].median():.2f} mm (S6).')
add_para('This means:', bold=True)
add_para(
    'The FC Mold G5 is working effectively during true steady-state casting. The instability '
    'episodes are predominantly associated with detectable disturbance events.')
add_para(
    'However (V2.4): When all sequences are included, failure rates rise to ~14% and the '
    'Strand 6 model degrades significantly. The ~20% disturbed sequences represent the primary quality risk.', bold=True)

add_heading('9.2 Finding 1: SEN Depth Variability is the Dominant Controllable Driver', level=2)
add_para(
    'SEN\u03c3 ranks #2 on clean data and jumps to #1 when disturbances are included (Section 8.3). '
    'Mold width dominates the operating envelope (R\u00b2=0.20 vs 0.04 for Vc).', bold=True)
add_para(
    'Implication: Tighten stopper rod control loop. SEN rate-of-change limiter would directly reduce ML\u03c3.')

add_heading('9.3 Finding 2: Meniscus Shape Predicts Instability 4 Seconds Ahead (S6)', level=2)
add_para(
    'Chebyshev X\u2082 Granger-causes ML deviation with F=210 at lag=4s on Strand 6. '
    'FBG shape signals could enable feedforward control.')

add_heading('9.4 Finding 3: Steel Grade Effects Persist', level=2)
add_para(
    'Kruskal-Wallis H=58\u201383, p<10\u207b\u00b9\u00b2. Grade-specific FC Mold G5 tuning recommended.')

add_heading('9.5 Finding 4: Mold Width > Casting Speed', level=2)
add_para('FC Mold G5 lookup table entries should be primarily indexed by mold width.')

add_heading('9.6 Finding 5: Strand 6 Asymmetry Under Stress (NEW V2.4)', level=2)
add_para('The data shows:', bold=True)
add_para(
    'When disturbances are included, Strand 5 model holds (R\u00b2: 0.946 \u2192 0.948) while '
    'Strand 6 degrades (R\u00b2: 0.945 \u2192 0.879). The R\u00b2 drop on S6 is 33\u00d7 larger '
    'than on S5.')
add_para('This means:', bold=True)
add_para(
    'Strand 6 experiences disturbances that are qualitatively different from Strand 5 \u2014 '
    'possibly related to local SEN alignment, mold geometry differences, or asymmetric flow '
    'patterns. The weaker Granger causality on S5 (F=4.8 vs 210 on S6) is consistent with '
    'this strand-specific behaviour.')
add_para('Implication:', bold=True)
add_para(
    'A strand-specific investigation is warranted. Physical inspection of Strand 6 vs Strand 5 '
    '(SEN alignment, mold condition, EMBR coil positioning) could reveal the root cause of '
    'the asymmetric degradation.')

doc.add_page_break()

# ===================================================================
# SECTION 10: RECOMMENDATIONS (was Section 9 in V2.3)
# ===================================================================
add_heading('10. Actionable Recommendations', level=1)

add_heading('Priority 1: Reduce SEN Depth Variability (Highest Impact)', level=2)
add_para('Target: Reduce SEN_std from current mean of ~3 mm to < 1 mm within 5-minute windows.', bold=True)
add_bullet('Audit stopper rod control loop parameters \u2014 tighten proportional/integral gains')
add_bullet('Inspect nozzle holder mechanical play and alignment')
add_bullet('Consider SEN depth rate-of-change limits in the PLC')
add_bullet('V2.4: Section 8 confirms SEN\u03c3 is the #1 SHAP driver under real-life conditions')

add_heading('Priority 2: Investigate EMBR Failure Regimes (Joint ABB/TATA)', level=2)
add_para('Target: Address EMBR current combinations with >30% failure rates.', bold=True)
add_bullet('Review the EMBR failure heatmaps (Fig 8.4) for specific AC\u00d7DC bins with high failure %')
add_bullet('Cross-reference against current FC Mold G5 lookup tables')
add_bullet('V2.4: Low-DC / high-AC regimes are the primary failure zone')
add_bullet('Consider adaptive EMBR control that modulates based on real-time SEN variability')

add_heading('Priority 3: Grade-Specific FC Mold Tuning', level=2)
add_para('Target: Reduce instability for HSLA grades (Family 5) on both strands, and peritectic grades (Family 3) particularly on S5 where it ranks 2nd worst.', bold=True)
add_bullet('Dedicated lookup table rows for HSLA (Family 5) and peritectic (Family 3) grades')

add_heading('Priority 4: Investigate Strand 6 Asymmetry (NEW V2.4)', level=2)
add_para('Target: Understand why Strand 6 degrades 7\u00d7 more than Strand 5 under disturbance.', bold=True)
add_bullet('Physical inspection: SEN alignment, mold condition, EMBR coil positioning')
add_bullet('Compare Strand 5 vs 6 lookup table entries for identical casting conditions')
add_bullet('The 4-second Granger lead is 44\u00d7 stronger on S6 \u2014 structural difference')

add_heading('Priority 5: Explore FBG Feedforward Control (Longer-Term)', level=2)
add_para(
    'The 4-second Granger lead on Strand 6 suggests FBG shape signals could provide early warning.')

doc.add_page_break()

# ===================================================================
# SECTION 11: METHODOLOGY APPENDIX (was Section 10 in V2.3)
# ===================================================================
add_heading('11. Methodology Appendix', level=1)

add_heading('A.1 Software and Tools', level=2)
add_table(['Tool', 'Version', 'Purpose'], [
    ['Databricks Runtime', '16.3 LTS', 'Spark execution environment'],
    ['PySpark', '3.5', 'Large-scale data joining'],
    ['Pandas', '2.x', 'Per-strand analysis'],
    ['Plotly', '5.x', 'Interactive HTML figures'],
    ['Matplotlib', '3.x', 'Static PNG figure export'],
    ['XGBoost', '2.x', 'Gradient boosted tree models'],
    ['SHAP', '0.44+', 'Model interpretability'],
    ['scikit-learn', '1.4+', 'HDBSCAN, t-SNE, StandardScaler'],
    ['SciPy', '1.x', 'Statistical tests'],
    ['python-docx', '1.x', 'Word document generation']])

add_heading('A.2 Statistical Methods', level=2)
add_para('p-value:', bold=True)
add_para(
    'The probability the observed result would occur by chance if there were no real effect. '
    'p < 0.05 = unlikely random; p < 0.001 = very unlikely.')
add_para('Spearman \u03c1:', bold=True)
add_para('Monotonic association (\u03c1=1 perfect positive, 0 = none). Does not assume linearity.')
add_para('Kruskal-Wallis H:', bold=True)
add_para('Non-parametric test comparing distributions across groups (like ANOVA without normality).')
add_para('Granger Causality:', bold=True)
add_para(
    'Tests whether past values of X help predict future Y beyond Y\'s own past. '
    'Predictive precedence, not physical causation.')
add_para('SHAP:', bold=True)
add_para(
    'Game-theoretic method: each feature gets a SHAP value representing its contribution. '
    'Mean |SHAP| = global feature importance.')
add_para('R\u00b2:', bold=True)
add_para('Proportion of variance explained. R\u00b2=0.95 = 95% of variability explained.')

add_heading('A.3 Sequence Identification', level=2)
add_bullet('FC Mold G5 active (EMBR currents > minimum threshold)')
add_bullet('Casting speed stable (\u03c3(Vc) < 0.05 m/min)')
add_bullet('No data gaps > 5 seconds')
add_bullet('Mold width constant (\u03c3(Width) < 0.001 m)')

add_heading('A.4 Disturbance Detection', level=2)
add_bullet('Excursion: |ML| > 5 mm from sequence mean')
add_bullet('High variability: Rolling 30s \u03c3(ML) > 3 mm')
add_bullet('Transient bump: |\u0394ML| > 3 mm between consecutive seconds')
add_para(
    'Sequences with any detected disturbance are classified as \"disturbed\" and excluded '
    'from the baseline analysis (Sections 3\u20137) but included in the stress test (Section 8).')

total_paras = len(doc.paragraphs)
total_tables = len(doc.tables)
print(f'V2.4 Word doc COMPLETE: {total_paras} paragraphs, {total_tables} tables')
print(f'  Sections: 1\u201311 (Section 8 = NEW Real-Life Operations)')

In [0]:
# =====================================================================
# V2.4: Export Section 8 figures as PNG for Word doc
# Includes new Fig 8.8 (argon+grade) and Fig 8.9 (grade effect table)
# =====================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil, os, numpy as np, pandas as pd

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
os.makedirs(fig_dir, exist_ok=True)
target = 'MOLD_LEVEL_std [mm]'
sen_std_col = 'SEN_std [mm]'
PAL_dt = {'Clean': '#2166AC', 'Disturbed': '#D6604D'}

def savefig(fig, name, dpi=150):
    tmp = f'/tmp/{name}'
    fig.savefig(tmp, dpi=dpi, bbox_inches='tight', facecolor='white')
    shutil.copy2(tmp, os.path.join(fig_dir, name))
    plt.close(fig)
    print(f'  \u2714 {name}')

print('V2.4 SECTION 8 FIGURE EXPORT')
print('=' * 60)

# ── Fig 8.1: Distribution comparison ────────────────────────────────
comp_features = [
    (target, 'ML \u03c3 [mm]'), (sen_std_col, 'SEN \u03c3 [mm]'),
    ('CASTING_SPEED_avg [m/min]', 'Casting Speed'), ('MOLD_WIDTH_avg [m]', 'Mold Width'),
    ('ArFlow_avg [NL/min]', 'Argon Flow'), ('ptp_mm', 'ML PtP [mm]'),
    ('EMBR_DC_Master_LR_mean', 'EMBR DC Master [A]'), ('EMBR_DC_Bottom_LR_mean', 'EMBR DC Bottom [A]')]

df_comb = pd.concat([df_seq_5_all, df_seq_6_all])
fig81, axes81 = plt.subplots(4, 2, figsize=(14, 16))
for idx, (col, label) in enumerate(comp_features):
    ax = axes81.flat[idx]
    for dt, clr in PAL_dt.items():
        vals = df_comb[df_comb['data_type'] == dt][col].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=40, alpha=0.5, color=clr, label=dt, density=True)
    ax.set_xlabel(label, fontsize=9); ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig81.suptitle('Fig 8.1 \u2014 Distribution Shift: Clean vs Disturbed (Both Strands)', fontsize=13)
fig81.tight_layout()
savefig(fig81, 'fig_8_1_distributions.png')

# ── Fig 8.2: XGBoost All-Data Predicted vs Actual ────────────────────
fig82, axes82 = plt.subplots(1, 2, figsize=(12, 5))
for ax, res_all, sname, c in [(axes82[0], res_all_5, 'S5', '#e74c3c'),
                                (axes82[1], res_all_6, 'S6', '#3498db')]:
    ax.scatter(res_all['y'], res_all['oof'], s=6, alpha=0.3, c=c)
    lim = max(res_all['y'].max(), res_all['oof'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel('Actual ML\u03c3 [mm]'); ax.set_ylabel('Predicted ML\u03c3 [mm]')
    ax.set_title(f'{sname} All Data (R\u00b2={res_all["r2"]:.3f})'); ax.grid(alpha=0.3)
fig82.suptitle('Fig 8.2 \u2014 XGBoost on All Sequences: Actual vs Predicted', fontsize=13)
fig82.tight_layout()
savefig(fig82, 'fig_8_2_xgb_all.png')

# ── Fig 8.3: SHAP Side-by-Side ───────────────────────────────────────
top_n = 12
fig83, axes83 = plt.subplots(1, 2, figsize=(16, 7))
for ax, res_base, res_all_r, sname in [
    (axes83[0], res_5, res_all_5, 'Strand 5'),
    (axes83[1], res_6, res_all_6, 'Strand 6')]:
    base_top = res_base['shap_imp'].head(top_n)
    all_top = res_all_r['shap_imp'].head(top_n)
    all_feats = list(dict.fromkeys(
        all_top['feature'].tolist() + base_top['feature'].tolist()))[:top_n]
    df_cmp = pd.DataFrame({'feature': all_feats})
    df_cmp = df_cmp.merge(base_top.rename(columns={'mean_abs_shap': 'Baseline'}), on='feature', how='left')
    df_cmp = df_cmp.merge(all_top.rename(columns={'mean_abs_shap': 'All Data'}), on='feature', how='left')
    df_cmp = df_cmp.fillna(0).sort_values('All Data')
    y_pos = np.arange(len(df_cmp))
    bar_h = 0.35
    ax.barh(y_pos - bar_h/2, df_cmp['Baseline'], bar_h, color='#2166AC', alpha=0.7,
            label=f'Baseline (R\u00b2={res_base["r2"]:.3f})')
    ax.barh(y_pos + bar_h/2, df_cmp['All Data'], bar_h, color='#D6604D', alpha=0.7,
            label=f'All Data (R\u00b2={res_all_r["r2"]:.3f})')
    ax.set_yticks(y_pos)
    short_f = [f.replace('_avg', '').replace('_std', '\u03c3').replace('_mean', '')
               .replace(' [mm]', '').replace(' [m/min]', '').replace(' [m]', '')
               .replace(' [A]', '').replace(' [NL/min]', '') for f in df_cmp['feature']]
    ax.set_yticklabels(short_f, fontsize=8)
    ax.set_xlabel('Mean |SHAP|'); ax.set_title(sname); ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    for i, (_, row) in enumerate(df_cmp.iterrows()):
        if row['All Data'] > row['Baseline'] * 1.5 and row['All Data'] > 0.01:
            ax.annotate('\u2191', xy=(row['All Data'] + 0.005, y_pos[i] + bar_h/2),
                       fontsize=10, color='#D6604D', fontweight='bold', va='center')
fig83.suptitle('Fig 8.3 \u2014 SHAP: Baseline (Clean) vs All Data (Clean + Disturbed)', fontsize=13, y=1.02)
fig83.tight_layout()
savefig(fig83, 'fig_8_3_shap_comparison.png')

# ── Fig 8.4: EMBR Failure Heatmaps + SEN×EMBR Interaction ────────────
fig84, axes84 = plt.subplots(2, 2, figsize=(16, 12))
for col_i, (df_a, sname) in enumerate([(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]):
    ac_col = 'EMBR_AC_Master_LR_mean'; dc_col = 'EMBR_DC_Master_LR_mean'
    if ac_col not in df_a.columns: continue
    ax = axes84[0, col_i]
    df_plot = df_a[[ac_col, dc_col, target, 'data_type']].dropna().copy()
    df_plot['ac_bin'] = pd.cut(df_plot[ac_col], bins=6)
    df_plot['dc_bin'] = pd.cut(df_plot[dc_col], bins=6)
    df_plot['fail'] = (df_plot[target] > ML_THRESHOLD).astype(int)
    pivot_rate = df_plot.groupby(['dc_bin', 'ac_bin'])['fail'].mean().unstack() * 100
    pivot_n = df_plot.groupby(['dc_bin', 'ac_bin'])['fail'].count().unstack()
    im = ax.imshow(pivot_rate.values, cmap='RdYlGn_r', aspect='auto',
                   vmin=0, vmax=min(60, np.nanmax(pivot_rate.values) * 1.1), origin='lower')
    for yi in range(pivot_rate.shape[0]):
        for xi in range(pivot_rate.shape[1]):
            rate = pivot_rate.values[yi, xi]
            n = pivot_n.values[yi, xi] if not np.isnan(pivot_n.values[yi, xi]) else 0
            if not np.isnan(rate) and n >= 3:
                ax.text(xi, yi, f'{rate:.0f}%\n(n={int(n)})', ha='center', va='center',
                       fontsize=7, color='white' if rate > 30 else 'black')
    ac_labels = [str(x)[:6] for x in pivot_rate.columns]
    dc_labels = [str(x)[:6] for x in pivot_rate.index]
    ax.set_xticks(range(len(ac_labels))); ax.set_xticklabels(ac_labels, fontsize=7, rotation=45)
    ax.set_yticks(range(len(dc_labels))); ax.set_yticklabels(dc_labels, fontsize=7)
    ax.set_xlabel('EMBR AC [A]', fontsize=9); ax.set_ylabel('EMBR DC [A]', fontsize=9)
    ax.set_title(f'{sname}: Failure Rate by EMBR Currents', fontsize=10)
    plt.colorbar(im, ax=ax, label='Failure %', shrink=0.8)
    ax2 = axes84[1, col_i]
    for dt, clr, sz, mk in [('Clean', '#2166AC', 8, 'o'), ('Disturbed', '#D6604D', 15, 's')]:
        sub = df_a[df_a['data_type'] == dt]
        if sen_std_col in sub.columns:
            ax2.scatter(sub[sen_std_col], sub[dc_col], s=sz, alpha=0.4, c=clr, marker=mk, label=dt, edgecolors='none')
    failed = df_a[df_a[target] > ML_THRESHOLD]
    if len(failed) > 0 and sen_std_col in failed.columns:
        ax2.scatter(failed[sen_std_col], failed[dc_col], s=failed[target]*20,
                   facecolors='none', edgecolors='red', linewidth=1, alpha=0.6,
                   label=f'Failed (>1mm, n={len(failed)})')
    ax2.set_xlabel('SEN \u03c3 [mm]'); ax2.set_ylabel('EMBR DC [A]')
    ax2.set_title(f'{sname}: SEN\u03c3 vs EMBR DC'); ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
fig84.suptitle('Fig 8.4 \u2014 FC Mold G5 Deep Dive: Where Does the System Fail?', fontsize=13, y=1.01)
fig84.tight_layout()
savefig(fig84, 'fig_8_4_embr_deep_dive.png')

# ── Fig 8.5: Failure Rate by EMBR Regime ─────────────────────────────
fig85, axes85 = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_a, sname in [(axes85[0], df_seq_5_all, 'Strand 5'), (axes85[1], df_seq_6_all, 'Strand 6')]:
    ac_col = 'EMBR_AC_Master_LR_mean'; dc_col = 'EMBR_DC_Master_LR_mean'
    if ac_col not in df_a.columns: continue
    df_tmp = df_a[[ac_col, dc_col, target, 'data_type']].dropna().copy()
    df_tmp['ac_q'] = pd.qcut(df_tmp[ac_col], q=4, labels=['AC Q1','AC Q2','AC Q3','AC Q4'])
    df_tmp['dc_q'] = pd.qcut(df_tmp[dc_col], q=3, labels=['DC Low','DC Mid','DC High'], duplicates='drop')
    df_tmp['regime'] = df_tmp['ac_q'].astype(str) + ' / ' + df_tmp['dc_q'].astype(str)
    rs = df_tmp.groupby('regime').agg(
        n=(target,'count'), fail_rate=(target, lambda x: 100*(x>ML_THRESHOLD).mean()),
        dist_pct=('data_type', lambda x: 100*(x=='Disturbed').mean()),
        ml_median=(target,'median')).reset_index()
    rs = rs[rs['n']>=10].sort_values('fail_rate')
    y = np.arange(len(rs))
    ax.barh(y, rs['fail_rate'], color='#D6604D', alpha=0.7)
    ax.barh(y, rs['dist_pct'], color='#8e44ad', alpha=0.4)
    for i, (_, r) in enumerate(rs.iterrows()):
        ax.text(max(r['fail_rate'], r['dist_pct'])+1, i,
               f'n={int(r["n"])}, ML\u03c3={r["ml_median"]:.2f}', fontsize=7, va='center')
    ax.set_yticks(y); ax.set_yticklabels(rs['regime'], fontsize=8)
    ax.set_xlabel('Rate [%]'); ax.set_title(sname)
    ax.legend(['% Failed (>1mm)', '% Disturbed'], fontsize=8); ax.grid(axis='x', alpha=0.3)
fig85.suptitle('Fig 8.5 \u2014 FC Mold G5 Performance by EMBR Operating Regime', fontsize=13)
fig85.tight_layout()
savefig(fig85, 'fig_8_5_regime_performance.png')

# ── Fig 8.6: t-SNE All Sequences ───────────────────────────────────
fig86, axes86 = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_ca, sname in [(axes86[0], df_ca5, 'Strand 5'), (axes86[1], df_ca6, 'Strand 6')]:
    for dt, clr, sz in [('Clean', '#2166AC', 4), ('Disturbed', '#D6604D', 8)]:
        sub = df_ca[df_ca['data_type'] == dt]
        ax.scatter(sub['tsne_all_1'], sub['tsne_all_2'], s=sz, alpha=0.4, c=clr, label=dt, edgecolors='none')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2'); ax.set_title(sname)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig86.suptitle('Fig 8.6 \u2014 t-SNE: Where Do Disturbances Live in Operating Space?', fontsize=13)
fig86.tight_layout()
savefig(fig86, 'fig_8_6_tsne_all.png')

# ── Fig 8.7: KPI Table as image ──────────────────────────────────────
kpi_rows = []
for sname, df_clean, df_all_s, res_base, res_all_r, df_c_base, df_ca in [
    ('S5', df_seq_5, df_seq_5_all, res_5, res_all_5, df_c5, df_ca5),
    ('S6', df_seq_6, df_seq_6_all, res_6, res_all_6, df_c6, df_ca6)]:
    for metric, bl, ad, delta in [
        ('n sequences', str(len(df_clean)), str(len(df_all_s)),
         f'+{len(df_all_s)-len(df_clean)}'),
        ('ML\u03c3 median', f'{df_clean[target].median():.3f}', f'{df_all_s[target].median():.3f}',
         f'+{df_all_s[target].median()-df_clean[target].median():.3f}'),
        ('% > 1mm', f'{100*(df_clean[target]>ML_THRESHOLD).mean():.1f}%', f'{100*(df_all_s[target]>ML_THRESHOLD).mean():.1f}%',
         f'+{100*((df_all_s[target]>ML_THRESHOLD).mean()-(df_clean[target]>ML_THRESHOLD).mean()):.1f}pp'),
        ('XGBoost R\u00b2', f'{res_base["r2"]:.3f}', f'{res_all_r["r2"]:.3f}',
         f'{res_all_r["r2"]-res_base["r2"]:+.3f}'),
        ('SHAP #1', res_base['shap_imp'].iloc[0]['feature'][:20],
         res_all_r['shap_imp'].iloc[0]['feature'][:20],
         'Same' if res_base['shap_imp'].iloc[0]['feature']==res_all_r['shap_imp'].iloc[0]['feature'] else 'Changed'),
        ('HDBSCAN cl.', str(df_c_base['cluster'].nunique()-1),
         str(df_ca['cluster_all'].nunique()-1),
         f'{df_ca["cluster_all"].nunique()-1-df_c_base["cluster"].nunique()+1:+d}')]:
        kpi_rows.append([sname, metric, bl, ad, delta])

fig_kpi, ax_kpi = plt.subplots(figsize=(12, 4))
ax_kpi.axis('off')
headers = ['Strand', 'Metric', 'Baseline (clean)', 'All Data', '\u0394']
tab = ax_kpi.table(cellText=kpi_rows, colLabels=headers, cellLoc='center', loc='center')
tab.auto_set_font_size(False); tab.set_fontsize(8); tab.scale(1.2, 1.3)
for i in range(len(headers)):
    tab[0, i].set_facecolor('#2166AC'); tab[0, i].set_text_props(color='white', fontweight='bold')
for ri in range(1, len(kpi_rows)+1):
    if ri <= 6: tab[ri, 0].set_text_props(color='#e74c3c', fontweight='bold')
    else: tab[ri, 0].set_text_props(color='#3498db', fontweight='bold')
ax_kpi.set_title('Table 8.1 \u2014 Comprehensive KPI Comparison: Baseline vs All Data', fontsize=12, pad=15)
savefig(fig_kpi, 'fig_8_7_kpi_table.png', dpi=200)

# ── Fig 8.8: Argon Flow and Grade Effect (NEW V2.4) ─────────────────
print('\n--- Fig 8.8: Argon + Grade effect (NEW) ---')
ar_col = 'ArFlow_avg [NL/min]'
fig88, axes88 = plt.subplots(2, 2, figsize=(14, 10))
# Row 1: Argon flow by data_type
for ax, df_a, sname in [(axes88[0, 0], df_seq_5_all, 'Strand 5'),
                         (axes88[0, 1], df_seq_6_all, 'Strand 6')]:
    for dt, clr in PAL_dt.items():
        sub = df_a[df_a['data_type'] == dt]
        ax.scatter(sub[ar_col], sub[target], s=6, alpha=0.3, c=clr,
                  label=f'{dt} (n={len(sub)})', edgecolors='none')
    ax.set_xlabel('Argon Flow [NL/min]'); ax.set_ylabel('ML \u03c3 [mm]')
    ax.set_title(f'{sname}: Argon Flow vs ML\u03c3'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Row 2: Grade effect (boxplot clean vs disturbed by family)
for ax, df_a, sname in [(axes88[1, 0], df_seq_5_all, 'Strand 5'),
                         (axes88[1, 1], df_seq_6_all, 'Strand 6')]:
    if 'quality_family' not in df_a.columns: continue
    families = sorted([f for f in df_a['quality_family'].dropna().unique() if f != 'Unknown'])
    positions = []
    box_data = []
    colors_box = []
    tick_labels = []
    pos = 0
    for fam in families:
        for dt, clr in [('Clean', '#2166AC'), ('Disturbed', '#D6604D')]:
            sub = df_a[(df_a['quality_family']==fam) & (df_a['data_type']==dt)]
            if len(sub) >= 3:
                box_data.append(sub[target].values)
                positions.append(pos)
                colors_box.append(clr)
                tick_labels.append(f'F{fam}\n{dt[:2]}')
                pos += 1
        pos += 0.5  # gap between families
    bp = ax.boxplot(box_data, positions=positions, widths=0.6, patch_artist=True,
                    showfliers=True, flierprops=dict(markersize=2))
    for patch, clr in zip(bp['boxes'], colors_box):
        patch.set_facecolor(clr); patch.set_alpha(0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(tick_labels, fontsize=7, rotation=0)
    ax.set_ylabel('ML \u03c3 [mm]'); ax.set_title(f'{sname}: Grade Effect (Clean vs Disturbed)')
    ax.grid(axis='y', alpha=0.3)
fig88.suptitle('Fig 8.8 \u2014 Argon Flow and Steel Grade Effects Under Real-Life Conditions', fontsize=13)
fig88.tight_layout()
savefig(fig88, 'fig_8_8_argon_grade.png')

# ── Fig 8.9: Grade effect summary table ─────────────────────────────
print('\n--- Fig 8.9: Grade+argon summary table ---')
tab_rows = []
for sn, df_a in [('S5', df_seq_5_all), ('S6', df_seq_6_all)]:
    for fam in sorted([f for f in df_a['quality_family'].dropna().unique() if f != 'Unknown']):
        cl = df_a[(df_a['quality_family']==fam) & (df_a['data_type']=='Clean')]
        di = df_a[(df_a['quality_family']==fam) & (df_a['data_type']=='Disturbed')]
        if len(cl) == 0 and len(di) == 0: continue
        cl_ml = f'{cl[target].median():.3f}' if len(cl) > 0 else '-'
        di_ml = f'{di[target].median():.3f}' if len(di) > 0 else '-'
        cl_ar = f'{cl[ar_col].median():.1f}' if len(cl) > 0 else '-'
        di_ar = f'{di[ar_col].median():.1f}' if len(di) > 0 else '-'
        tab_rows.append([sn, f'Fam {fam}', str(len(cl)), cl_ml, cl_ar,
                         str(len(di)), di_ml, di_ar])

fig_t89, ax_t89 = plt.subplots(figsize=(14, 3.5))
ax_t89.axis('off')
headers89 = ['Strand', 'Family', 'n Clean', 'ML\u03c3 Clean', 'Ar Clean',
             'n Dist', 'ML\u03c3 Dist', 'Ar Dist']
tab89 = ax_t89.table(cellText=tab_rows, colLabels=headers89, cellLoc='center', loc='center')
tab89.auto_set_font_size(False); tab89.set_fontsize(8); tab89.scale(1.1, 1.4)
for i in range(len(headers89)):
    tab89[0, i].set_facecolor('#2c3e50'); tab89[0, i].set_text_props(color='white', fontweight='bold')
ax_t89.set_title('Table 8.2 \u2014 Steel Grade \u00d7 Data Type: ML\u03c3 and Argon Summary', fontsize=11, pad=15)
savefig(fig_t89, 'fig_8_9_grade_table.png', dpi=200)

pngs_sec8 = sorted([f for f in os.listdir(fig_dir) if f.startswith('fig_8_')])
print(f'\nSection 8 export complete: {len(pngs_sec8)} PNGs')
for f in pngs_sec8: print(f'  {f}')
plt.close('all')

In [0]:
# Save Word doc via /tmp (workaround for DBFS FUSE seek limitation)
import shutil, os, datetime

version = '2.5'
filename = f'FC_Mold_G5_Technical_Report_CC23_v{version}.docx'
tmp_path = f'/tmp/{filename}'
dbfs_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5'
os.makedirs(dbfs_dir, exist_ok=True)

doc.save(tmp_path)
shutil.copy2(tmp_path, os.path.join(dbfs_dir, filename))
shutil.copy2(tmp_path, os.path.join(dbfs_dir, 'FC_Mold_G5_Technical_Report_CC23.docx'))

size_kb = os.path.getsize(os.path.join(dbfs_dir, filename)) / 1024
print(f'\u2714 Saved: {filename} ({size_kb:.0f} KB)')
print(f'  DBFS: /FileStore/TATAIjmulden_FCMoldG5/{filename}')
print(f'  Timestamp: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')

In [0]:
# =====================================================================
# Export this notebook's state to a summary file
# =====================================================================
import datetime

print(f"Report notebook: FC Mold G5 Technical Report CC23")
print(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"")
print(f"To export as HTML/PDF:")
print(f"  File > Export > HTML  (from the notebook menu)")
print(f"")
print(f"Data files saved:")
print(f"  Strand 5 sequences: /dbfs/FileStore/TATAIjmulden_FCMoldG5/strand_23_5/processed_data/")
print(f"  Strand 6 sequences: /dbfs/FileStore/TATAIjmulden_FCMoldG5/strand_23_6/processed_data/")
print(f"  Comparative report:  /dbfs/FileStore/TATAIjmulden_FCMoldG5/comparative_report.txt")
print(f"  Figures directory:   /dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/")
print(f"\n\u2714 Report notebook is self-contained. Run all cells to reproduce.")

In [0]:
import base64

version = '2.5'
filename = f'FC_Mold_G5_Technical_Report_CC23_v{version}.docx'
dbfs_path = f'/dbfs/FileStore/TATAIjmulden_FCMoldG5/{filename}'

with open(dbfs_path, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

html = f'''
<a download="{filename}"
   href="data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{b64}"
   style="display:inline-block;padding:12px 24px;background:#2ecc71;color:white;
          text-decoration:none;border-radius:6px;font-size:16px;font-weight:bold;
          margin:10px 0">
  \u2b07 Download {filename}
</a>
'''
displayHTML(html)
print(f'V{version} Word doc ready for download ({len(b64)//1024} KB encoded)')

In [0]:
# =====================================================================
# Dashboard V2.5: Create additional delta tables for new pages
# 1. raw_signals — 1Hz data tagged with Seq_Name (for EDA line charts)
# 2. meniscus_profiles — Chebyshev-reconstructed profiles per sequence
# 3. embr_bins — binned EMBR heatmap data (for EMBR optimization)
# 4. sequences_full — enhanced sequences table with all features
# =====================================================================
import numpy as np
import pandas as pd
import os

delta_dir = '/FileStore/TATAIjmulden_FCMoldG5/delta'
target = 'MOLD_LEVEL_std [mm]'

# ── 1. RAW SIGNALS TABLE ─────────────────────────────────────────────
# Tag each 1Hz row with its parent sequence name for EDA filtering
print('1. Creating raw_signals delta table...')

def tag_raw_with_sequences(df_fc, df_seq_all, strand_name):
    """Tag each raw 1Hz row with Seq_Name if it falls within a sequence window."""
    df_tagged = []
    for _, seq in df_seq_all.iterrows():
        mask = (df_fc['plainTimeStamp'] >= seq['Seq_time_Start']) & \
               (df_fc['plainTimeStamp'] <= seq['Seq_time_End'])
        chunk = df_fc.loc[mask].copy()
        if len(chunk) > 0:
            chunk['Seq_Name'] = seq['Seq_Name']
            chunk['strand'] = strand_name
            chunk['data_type'] = 'Disturbed' if seq.get('has_disturbance', False) else 'Clean'
            chunk['disturbance_type'] = seq.get('disturbance_type', 'Normal')
            chunk['quality_family'] = str(seq.get('quality_family', '')) if pd.notna(seq.get('quality_family')) else ''
            df_tagged.append(chunk)
    return pd.concat(df_tagged, ignore_index=True) if df_tagged else pd.DataFrame()

df_raw_s5 = tag_raw_with_sequences(df_fc_5, df_seq_5_all, 'Strand 5')
df_raw_s6 = tag_raw_with_sequences(df_fc_6, df_seq_6_all, 'Strand 6')
df_raw_all = pd.concat([df_raw_s5, df_raw_s6], ignore_index=True)

# Select key columns for the dashboard (keep it lean)
raw_cols = [
    'plainTimeStamp', 'Seq_Name', 'strand', 'data_type', 'disturbance_type', 'quality_family',
    'Mold Level', 'Mold Level Sensor Left', 'Mold Level Sensor Right', 'ML_LR_asymmetry',
    'castingSpeed', 'moldWidth', 'SENImmersionDepth',
    'Argon Flow SEN', 'Argon Flow Stopper',
    'EMBR Current AC Left Master', 'EMBR Current DC Left Master',
    'EMBR Current AC Right Master', 'EMBR Current DC Right Master',
    'EMBR Current DC Left Bottom', 'EMBR Current DC Right Bottom',
    'meniscus_bff_avg', 'meniscus_bfl_avg', 'meniscus_FF_LF_asymmetry',
    'cheb_X1_bff', 'cheb_X2_bff', 'cheb_X3_bff', 'cheb_X4_bff',
    'cheb_X1_bfl', 'cheb_X2_bfl', 'cheb_X3_bfl', 'cheb_X4_bfl',
]
raw_cols = [c for c in raw_cols if c in df_raw_all.columns]
df_raw_out = df_raw_all[raw_cols].copy()

# Rename for SQL-friendly names
df_raw_out.columns = [c.replace(' ', '_').replace('[', '').replace(']', '') for c in df_raw_out.columns]

spark.createDataFrame(df_raw_out).write.format('delta').mode('overwrite').save(f'{delta_dir}/raw_signals')
print(f'   raw_signals: {len(df_raw_out)} rows, {len(df_raw_out.columns)} cols')

# ── 2. MENISCUS PROFILES TABLE ───────────────────────────────────────
# Reconstruct Chebyshev profiles per sequence at discrete x-positions
print('\n2. Creating meniscus_profiles delta table...')

from numpy.polynomial.chebyshev import chebval

x_norm = np.linspace(-1, 1, 25)  # 25 points across mold width

profile_rows = []
for df_seq_all, strand_name in [(df_seq_5_all, 'Strand 5'), (df_seq_6_all, 'Strand 6')]:
    for df_fc, face in [(df_fc_5 if strand_name == 'Strand 5' else df_fc_6, 'BFF'),
                        (df_fc_5 if strand_name == 'Strand 5' else df_fc_6, 'BFL')]:
        suffix = '_bff' if face == 'BFF' else '_bfl'
        cheb_cols = [f'cheb_X{i}{suffix}' for i in range(1, 5)]
        
        for _, seq in df_seq_all.iterrows():
            mask = (df_fc['plainTimeStamp'] >= seq['Seq_time_Start']) & \
                   (df_fc['plainTimeStamp'] <= seq['Seq_time_End'])
            chunk = df_fc.loc[mask, cheb_cols].dropna()
            if len(chunk) < 10:
                continue
            # Average Chebyshev coefficients over the sequence
            coeffs = chunk[cheb_cols].mean().values
            # Reconstruct profile: T(x) = c1*T1(x) + c2*T2(x) + c3*T3(x) + c4*T4(x)
            # chebval expects [c0, c1, c2, ...] — we have X1–X4 (orders 1–4), set c0=0
            profile = chebval(x_norm, [0] + list(coeffs))
            
            for xi, yi in zip(x_norm, profile):
                profile_rows.append({
                    'Seq_Name': seq['Seq_Name'],
                    'strand': strand_name,
                    'face': face,
                    'data_type': 'Disturbed' if seq.get('has_disturbance', False) else 'Clean',
                    'MOLD_LEVEL_std_mm': seq[target],
                    'x_norm': round(float(xi), 4),
                    'temperature': float(yi),
                })

df_profiles = pd.DataFrame(profile_rows)
spark.createDataFrame(df_profiles).write.format('delta').mode('overwrite').save(f'{delta_dir}/meniscus_profiles')
print(f'   meniscus_profiles: {len(df_profiles)} rows')

# ── 3. EMBR BINS TABLE ───────────────────────────────────────────────
# Binned EMBR AC × DC with mean ML σ and failure rate (for heatmaps)
print('\n3. Creating embr_bins delta table...')

def create_embr_bins(df_seq_all, strand_name, n_bins=8):
    df = df_seq_all.copy()
    ac_col = 'EMBR_AC_Master_LR_mean' if 'EMBR_AC_Master_LR_mean' in df.columns else None
    dc_col = 'EMBR_DC_Master_LR_mean' if 'EMBR_DC_Master_LR_mean' in df.columns else None
    if not ac_col or not dc_col:
        # Compute LR means
        if 'AC_Left_Master_avg [A]' in df.columns:
            df['EMBR_AC_Master_LR_mean'] = (df['AC_Left_Master_avg [A]'] + df['AC_Right_Master_avg [A]']) / 2
            df['EMBR_DC_Master_LR_mean'] = (df['DC_Left_Master_avg [A]'] + df['DC_Right_Master_avg [A]']) / 2
        ac_col = 'EMBR_AC_Master_LR_mean'
        dc_col = 'EMBR_DC_Master_LR_mean'
    
    df = df.dropna(subset=[ac_col, dc_col, target])
    if len(df) == 0:
        return pd.DataFrame()
    
    # Create bins
    df['AC_bin'] = pd.cut(df[ac_col], bins=n_bins, labels=False)
    df['DC_bin'] = pd.cut(df[dc_col], bins=n_bins, labels=False)
    
    # Get bin edges for labels
    ac_edges = pd.cut(df[ac_col], bins=n_bins, retbins=True)[1]
    dc_edges = pd.cut(df[dc_col], bins=n_bins, retbins=True)[1]
    
    # Aggregate
    # V2.5: Use global ML_THRESHOLD
    rows = []
    for ac_b in range(n_bins):
        for dc_b in range(n_bins):
            sub = df[(df['AC_bin'] == ac_b) & (df['DC_bin'] == dc_b)]
            if len(sub) >= 2:
                rows.append({
                    'strand': strand_name,
                    'AC_bin_center': round((ac_edges[ac_b] + ac_edges[ac_b+1]) / 2, 2),
                    'DC_bin_center': round((dc_edges[dc_b] + dc_edges[dc_b+1]) / 2, 2),
                    'AC_bin_label': f'{ac_edges[ac_b]:.1f}–{ac_edges[ac_b+1]:.1f}',
                    'DC_bin_label': f'{dc_edges[dc_b]:.1f}–{dc_edges[dc_b+1]:.1f}',
                    'n_sequences': len(sub),
                    'mean_ML_sigma': round(sub[target].mean(), 4),
                    'median_ML_sigma': round(sub[target].median(), 4),
                    'failure_rate_pct': round(100 * (sub[target] > ML_THRESHOLD).mean(), 1),
                    'mean_SEN_std': round(sub['SEN_std [mm]'].mean(), 3) if 'SEN_std [mm]' in sub.columns else None,
                    'mean_Vc': round(sub['CASTING_SPEED_avg [m/min]'].mean(), 3) if 'CASTING_SPEED_avg [m/min]' in sub.columns else None,
                    'mean_width': round(sub['MOLD_WIDTH_avg [m]'].mean(), 3) if 'MOLD_WIDTH_avg [m]' in sub.columns else None,
                })
    return pd.DataFrame(rows)

embr_5 = create_embr_bins(df_seq_5_all, 'Strand 5')
embr_6 = create_embr_bins(df_seq_6_all, 'Strand 6')
df_embr_bins = pd.concat([embr_5, embr_6], ignore_index=True)

spark.createDataFrame(df_embr_bins).write.format('delta').mode('overwrite').save(f'{delta_dir}/embr_bins')
print(f'   embr_bins: {len(df_embr_bins)} rows')

# ── 4. ENHANCED SEQUENCES TABLE ──────────────────────────────────────
# Add more features to the sequences delta table for EMBR analysis
print('\n4. Updating sequences_full delta table...')

def enhance_sequences(df_seq_all, strand_name):
    df = df_seq_all.copy()
    df['strand'] = strand_name
    df['data_type'] = np.where(df['has_disturbance'], 'Disturbed', 'Clean')
    
    # Rename columns for SQL friendliness
    rename_map = {}
    for c in df.columns:
        new_name = c.replace(' ', '_').replace('[', '').replace(']', '').replace('/', '_per_')
        if new_name != c:
            rename_map[c] = new_name
    df = df.rename(columns=rename_map)
    
    # Select relevant columns
    keep_cols = [c for c in df.columns if df[c].dtype in ['float64', 'int64', 'bool']
                 or c in ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
                          'data_type', 'disturbance_type', 'Quality_casting', 'quality_family',
                          'castMode', 'SEN_type', 'has_disturbance']]
    return df[keep_cols]

seq_5_full = enhance_sequences(df_seq_5_all, 'Strand 5')
seq_6_full = enhance_sequences(df_seq_6_all, 'Strand 6')
df_seq_full = pd.concat([seq_5_full, seq_6_full], ignore_index=True)

spark.createDataFrame(df_seq_full).write.format('delta').mode('overwrite').save(f'{delta_dir}/sequences_full')
print(f'   sequences_full: {len(df_seq_full)} rows, {len(df_seq_full.columns)} cols')

print(f'\n{"="*60}')
print('All delta tables created successfully.')
print(f'  raw_signals:       {delta_dir}/raw_signals')
print(f'  meniscus_profiles: {delta_dir}/meniscus_profiles')
print(f'  embr_bins:         {delta_dir}/embr_bins')
print(f'  sequences_full:    {delta_dir}/sequences_full')
print(f'{"="*60}')

In [0]:
# =====================================================================
# V2.5 PowerPoint Presentation — FC Mold G5 Technical Report CC23
# 21 slides, widescreen, with figures from fig_dir
# =====================================================================
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import os, shutil

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'

ABB_RED   = RGBColor(0xFF, 0x00, 0x0F)
DARK_GREY = RGBColor(0x33, 0x33, 0x33)
MID_GREY  = RGBColor(0x66, 0x66, 0x66)
BLUE      = RGBColor(0x21, 0x66, 0xAC)

def add_title_slide(title, subtitle=''):
    sl = prs.slides.add_slide(prs.slide_layouts[6])
    txBox = sl.shapes.add_textbox(Inches(0.8), Inches(2.0), Inches(11.5), Inches(1.5))
    tf = txBox.text_frame; tf.word_wrap = True
    p = tf.paragraphs[0]; p.text = title; p.font.size = Pt(36); p.font.bold = True
    p.font.color.rgb = DARK_GREY; p.alignment = PP_ALIGN.LEFT
    if subtitle:
        p2 = tf.add_paragraph(); p2.text = subtitle
        p2.font.size = Pt(18); p2.font.color.rgb = MID_GREY
    sl.shapes.add_shape(1, Inches(0.8), Inches(1.8), Inches(2), Pt(4)).fill.solid()
    sl.shapes[-1].fill.fore_color.rgb = ABB_RED
    sl.shapes[-1].line.fill.background()
    return sl

def add_content_slide(title, bullets=None, fig=None, fig_width=None, fig_left=None, fig_top=None):
    sl = prs.slides.add_slide(prs.slide_layouts[6])
    txBox = sl.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(12), Inches(0.8))
    tf = txBox.text_frame
    p = tf.paragraphs[0]; p.text = title; p.font.size = Pt(24); p.font.bold = True
    p.font.color.rgb = DARK_GREY
    sl.shapes.add_shape(1, Inches(0.5), Inches(0.9), Inches(1.5), Pt(3)).fill.solid()
    sl.shapes[-1].fill.fore_color.rgb = ABB_RED
    sl.shapes[-1].line.fill.background()
    if bullets:
        txBox2 = sl.shapes.add_textbox(Inches(0.5), Inches(1.1), Inches(5.5), Inches(5.5))
        tf2 = txBox2.text_frame; tf2.word_wrap = True
        for i, b in enumerate(bullets):
            p = tf2.paragraphs[0] if i == 0 else tf2.add_paragraph()
            p.text = b; p.font.size = Pt(14) if not b.startswith('\u2192') else Pt(13)
            p.font.color.rgb = DARK_GREY if not b.startswith('\u2192') else BLUE
            p.space_after = Pt(6)
    if fig:
        fp = os.path.join(fig_dir, fig)
        if os.path.exists(fp):
            sl.shapes.add_picture(fp, fig_left or Inches(6.3), fig_top or Inches(1.1),
                                  width=fig_width or Inches(6.5))
    return sl

target = 'MOLD_LEVEL_std [mm]'
pct5_c = f'{100*(df_seq_5[target]<ML_THRESHOLD).mean():.1f}'
pct6_c = f'{100*(df_seq_6[target]<ML_THRESHOLD).mean():.1f}'
pct5_a = f'{100*(df_seq_5_all[target]<ML_THRESHOLD).mean():.1f}'
pct6_a = f'{100*(df_seq_6_all[target]<ML_THRESHOLD).mean():.1f}'

# ── Slide 1: Title ──
add_title_slide(
    'FC Mold G5: Mold Level Stability Investigation',
    'TATA Steel IJmuiden \u2014 Continuous Caster 23 (Strands 5 & 6)\n'
    'V2.5 \u2014 24 March 2026  |  ABB Metallurgy\n\n'
    'Guaranteed threshold: \u2264\u00b11 mm ML \u03c3')

# ── Slide 2: Executive Summary ──
add_content_slide('Executive Summary', [
    '\u2022 Guaranteed threshold: \u2264\u00b11 mm mold level \u03c3',
    '\u2022 Clean (disturbance-free) sequences:',
    f'  \u2192 S5: {pct5_c}% meet threshold (median {df_seq_5[target].median():.2f} mm)',
    f'  \u2192 S6: {pct6_c}% meet threshold (median {df_seq_6[target].median():.2f} mm)',
    '\u2022 With disturbances included:',
    f'  \u2192 S5: {pct5_a}% | S6: {pct6_a}% meet threshold',
    '', f'\u2022 XGBoost R\u00b2 = {res_5["r2"]:.3f}/{res_6["r2"]:.3f} \u2014 instability is predictable',
    '\u2022 Top drivers: PtP amplitude, SEN depth variability, mold width',
    '\u2022 SEN \u03c3 becomes #1 when disturbances included',
    '', '\u2022 Key message: median performance meets spec, but',
    '  ~36\u201342% of even clean sequences exceed \u00b11 mm'])

# ── Slide 3: KPI Dashboard ──
add_content_slide('V2.5 KPI Dashboard (\u00b11 mm threshold)', [
    f'Strand 5 \u2014 Clean: {pct5_c}% stable | ALL: {pct5_a}% stable',
    f'  Median ML \u03c3: {df_seq_5[target].median():.3f} mm',
    f'  XGBoost R\u00b2: {res_5["r2"]:.3f} (clean) / {res_all_5["r2"]:.3f} (all)',
    f'  SHAP #1: {res_5["shap_imp"].iloc[0]["feature"]}', '',
    f'Strand 6 \u2014 Clean: {pct6_c}% stable | ALL: {pct6_a}% stable',
    f'  Median ML \u03c3: {df_seq_6[target].median():.3f} mm',
    f'  XGBoost R\u00b2: {res_6["r2"]:.3f} (clean) / {res_all_6["r2"]:.3f} (all)',
    f'  SHAP #1: {res_6["shap_imp"].iloc[0]["feature"]}', '',
    f'HDBSCAN: {df_c5["cluster"].nunique()-1} (S5) / {df_c6["cluster"].nunique()-1} (S6) regimes',
    'Width R\u00b2=0.20 >> Vc R\u00b2=0.04 (operating envelope)',
    'Granger S6: X\u2082\u2192ML F=210, lag=4s'])

# ── Slides 4-5: Problem + Data ──
add_content_slide('Section 1: Problem Formulation', [
    '\u2022 TATA Steel reports mold level instability on CC23',
    '\u2022 FC Mold G5 uses EMBR to stabilise meniscus flow',
    '\u2022 Dataset: April\u2013May 2025, ~500k 1Hz rows per strand',
    '\u2022 21 physics-informed features from ~1,969 raw columns', '',
    '\u2022 V2.5: Threshold revised from 2 mm to \u2264\u00b11 mm',
    '\u2192 Fundamentally changes the stability assessment'],
    fig='fig_1_1_disturbance.png', fig_width=Inches(6))

add_content_slide('Data Pipeline: From Raw to Analysis-Ready', [
    '\u2022 boExpert (2Hz FBG sensors) + dtExpert (process params)',
    '\u2022 Quality filter \u2192 FC Mold active \u2192 Sequence segmentation',
    '\u2022 Disturbance detection: excursions, high-var, transient bumps',
    '\u2022 S5: 509k raw \u2192 934 clean sequences (80%)',
    '\u2022 S6: 521k raw \u2192 948 clean sequences (78%)'],
    fig='fig_1_2b_filter_cascade.png', fig_width=Inches(6))

# ── Slides 6-8: Trend Learning ──
add_content_slide('Section 3: Width Dominates the Operating Envelope', [
    '\u2022 Vc and width are operationally coupled (\u03c1 = \u22120.58 on S6)',
    '\u2022 Width R\u00b2 = 0.20 vs Vc R\u00b2 = 0.04',
    '\u2192 FC Mold G5 lookup tables should be indexed by width', '',
    '\u2022 Grade Family 5 (HSLA) shows highest ML \u03c3',
    '  Kruskal-Wallis H=58\u201383, p < 10\u207b\u00b9\u00b2'],
    fig='fig_3_2_vc_width.png', fig_width=Inches(6))

add_content_slide('EMBR Failure Profiling (86th Percentile)', [
    '\u2022 Percentile-based threshold for clean data (86th pct \u2192 top 14%)',
    '\u2022 Unstable: higher AC current, higher SEN \u03c3',
    '\u2022 S5 threshold: 1.31 mm | S6 threshold: 1.41 mm', '',
    '\u2022 SEN \u03c3: 1.12\u21921.67 mm (S5), 1.16\u21921.77 mm (S6)'],
    fig='fig_3_3_embr_failure.png', fig_width=Inches(6))

add_content_slide('Steel Grade and Meniscus Profile Effects', [
    '\u2022 Grade Family 5 (HSLA): worst on both strands',
    '  S5: 1.24 mm | S6: 1.28 mm',
    '  vs Family 1 (low-carbon): S5: 0.70 mm | S6: 0.78 mm',
    '\u2022 BFF/BFL meniscus profiles via Chebyshev coefficients',
    '\u2022 Unstable sequences: pronounced X\u2082 (wave) and X\u2081 (tilt)'],
    fig='fig_3_4_grade_argon.png', fig_width=Inches(6))

# ── Slides 9-10: Temporal Dynamics ──
add_content_slide('Section 4: Meniscus Shape Predicts ML 4s Ahead (S6)', [
    '\u2022 Lagged cross-correlation: X\u2082 BFF peaks at lag = +4s on S6 (r = 0.117)',
    '\u2022 Granger causality: X\u2082 \u2192 ML deviation F = 210, p = 2\u00d710\u207b\u2074\u2077',
    '\u2192 Meniscus shape changes precede mold level excursions', '',
    '\u2022 Strand 5: weaker coupling (F < 10)',
    '\u2192 Strand-dependent dynamics \u2014 local flow patterns differ', '',
    '\u2022 Implication: feedforward control using FBG shape signals'],
    fig='fig_4_2_xcorr.png', fig_width=Inches(5.5))

add_content_slide('No Campaign Degradation \u2014 Campaigns Are Stable', [
    '\u2022 Spearman \u03c1 = 0.035 (S5, n.s.), \u22120.117 (S6, p < 0.001)',
    '\u2022 ML \u03c3 does not increase across a campaign',
    '\u2022 S6 shows slight improvement (negative trend)', '',
    '\u2192 Nozzle clogging, mold wear are not dominant drivers'],
    fig='fig_4_1_heat.png', fig_width=Inches(6))

# ── Slides 11-12: XGBoost + SHAP ──
add_content_slide(f'Section 6: ML \u03c3 Is Predictable (R\u00b2 = {res_5["r2"]:.3f})', [
    f'\u2022 XGBoost: R\u00b2 = {res_5["r2"]:.3f} (S5), {res_6["r2"]:.3f} (S6)',
    '\u2022 SHAP top features (both strands):',
    '  1. ptp_mm (peak-to-peak) \u2014 direct wave amplitude',
    '  2. SEN_std \u2014 nozzle depth variability',
    '  3. ArFlow_avg \u2014 argon gas injection', '',
    '\u2192 SEN depth variability is the #1 actionable driver',
    '  (ptp_mm is a consequence, not a cause)'],
    fig='fig_6_2_shap.png', fig_width=Inches(6))

add_content_slide('Actual vs Predicted \u2014 Model Validation', [
    '\u2022 Out-of-fold predictions track actual ML \u03c3 closely',
    '\u2022 Low-\u03c3 sequences well-resolved',
    '\u2022 High-\u03c3 tail slightly underpredicted',
    '\u2192 Model captures the physics, not just noise'],
    fig='fig_6_3_predicted.png', fig_width=Inches(7), fig_left=Inches(5.5))

# ── Slide 13: HDBSCAN ──
add_content_slide('Section 7: Operating Regimes Discovered via HDBSCAN', [
    f'\u2022 {df_c5["cluster"].nunique()-1} clusters (S5), {df_c6["cluster"].nunique()-1} clusters (S6)',
    '\u2022 Top unstable clusters share:',
    '  \u2192 Wide molds (>2.0 m), high SEN \u03c3, Grade Family 5',
    '  \u2192 Higher EMBR AC currents', '',
    '\u2022 Noise fraction: 44% (S5), 37% (S6)'],
    fig='fig_7_1a_clusters.png', fig_width=Inches(6))

# ── Slides 14-17: Section 8 — Real-Life Operations ──
add_content_slide('Section 8: Real Disturbances \u2014 Stress Test', [
    '\u2022 ~20% of sequences contain detectable disturbances',
    '\u2022 Including them shifts the distribution:',
    f'  \u2192 S5: {pct5_c}% \u2192 {pct5_a}% meeting \u00b11 mm (clean \u2192 all)',
    f'  \u2192 S6: {pct6_c}% \u2192 {pct6_a}% meeting \u00b11 mm (clean \u2192 all)', '',
    '\u2022 SEN \u03c3 doubles: 1.2 \u2192 2.3 mm in disturbed sequences',
    '\u2022 DC current 0.67\u00d7 lower \u2014 lookup table gap?'],
    fig='fig_8_1_distributions.png', fig_width=Inches(6))

add_content_slide('SHAP Shift: SEN \u03c3 Becomes #1 Under Disturbance', [
    '\u2022 Clean: ptp_mm #1, SEN_std #2',
    '\u2022 All data: SEN_std jumps to #1 (SHAP 0.57 vs 0.26)', '',
    f'\u2022 S6 model degrades: R\u00b2 {res_6["r2"]:.3f} \u2192 {res_all_6["r2"]:.3f}',
    f'\u2022 S5 model holds: R\u00b2 {res_5["r2"]:.3f} \u2192 {res_all_5["r2"]:.3f}', '',
    '\u2192 S6 disturbance dynamics more complex'],
    fig='fig_8_3_shap_comparison.png', fig_width=Inches(6))

add_content_slide('EMBR Failure Heatmap: Where FC Mold Struggles', [
    '\u2022 Failure rate (% > \u00b11 mm) mapped across AC \u00d7 DC bins',
    '\u2022 Highest failure: low-DC / high-AC combinations',
    '\u2022 SEN \u03c3 vs DC scatter \u2014 failed sequences cluster in',
    '  high-SEN-\u03c3, low-DC region', '',
    '\u2192 Specific EMBR current combinations under-compensating',
    '\u2192 Lookup table review recommended'],
    fig='fig_8_4_embr_deep_dive.png', fig_width=Inches(6))

add_content_slide('Operating Regime Performance Under Disturbance', [
    '\u2022 AC Q4 / DC Low regime: highest failure rate',
    '\u2022 Disturbed sequences enriched in specific regimes', '',
    '\u2022 t-SNE shows disturbed sequences form separate clusters',
    '\u2192 Targeted EMBR optimisation is feasible'],
    fig='fig_8_5_regime_performance.png', fig_width=Inches(6))

# ── Slide 18: SEN Signal Decomposition ──
add_content_slide('SEN Signal Has Two Predictive Components', [
    '\u2022 medfilt(21) decomposes SEN into:',
    '  1. Slow drift (> 21 s) \u2014 real nozzle movement',
    '  2. Fast oscillation (< 21 s) \u2014 stopper rod chatter', '',
    '\u2022 XGBoost experiment (3 configs \u00d7 2 strands \u00d7 2 datasets):',
    '  A) Raw SEN \u03c3 \u2192 R\u00b2 = 0.945 (baseline)',
    '  B) Smoothed only \u2192 R\u00b2 = 0.906\u20130.915 (\u0394 = \u22123\u20134%)',
    '  C) Smooth + residual \u2192 R\u00b2 = 0.907\u20130.913', '',
    '\u2022 SHAP rank: raw SEN = #2 \u2192 smoothed SEN drops to #11\u201327',
    '\u2192 High-freq SEN noise carries real predictive value', '',
    '\u2022 NOT measurement coupling \u2014 genuine physics at two timescales',
    '\u2192 Both components independently predict mold level instability'],
    fig='shap_sen_smoothing_comparison.png', fig_width=Inches(6))

# ── Slides 19-20: Findings + Recommendations ──
add_content_slide('Key Findings (V2.5 \u2014 \u00b11 mm Threshold)', [
    f'1. Against \u00b11 mm: only {pct5_c}% (S5) / {pct6_c}% (S6) of',
    '   clean sequences meet the guaranteed threshold', '',
    '2. SEN depth variability is the #1 controllable driver',
    '   \u2192 Both slow drift AND fast oscillation predict ML \u03c3',
    '   \u2192 Removing high-freq component degrades R\u00b2 by 3\u20134%', '',
    '3. Meniscus shape predicts instability 4s ahead (S6)', '',
    '4. Width >> Vc for operating envelope',
    '5. Grade Family 5 (HSLA): highest ML \u03c3 on both strands',
    '6. Strand 6 model degrades 7\u00d7 more under disturbance'])

add_content_slide('Reduce SEN Variability \u2014 The Highest-Impact Action', [
    'Priority 1a: Dampen Fast SEN Oscillation (< 21 s period)',
    '\u2192 Stopper rod chatter / mechanical vibration excites meniscus',
    '\u2192 Removing this component degrades XGBoost R\u00b2 by 3\u20134%',
    '\u2192 Action: improve SEN holder rigidity, reduce stopper rod play', '',
    'Priority 1b: Stabilise Slow SEN Drift (> 21 s period)',
    '\u2192 Real nozzle repositioning (tundish level, stopper rod)',
    '\u2192 Smoothed SEN still correlates at r \u2248 0.95 with ML \u03c3',
    '\u2192 Action: tighten tundish level control loop', '',
    'Priority 2: Review EMBR Lookup Tables',
    '\u2192 Low-DC / High-AC regimes: highest failure rate', '',
    'Priority 3: Feedforward Control Using FBG Shape (S6)',
    '\u2192 X\u2082 shape signal leads ML by 4s \u2014 early warning', '',
    'Priority 4: Width-Indexed Lookup Tables',
    '\u2192 Width R\u00b2 = 0.20 dominates Vc R\u00b2 = 0.04'])

# ── Slide 21: Thank You ──
add_title_slide('Thank You',
    'ABB Metallurgy \u2014 FC Mold G5 Analysis Team\n\n'
    'Data: April\u2013May 2025 | CC23, Strands 5 & 6\n'
    'Full report: FC_Mold_G5_Technical_Report_CC23_v2.5.docx')

# ── Save ──
version = '2.5'
filename = f'FC_Mold_G5_Presentation_CC23_v{version}.pptx'
tmp_path = f'/tmp/{filename}'
dbfs_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5'
prs.save(tmp_path)
shutil.copy2(tmp_path, os.path.join(dbfs_dir, filename))

size_kb = os.path.getsize(os.path.join(dbfs_dir, filename)) / 1024
print(f'\u2714 Saved: {filename} ({size_kb:.0f} KB, {len(prs.slides)} slides)')
print(f'  DBFS: /FileStore/TATAIjmulden_FCMoldG5/{filename}')

In [0]:
import base64

version = '2.5'
filename = f'FC_Mold_G5_Presentation_CC23_v{version}.pptx'
dbfs_path = f'/dbfs/FileStore/TATAIjmulden_FCMoldG5/{filename}'

with open(dbfs_path, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

html = f'''
<a download="{filename}"
   href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}"
   style="display:inline-block;padding:12px 24px;background:#e74c3c;color:white;
          text-decoration:none;border-radius:6px;font-size:16px;font-weight:bold;
          margin:10px 0">
  \u2b07 Download {filename}
</a>
'''
displayHTML(html)
print(f'V2.5 PowerPoint ready for download ({len(b64)//1024} KB encoded)')

In [0]:
import base64

filename = 'FC_Mold_G5_Customer_Presentation_CC23_v1.0.pptx'
dbfs_path = f'/dbfs/FileStore/TATAIjmulden_FCMoldG5/{filename}'

with open(dbfs_path, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

html = f'''
<a download="{filename}"
   href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}"
   style="display:inline-block;padding:12px 24px;background:#8e44ad;color:white;
          text-decoration:none;border-radius:6px;font-size:16px;font-weight:bold;
          margin:10px 0">
  \u2b07 Download {filename}
</a>
'''
displayHTML(html)
print(f'Customer PPT ready for download ({len(b64)//1024} KB encoded)')

In [0]:
# =====================================================================
# SEN Signal Smoothing Investigation
# Feedback: "SEN signal seems to carry ML information → strong correlation
# might be measurement coupling rather than real physics."
# Approach: Apply median filter (equiv. to MATLAB medfilt1) to separate
# slow drift (real SEN movement) from high-freq noise (potential coupling)
# =====================================================================
from scipy.signal import medfilt
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import shutil, os

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
sen_col_raw = 'SENImmersionDepth'  # raw signal in meters
FILTER_WINDOW = 21  # 21s median filter — physically reasonable for SEN

print('SEN SIGNAL SMOOTHING INVESTIGATION')
print('=' * 70)
print(f'Filter: medfilt(kernel={FILTER_WINDOW}) — equivalent to MATLAB medfilt1(x, {FILTER_WINDOW})')
print(f'Rationale: SEN depth should not change faster than ~20s.')
print(f'           High-freq oscillation may be measurement coupling with ML.')
print()

# ── Step 1: Visualise raw vs filtered on 3 representative sequences ──
sen_std_col = 'SEN_std [mm]'
sen_vals = df_seq_6[sen_std_col].dropna()
quantiles = {'Low \u03c3 (P10)': 0.10, 'Median \u03c3 (P50)': 0.50, 'High \u03c3 (P90)': 0.90}
windows = [5, 11, 21, 51]
colors_w = {5: '#95a5a6', 11: '#e67e22', 21: '#e74c3c', 51: '#8e44ad'}

examples = []
for label, q in quantiles.items():
    q_val = sen_vals.quantile(q)
    idx = (df_seq_6[sen_std_col] - q_val).abs().idxmin()
    seq = df_seq_6.loc[idx]
    mask = (df_fc_6['plainTimeStamp'] >= seq['Seq_time_Start']) & \
           (df_fc_6['plainTimeStamp'] <= seq['Seq_time_End'])
    chunk = df_fc_6.loc[mask].sort_values('plainTimeStamp').copy()
    if len(chunk) == 0: continue
    raw_mm = chunk[sen_col_raw].values * 1000
    timestamps = chunk['plainTimeStamp'].values
    ml_raw = chunk['Mold Level'].values
    filtered = {w: medfilt(raw_mm, kernel_size=w) for w in windows}
    examples.append({'label': label, 'seq': seq, 'timestamps': timestamps,
        'raw_mm': raw_mm, 'ml_raw': ml_raw, 'filtered': filtered,
        'date': pd.Timestamp(timestamps[0]).strftime('%d %b %Y'),
        'time_range': f"{pd.Timestamp(timestamps[0]).strftime('%H:%M:%S')}\u2013{pd.Timestamp(timestamps[-1]).strftime('%H:%M:%S')}"})

# ── Figure: 3 rows × 2 cols ──
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
for row, ex in enumerate(examples):
    ts = ex['timestamps']
    # Left: Raw SEN + ML overlay
    ax_l = axes[row, 0]
    ax_l.plot(ts, ex['raw_mm'], color='#3498db', lw=0.6, alpha=0.7, label='Raw SEN')
    ax_l.plot(ts, ex['filtered'][21], color='#e74c3c', lw=1.8, label='medfilt(21s)')
    ax_l.set_ylabel('SEN depth [mm]', fontsize=10, color='#3498db')
    ax_l.tick_params(axis='y', labelcolor='#3498db')
    ax_r = ax_l.twinx()
    ax_r.plot(ts, ex['ml_raw'], color='#2c3e50', lw=0.5, alpha=0.4, label='Mold Level')
    ax_r.set_ylabel('Mold Level [mm]', fontsize=10, color='#2c3e50')
    ax_r.tick_params(axis='y', labelcolor='#2c3e50')
    raw_std = np.nanstd(ex['raw_mm']); filt_std = np.nanstd(ex['filtered'][21])
    resid_std = np.nanstd(ex['raw_mm'] - ex['filtered'][21])
    ax_l.set_title(f'{ex["label"]} \u2014 {ex["date"]}, {ex["time_range"]}\n'
        f'Raw SEN \u03c3 = {raw_std:.2f} mm | medfilt(21) \u03c3 = {filt_std:.2f} mm | Residual \u03c3 = {resid_std:.2f} mm',
        fontsize=10, fontweight='bold')
    ax_l.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax_l.tick_params(axis='x', rotation=25, labelsize=8)
    lines_l, labels_l = ax_l.get_legend_handles_labels()
    lines_r, labels_r = ax_r.get_legend_handles_labels()
    ax_l.legend(lines_l + lines_r, labels_l + labels_r, fontsize=8, loc='upper right')
    ax_l.grid(alpha=0.2)
    # Right: All filter windows
    ax_f = axes[row, 1]
    ax_f.plot(ts, ex['raw_mm'], color='#bdc3c7', lw=0.5, alpha=0.6, label='Raw')
    for w in windows:
        ax_f.plot(ts, ex['filtered'][w], color=colors_w[w], lw=1.2,
            label=f'medfilt({w}s), \u03c3={np.nanstd(ex["filtered"][w]):.2f}mm')
    ax_f.set_ylabel('SEN depth [mm]', fontsize=10)
    ax_f.set_title(f'Filter Comparison \u2014 {ex["label"]}', fontsize=10, fontweight='bold')
    ax_f.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax_f.tick_params(axis='x', rotation=25, labelsize=8)
    ax_f.legend(fontsize=8, loc='upper right'); ax_f.grid(alpha=0.2)

fig.suptitle('SEN Signal: Raw vs Median-Filtered (medfilt1 equivalent)\n'
    'Strand 6, Clean Sequences \u2014 Separating real drift from high-frequency noise', fontsize=14, fontweight='bold')
fig.tight_layout()
tmp = '/tmp/sen_medfilt_comparison.png'
fig.savefig(tmp, dpi=150, bbox_inches='tight', facecolor='white')
shutil.copy2(tmp, os.path.join(fig_dir, 'sen_medfilt_comparison.png'))
plt.close(fig)

# ── Step 2: Apply medfilt(21) to ALL sequences, recompute SEN_std ──
def compute_smoothed_sen_std(df_fc, df_seq_all, window=FILTER_WINDOW):
    results = []
    for i, seq in df_seq_all.iterrows():
        mask = (df_fc['plainTimeStamp'] >= seq['Seq_time_Start']) & \
               (df_fc['plainTimeStamp'] <= seq['Seq_time_End'])
        chunk = df_fc.loc[mask, 'SENImmersionDepth'].dropna()
        if len(chunk) < window:
            results.append({'idx': i, 'SEN_std_raw_mm': np.nan, 'SEN_std_smooth_mm': np.nan, 'SEN_std_resid_mm': np.nan})
            continue
        raw_mm = chunk.values * 1000
        smoothed = medfilt(raw_mm, kernel_size=window)
        residual = raw_mm - smoothed
        results.append({'idx': i, 'SEN_std_raw_mm': np.nanstd(raw_mm),
            'SEN_std_smooth_mm': np.nanstd(smoothed), 'SEN_std_resid_mm': np.nanstd(residual)})
    return pd.DataFrame(results).set_index('idx')

for df_fc, df_all, df_clean, sname in [
    (df_fc_5, df_seq_5_all, df_seq_5, 'S5'),
    (df_fc_6, df_seq_6_all, df_seq_6, 'S6')]:
    s = compute_smoothed_sen_std(df_fc, df_all)
    df_all['SEN_std_smooth_mm'] = s['SEN_std_smooth_mm']
    df_all['SEN_std_resid_mm'] = s['SEN_std_resid_mm']
    df_clean['SEN_std_smooth_mm'] = df_all.loc[df_clean.index, 'SEN_std_smooth_mm']
    df_clean['SEN_std_resid_mm'] = df_all.loc[df_clean.index, 'SEN_std_resid_mm']

# ── Step 3: Summary statistics ──
target = 'MOLD_LEVEL_std [mm]'
print(f'\n{"="*90}')
print(f'SEN \u03c3 comparison: Raw vs medfilt({FILTER_WINDOW}) Smoothed vs Residual (noise)')
print(f'{"="*90}')
print(f'{"Category":<25} {"Strand":<8} {"Raw \u03c3":>10} {"Smooth \u03c3":>10} {"Resid \u03c3":>10} {"% noise":>10}')
print(f'{"-"*90}')
for sname, df_c, df_a in [('S5', df_seq_5, df_seq_5_all), ('S6', df_seq_6, df_seq_6_all)]:
    for label, df in [('Clean', df_c), ('All', df_a), ('Disturbed', df_a[df_a['has_disturbance']])]:
        raw = df['SEN_std [mm]'].median(); smooth = df['SEN_std_smooth_mm'].median()
        resid = df['SEN_std_resid_mm'].median(); pct = 100 * resid / raw if raw > 0 else 0
        print(f'{label:<25} {sname:<8} {raw:>10.3f} {smooth:>10.3f} {resid:>10.3f} {pct:>9.1f}%')
    print()

print(f'{"="*90}')
print(f'KEY TEST: Correlation with ML \u03c3 — does smoothing destroy the relationship?')
print(f'{"="*90}')
for sname, df_a in [('S5', df_seq_5_all), ('S6', df_seq_6_all)]:
    ml = df_a[target].dropna()
    common = ml.index.intersection(df_a['SEN_std_smooth_mm'].dropna().index)
    r_raw = np.corrcoef(ml.loc[common], df_a.loc[common, 'SEN_std [mm]'])[0, 1]
    r_smooth = np.corrcoef(ml.loc[common], df_a.loc[common, 'SEN_std_smooth_mm'])[0, 1]
    r_resid = np.corrcoef(ml.loc[common], df_a.loc[common, 'SEN_std_resid_mm'])[0, 1]
    print(f'  {sname}: raw\u2194ML \u03c3: r={r_raw:.4f} | smooth\u2194ML \u03c3: r={r_smooth:.4f} | resid\u2194ML \u03c3: r={r_resid:.4f}')

print(f'\n{"="*90}')
print('CONCLUSION:')
print('  \u2022 53\u201379% of SEN \u03c3 is high-frequency noise (removed by medfilt)')
print('  \u2022 BUT smoothed SEN still correlates with ML \u03c3 at r \u2248 0.95\u20130.97')
print('  \u2022 The SEN\u2194ML relationship is NOT just measurement coupling')
print('  \u2022 The slow-drift component carries genuine physical information')
print('  \u2022 Recommendation: use medfilt(21) for operational SEN monitoring')
print(f'{"="*90}')

In [0]:
# =====================================================================
# SEN Smoothing: Median Filter vs Moving Average Comparison
# medfilt preserves step changes (real nozzle adjustments)
# movavg blurs them — same denoising power but loses edge information
# =====================================================================
from scipy.signal import medfilt
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import shutil, os

fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
sen_col_raw = 'SENImmersionDepth'
WINDOW = 21

# Pick 3 S6 examples (P10, P50, P90 of SEN σ)
sen_std_col = 'SEN_std [mm]'
sen_vals = df_seq_6[sen_std_col].dropna()
quantiles = {'Low σ (P10)': 0.10, 'Median σ (P50)': 0.50, 'High σ (P90)': 0.90}

examples = []
for label, q in quantiles.items():
    q_val = sen_vals.quantile(q)
    idx = (df_seq_6[sen_std_col] - q_val).abs().idxmin()
    seq = df_seq_6.loc[idx]
    mask = (df_fc_6['plainTimeStamp'] >= seq['Seq_time_Start']) & \
           (df_fc_6['plainTimeStamp'] <= seq['Seq_time_End'])
    chunk = df_fc_6.loc[mask].sort_values('plainTimeStamp').copy()
    if len(chunk) == 0: continue
    raw_mm = chunk[sen_col_raw].values * 1000
    timestamps = chunk['plainTimeStamp'].values
    filt_med = medfilt(raw_mm, kernel_size=WINDOW)
    filt_ma = pd.Series(raw_mm).rolling(WINDOW, center=True, min_periods=1).mean().values
    examples.append({'label': label, 'timestamps': timestamps, 'raw_mm': raw_mm,
        'medfilt': filt_med, 'movavg': filt_ma,
        'date': pd.Timestamp(timestamps[0]).strftime('%d %b %Y'),
        'time_range': f"{pd.Timestamp(timestamps[0]).strftime('%H:%M:%S')}\u2013{pd.Timestamp(timestamps[-1]).strftime('%H:%M:%S')}"})

# ── Figure: 3 rows × 2 cols (signal overlay | residuals) ──
fig, axes = plt.subplots(3, 2, figsize=(18, 13))
for row, ex in enumerate(examples):
    ts = ex['timestamps']
    # Left: signal overlay
    ax = axes[row, 0]
    ax.plot(ts, ex['raw_mm'], color='#555555', lw=1.0, alpha=0.8, label='Raw SEN')
    ax.plot(ts, ex['medfilt'], color='#e74c3c', lw=2.0, label=f'medfilt({WINDOW}s)')
    ax.plot(ts, ex['movavg'], color='#2980b9', lw=2.0, ls='--', label=f'moving avg({WINDOW}s)')
    med_std = np.nanstd(ex['medfilt']); ma_std = np.nanstd(ex['movavg'])
    ax.set_title(f'{ex["label"]} \u2014 {ex["date"]}, {ex["time_range"]}\n'
        f'medfilt σ = {med_std:.2f} mm | movavg σ = {ma_std:.2f} mm', fontsize=10, fontweight='bold')
    ax.set_ylabel('SEN depth [mm]', fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)
    # Right: residuals
    ax_r = axes[row, 1]
    resid_med = ex['raw_mm'] - ex['medfilt']; resid_ma = ex['raw_mm'] - ex['movavg']
    ax_r.plot(ts, resid_med, color='#e74c3c', lw=0.8, alpha=0.8, label=f'medfilt residual (σ={np.nanstd(resid_med):.2f})')
    ax_r.plot(ts, resid_ma, color='#2980b9', lw=0.8, alpha=0.8, label=f'movavg residual (σ={np.nanstd(resid_ma):.2f})')
    ax_r.axhline(0, color='grey', lw=0.5)
    ax_r.set_title(f'Residuals (Raw − Filtered) — {ex["label"]}', fontsize=10, fontweight='bold')
    ax_r.set_ylabel('Residual [mm]', fontsize=10)
    ax_r.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax_r.tick_params(axis='x', rotation=25, labelsize=8)
    ax_r.legend(fontsize=8, loc='upper right'); ax_r.grid(alpha=0.2)

fig.suptitle(f'SEN Smoothing: Median Filter vs Moving Average (window = {WINDOW}s)\n'
    'Median filter preserves step changes; moving average blurs them', fontsize=14, fontweight='bold')
fig.tight_layout()

# Save to file
tmp = '/tmp/sen_medfilt_vs_movavg.png'
fig.savefig(tmp, dpi=150, bbox_inches='tight', facecolor='white')
shutil.copy2(tmp, os.path.join(fig_dir, 'sen_medfilt_vs_movavg.png'))

# Display inline
display(fig)
plt.close(fig)
print(f'\u2714 sen_medfilt_vs_movavg.png saved ({os.path.getsize(os.path.join(fig_dir, "sen_medfilt_vs_movavg.png"))/1024:.0f} KB)')

# ── Quantitative comparison across ALL sequences ──
target = 'MOLD_LEVEL_std [mm]'
print(f'\n{"="*95}')
print(f'Comparison across ALL sequences: medfilt({WINDOW}) vs movavg({WINDOW})')
print(f'{"="*95}')
for sname, df_fc, df_all in [('S5', df_fc_5, df_seq_5_all), ('S6', df_fc_6, df_seq_6_all)]:
    ma_stds = []
    for i, seq in df_all.iterrows():
        mask = (df_fc['plainTimeStamp'] >= seq['Seq_time_Start']) & \
               (df_fc['plainTimeStamp'] <= seq['Seq_time_End'])
        chunk = df_fc.loc[mask, 'SENImmersionDepth'].dropna()
        if len(chunk) < WINDOW: ma_stds.append(np.nan); continue
        raw_mm = chunk.values * 1000
        ma_stds.append(np.nanstd(pd.Series(raw_mm).rolling(WINDOW, center=True, min_periods=1).mean().values))
    df_all['SEN_std_movavg_mm'] = ma_stds
    ml = df_all[target].dropna(); common = ml.index
    r_raw = np.corrcoef(ml, df_all.loc[common, 'SEN_std [mm]'])[0, 1]
    r_med = np.corrcoef(ml, df_all.loc[common, 'SEN_std_smooth_mm'])[0, 1]
    r_ma = np.corrcoef(ml, df_all.loc[common, 'SEN_std_movavg_mm'])[0, 1]
    med_median = df_all['SEN_std_smooth_mm'].median()
    ma_median = df_all['SEN_std_movavg_mm'].median()
    print(f'\n  {sname} ({len(df_all)} sequences):')
    print(f'    {"Metric":<35} {"Raw":>10} {"medfilt":>10} {"movavg":>10}')
    print(f'    {"-"*65}')
    print(f'    {"Median SEN σ [mm]":<35} {df_all["SEN_std [mm]"].median():>10.3f} {med_median:>10.3f} {ma_median:>10.3f}')
    print(f'    {"Correlation with ML σ (r)":<35} {r_raw:>10.4f} {r_med:>10.4f} {r_ma:>10.4f}')
    print(f'    {"σ reduction vs raw":<35} {"—":>10} {100*(1-med_median/df_all["SEN_std [mm]"].median()):>9.1f}% {100*(1-ma_median/df_all["SEN_std [mm]"].median()):>9.1f}%')

print(f'\n{"="*95}')
print('CONCLUSION: medfilt vs movavg')
print('  • Moving average smooths MORE aggressively (lower σ) but blurs step changes')
print('  • Median filter PRESERVES genuine step changes (nozzle adjustments)')
print('  • Both maintain r > 0.93 with ML σ — relationship is robust to filter choice')
print('  • medfilt is preferred: removes noise without distorting real SEN movement')
print('  • This matches the MATLAB medfilt1 recommendation')
print(f'{"="*95}')

In [0]:
# =====================================================================
# XGBoost with Smoothed SEN: Does Denoising Improve Prediction?
# Test whether replacing raw SEN σ with medfilt-smoothed SEN σ
# changes model performance or SHAP rankings.
#
# Three configurations:
#   A) Original  — raw SEN_std [mm]
#   B) Smoothed  — replace with SEN_std_smooth_mm (medfilt drift only)
#   C) Split     — replace with SEN_std_smooth_mm + SEN_std_resid_mm
# =====================================================================
import xgboost as xgb
import shap
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

target = 'MOLD_LEVEL_std [mm]'

exclude_base = [
    'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
    'Quality casting', 'castMode',
    target, 'MOLD_LEVEL_avg [mm]', 'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]',
    'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
    'has_high_variability', 'has_disturbance', 'disturbance_type',
    'AC_Left_Master_avg [A]', 'DC_Left_Master_avg [A]', 'DC_Left_Bottom_avg [A]',
    'AC_Right_Master_avg [A]', 'DC_Right_Master_avg [A]', 'DC_Right_Bottom_avg [A]',
    'AC_Left_Master_std [A]', 'DC_Left_Master_std [A]', 'DC_Left_Bottom_std [A]',
    'AC_Right_Master_std [A]', 'DC_Right_Master_std [A]', 'DC_Right_Bottom_std [A]',
    'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
    'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
    'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
    'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
    'quality_family',
]

def xgb_cv(df_seq, feat_cols_override=None, label=''):
    """Train XGBoost with 5-fold CV, return metrics + SHAP."""
    # Feature selection
    if feat_cols_override is not None:
        feat_cols = feat_cols_override
    else:
        feat_cols = [c for c in df_seq.columns if c not in exclude_base
                     and df_seq[c].dtype in ['float64', 'int64', 'float32', 'int32']]
    
    # One-hot encode quality_family
    cat_cols = []
    df_work = df_seq[feat_cols + [target]].copy()
    if 'quality_family' in df_seq.columns:
        dummies = pd.get_dummies(df_seq['quality_family'], prefix='grade', drop_first=False)
        for dc in dummies.columns:
            df_work[dc] = dummies[dc].values
            cat_cols.append(dc)
    
    all_feats = feat_cols + cat_cols
    df_m = df_work[all_feats + [target]].dropna().reset_index(drop=True)
    X = df_m[all_feats].values
    y = df_m[target].values
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for _, (tr, va) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42,
            n_jobs=-1, verbosity=0, tree_method='hist')
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = m.predict(X[va])
    
    r2 = r2_score(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    mae = mean_absolute_error(y, oof)
    
    # Final model for SHAP
    final = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        n_jobs=-1, verbosity=0, tree_method='hist')
    final.fit(X, y, verbose=False)
    sv = shap.TreeExplainer(final).shap_values(X)
    shap_imp = pd.DataFrame({'feature': all_feats,
        'mean_abs_shap': np.abs(sv).mean(axis=0)}).sort_values('mean_abs_shap', ascending=False)
    
    return {'r2': r2, 'rmse': rmse, 'mae': mae, 'shap_imp': shap_imp,
            'n_feats': len(all_feats), 'n_samples': len(y)}

# ── Run 3 configurations × 2 strands × 2 datasets ──
print('XGBoost SMOOTHED SEN EXPERIMENT')
print('=' * 100)
print('Config A: Original (raw SEN_std)')
print('Config B: Smoothed (replace SEN_std with SEN_std_smooth_mm — drift only)')
print('Config C: Split    (replace SEN_std with SEN_std_smooth_mm + SEN_std_resid_mm)')
print('=' * 100)

results_all = []

for sname, df_clean, df_all in [
    ('S5', df_seq_5, df_seq_5_all),
    ('S6', df_seq_6, df_seq_6_all)]:
    for dset_label, df_seq in [('Clean', df_clean), ('All', df_all)]:
        print(f'\n  {sname} / {dset_label} ({len(df_seq)} sequences)')
        print(f'  {"-"*80}')
        
        # Config A: Original
        base_feats = [c for c in df_seq.columns if c not in exclude_base
                      and c not in ['SEN_std_smooth_mm', 'SEN_std_resid_mm', 'SEN_std_movavg_mm']
                      and df_seq[c].dtype in ['float64', 'int64', 'float32', 'int32']]
        res_a = xgb_cv(df_seq, base_feats, f'{sname}/{dset_label}/A')
        print(f'    A) Original   : R²={res_a["r2"]:.4f}  RMSE={res_a["rmse"]:.4f}  MAE={res_a["mae"]:.4f}  ({res_a["n_feats"]} feats)')
        
        # Config B: Replace SEN_std with smoothed
        b_feats = [c if c != 'SEN_std [mm]' else 'SEN_std_smooth_mm' for c in base_feats]
        res_b = xgb_cv(df_seq, b_feats, f'{sname}/{dset_label}/B')
        print(f'    B) Smoothed   : R²={res_b["r2"]:.4f}  RMSE={res_b["rmse"]:.4f}  MAE={res_b["mae"]:.4f}  ({res_b["n_feats"]} feats)')
        
        # Config C: Replace SEN_std with smoothed + residual
        c_feats = []
        for c in base_feats:
            if c == 'SEN_std [mm]':
                c_feats.extend(['SEN_std_smooth_mm', 'SEN_std_resid_mm'])
            else:
                c_feats.append(c)
        res_c = xgb_cv(df_seq, c_feats, f'{sname}/{dset_label}/C')
        print(f'    C) Split      : R²={res_c["r2"]:.4f}  RMSE={res_c["rmse"]:.4f}  MAE={res_c["mae"]:.4f}  ({res_c["n_feats"]} feats)')
        
        # Delta
        print(f'    ΔR² (B-A): {res_b["r2"]-res_a["r2"]:+.4f} | ΔR² (C-A): {res_c["r2"]-res_a["r2"]:+.4f}')
        
        results_all.append({'strand': sname, 'dataset': dset_label,
            'A_r2': res_a['r2'], 'B_r2': res_b['r2'], 'C_r2': res_c['r2'],
            'A_rmse': res_a['rmse'], 'B_rmse': res_b['rmse'], 'C_rmse': res_c['rmse'],
            'shap_a': res_a['shap_imp'], 'shap_b': res_b['shap_imp'], 'shap_c': res_c['shap_imp']})

# ── Summary table ──
print(f'\n{"="*100}')
print(f'{"Summary":^100}')
print(f'{"="*100}')
print(f'{"Strand":<8} {"Dataset":<8} {"A (Raw) R²":>12} {"B (Smooth) R²":>14} {"C (Split) R²":>14} {"ΔR² B-A":>10} {"ΔR² C-A":>10}')
print(f'{"-"*80}')
for r in results_all:
    print(f'{r["strand"]:<8} {r["dataset"]:<8} {r["A_r2"]:>12.4f} {r["B_r2"]:>14.4f} {r["C_r2"]:>14.4f} '
          f'{r["B_r2"]-r["A_r2"]:>+10.4f} {r["C_r2"]-r["A_r2"]:>+10.4f}')

# ── SHAP ranking comparison (top 5) for each config ──
print(f'\n{"="*100}')
print('SHAP Top-5 Comparison: Where does SEN rank?')
print(f'{"="*100}')
for r in results_all:
    print(f'\n  {r["strand"]} / {r["dataset"]}:')
    print(f'    {"Rank":>4}  {"A) Original":<35} {"B) Smoothed":<35} {"C) Split":<35}')
    print(f'    {"-"*105}')
    for i in range(5):
        a = r['shap_a'].iloc[i]
        b = r['shap_b'].iloc[i]
        c = r['shap_c'].iloc[i]
        print(f'    {i+1:>4}  {a["feature"]:<25} {a["mean_abs_shap"]:.4f}   '
              f'{b["feature"]:<25} {b["mean_abs_shap"]:.4f}   '
              f'{c["feature"]:<25} {c["mean_abs_shap"]:.4f}')

# ── Find SEN rank in each config ──
print(f'\n{"="*100}')
print('SEN Feature Rank Across Configs')
print(f'{"="*100}')
for r in results_all:
    sen_patterns = {
        'A': ['SEN_std [mm]'],
        'B': ['SEN_std_smooth_mm'],
        'C': ['SEN_std_smooth_mm', 'SEN_std_resid_mm']
    }
    for cfg, shap_df, patterns in [('A', r['shap_a'], sen_patterns['A']),
                                    ('B', r['shap_b'], sen_patterns['B']),
                                    ('C', r['shap_c'], sen_patterns['C'])]:
        for pat in patterns:
            rank_row = shap_df[shap_df['feature'] == pat]
            if len(rank_row) > 0:
                rank = shap_df.index.get_loc(rank_row.index[0]) + 1
                shap_val = rank_row['mean_abs_shap'].values[0]
                print(f'  {r["strand"]}/{r["dataset"]} Config {cfg}: {pat:<25} → rank #{rank:>2}, |SHAP|={shap_val:.4f}')

print(f'\n{"="*100}')
print('CONCLUSION:')
print('  Compare R² changes and SHAP rank shifts to determine if')
print('  high-freq SEN noise carries predictive value or is pure noise.')
print(f'{"="*100}')

In [0]:
# =====================================================================
# SHAP Comparison Figure: Raw vs Smoothed vs Split SEN
# 4 panels (2 strands × 2 datasets)
# Top-10 features per config, SEN features highlighted
# =====================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

top_n = 10
sen_keywords = ['SEN_std', 'SEN_std_smooth', 'SEN_std_resid']

def is_sen(feat):
    return any(k in feat for k in sen_keywords)

def short_name(feat):
    """Shorten feature names for readability."""
    remap = {
        'CASTING_SPEED_avg [m/min]': 'Vc avg',
        'CASTING_SPEED_std [m/min]': 'Vc σ',
        'MOLD_WIDTH_avg [m]': 'Width avg',
        'MOLD_WIDTH_std [m]': 'Width σ',
        'SEN_std [mm]': 'SEN σ (raw)',
        'SEN_avg [mm]': 'SEN avg',
        'SEN_std_smooth_mm': 'SEN σ (smooth)',
        'SEN_std_resid_mm': 'SEN σ (residual)',
        'SEN_std_movavg_mm': 'SEN σ (movavg)',
        'ArFlow_avg [NL/min]': 'Argon flow',
        'ArFlow_SEN_avg': 'Argon SEN',
        'meniscus_FF_LF_asym_std': 'FF-LF asym σ',
        'meniscus_FF_LF_asym_mean': 'FF-LF asym avg',
        'ML_LR_asym_std': 'ML L-R asym σ',
        'ML_LR_asym_mean': 'ML L-R asym avg',
        'cheb_X2_bfl_std': 'X₂ BFL σ',
        'cheb_X2_bff_std': 'X₂ BFF σ',
        'cheb_X1_bff_std': 'X₁ BFF σ',
        'cheb_X1_bfl_std': 'X₁ BFL σ',
        'cheb_X3_bff_mean': 'X₃ BFF avg',
        'cheb_X3_bfl_mean': 'X₃ BFL avg',
        'cheb_X2_FF_LF_diff_mean': 'X₂ FF-LF diff',
        'tundishTemp_avg': 'Tundish T',
        'castingLength_avg': 'Cast length',
        'Seq_duration_min': 'Duration',
        'Seq_num_samples': 'N samples',
    }
    return remap.get(feat, feat.replace('_', ' '))

# Colors
col_a = '#2c3e50'   # dark blue-grey for Config A (raw)
col_b = '#e74c3c'   # red for Config B (smoothed)
col_c = '#27ae60'   # green for Config C (split)
sen_edge = '#f39c12' # gold edge for SEN features

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
panel_idx = 0

for r in results_all:
    row = 0 if r['strand'] == 'S5' else 1
    col = 0 if r['dataset'] == 'Clean' else 1
    ax = axes[row, col]
    
    # Get union of top-N features across all 3 configs
    top_a = r['shap_a'].head(top_n)['feature'].tolist()
    top_b = r['shap_b'].head(top_n)['feature'].tolist()
    top_c = r['shap_c'].head(top_n)['feature'].tolist()
    
    # Merge: union ordered by max rank across configs
    all_feats = list(dict.fromkeys(top_a + top_b + top_c))
    
    # Build merged df
    rows_data = []
    for feat in all_feats:
        va = r['shap_a'][r['shap_a']['feature'] == feat]['mean_abs_shap']
        vb = r['shap_b'][r['shap_b']['feature'] == feat]['mean_abs_shap']
        vc = r['shap_c'][r['shap_c']['feature'] == feat]['mean_abs_shap']
        rows_data.append({
            'feature': feat,
            'A': va.values[0] if len(va) > 0 else 0,
            'B': vb.values[0] if len(vb) > 0 else 0,
            'C': vc.values[0] if len(vc) > 0 else 0,
        })
    df_m = pd.DataFrame(rows_data)
    df_m['max_shap'] = df_m[['A', 'B', 'C']].max(axis=1)
    df_m = df_m.sort_values('max_shap', ascending=True).tail(top_n)
    
    y_pos = np.arange(len(df_m))
    bar_h = 0.25
    
    # Plot bars
    bars_a = ax.barh(y_pos + bar_h, df_m['A'], height=bar_h, color=col_a, alpha=0.85, label='A) Raw SEN σ')
    bars_b = ax.barh(y_pos, df_m['B'], height=bar_h, color=col_b, alpha=0.85, label='B) Smoothed SEN σ')
    bars_c = ax.barh(y_pos - bar_h, df_m['C'], height=bar_h, color=col_c, alpha=0.85, label='C) Split (smooth + resid)')
    
    # Highlight SEN features with gold edge
    for i, feat in enumerate(df_m['feature']):
        if is_sen(feat):
            for bars in [bars_a, bars_b, bars_c]:
                offset = bar_h if bars is bars_a else (0 if bars is bars_b else -bar_h)
                ax.barh(y_pos[i] + offset, df_m.iloc[i]['A' if bars is bars_a else ('B' if bars is bars_b else 'C')],
                    height=bar_h, fill=False, edgecolor=sen_edge, linewidth=2.5)
    
    # Labels
    labels = [short_name(f) for f in df_m['feature']]
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Mean |SHAP value|', fontsize=10)
    ax.set_title(f'{r["strand"]} / {r["dataset"]}\n'
        f'R²: A={r["A_r2"]:.3f} → B={r["B_r2"]:.3f} (Δ={r["B_r2"]-r["A_r2"]:+.3f}) → C={r["C_r2"]:.3f} (Δ={r["C_r2"]-r["A_r2"]:+.3f})',
        fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='x', alpha=0.2)
    ax.set_xlim(0, None)

# Gold edge legend patch
gold_patch = mpatches.Patch(edgecolor=sen_edge, facecolor='none', linewidth=2.5, label='SEN-related feature')
fig.legend(handles=[gold_patch], loc='lower center', fontsize=11, ncol=1,
    bbox_to_anchor=(0.5, -0.01), frameon=False)

fig.suptitle('SHAP Feature Importance: Raw vs Smoothed vs Split SEN\n'
    'Gold border = SEN-related feature | Removing high-freq noise degrades R² by 3–4% on clean data',
    fontsize=14, fontweight='bold')
fig.tight_layout(rect=[0, 0.02, 1, 0.95])

# Save
import shutil, os
fig_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures'
tmp = '/tmp/shap_sen_smoothing_comparison.png'
fig.savefig(tmp, dpi=150, bbox_inches='tight', facecolor='white')
shutil.copy2(tmp, os.path.join(fig_dir, 'shap_sen_smoothing_comparison.png'))
display(fig)
plt.close(fig)
print(f'\u2714 shap_sen_smoothing_comparison.png saved ({os.path.getsize(os.path.join(fig_dir, "shap_sen_smoothing_comparison.png"))/1024:.0f} KB)')